# Colab Session: ut
Generated from colab-cli history log.

**Session Created**: 2026-11-21 17:33:52
- Endpoint: `gpu-t4-hm-use1c0-3a97zthtxl6d5`
- Hardware: `T4`

In [ ]:
import subprocess; print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],capture_output=True,text=True).stdout)


name, memory.total [MiB]
Tesla T4, 15360 MiB



### Automation: install (2026-11-21 17:34:05)

In [ ]:

import subprocess, sys
def install():
    packages = ['-r', '/content/requirements.txt']
    try:
        subprocess.check_call(['uv', 'pip', 'install', '--system'] + packages)
        print('Installation Complete (via uv)!')
    except:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + packages)
        print('Installation Complete (via pip)!')
install()


In [ ]:
# Result of previous automation

Installation Complete (via uv)!


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"


In [ ]:
# 00_setup.py: run once per Colab session (via `colab exec -f`), before any stage.
# Settings can be overridden by executing assignments first, e.g.  echo "MODEL_ID = '...'" | colab exec -s ut
g = globals()
MODEL_ID    = g.get("MODEL_ID", "unsloth/Llama-3.3-70B-Instruct-bnb-4bit")   # alt: unsloth/Qwen2.5-72B-Instruct-bnb-4bit; dry run: Qwen/Qwen2.5-0.5B-Instruct
DRY_RUN     = g.get("DRY_RUN", any(s in MODEL_ID for s in ["0.5B", "1.5B", "tiny"]))
OUT_ROOT    = g.get("OUT_ROOT", "/content/ut_results")       # on the VM; fetched with `colab download` (see pack())
LOCAL_CACHE = g.get("LOCAL_CACHE", "/content/cache")         # big covariance files; stay on the VM
RUN_A, RUN_B, RUN_C, RUN_D = g.get("RUN_A", True), g.get("RUN_B", True), g.get("RUN_C", True), g.get("RUN_D", True)
PAIRS       = g.get("PAIRS", ["es", "hi"])
N_SENT_FIT  = g.get("N_SENT_FIT", 2000 if DRY_RUN else 30000)
N_SENT_HELD = g.get("N_SENT_HELD", 200 if DRY_RUN else 1200)
LAMBDAS     = g.get("LAMBDAS", [1e-3, 1e-2, 1e-1])
SEED        = g.get("SEED", 0)
CHAT_DATE   = "26 Jul 2024"
MIN_GPU_GB  = g.get("MIN_GPU_GB", 0 if DRY_RUN else 75)

import subprocess, sys, matplotlib
matplotlib.use("Agg")
import torch
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0); print(f"GPU: {_p.name}, {_p.total_memory/1e9:.0f} GB")
    if _p.total_memory / 1e9 < MIN_GPU_GB:
        raise RuntimeError(f"This GPU has {_p.total_memory/1e9:.0f} GB; the 4-bit 70B model needs an 80 GB card. "
                           "Stop this session and create one with --gpu H100 (or an 80 GB A100).")
else:
    print("WARNING: no GPU visible")
try:
    import spacy; spacy.load("es_core_news_sm")
except Exception:
    subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm", "-q"], check=False)


import os, re, json, math, time, random, difflib, gc
from collections import defaultdict
import numpy as np
import torch
import matplotlib.pyplot as plt

def slug(s): return re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")
OUT = os.path.join(OUT_ROOT, slug(MODEL_ID)); os.makedirs(OUT, exist_ok=True)
FIG = os.path.join(OUT, "figures"); os.makedirs(FIG, exist_ok=True)
CACHE = os.path.join(LOCAL_CACHE, slug(MODEL_ID)); os.makedirs(CACHE, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def _np(o):
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, torch.Tensor): return o.tolist()
    raise TypeError(type(o))
def save_json(name, obj):
    with open(os.path.join(OUT, name), "w") as f: json.dump(obj, f, indent=1, default=_np, ensure_ascii=False)
def load_json(name):
    with open(os.path.join(OUT, name)) as f: return json.load(f)
def have(name): return os.path.exists(os.path.join(OUT, name))

class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); print(f"▶ {self.label} ..."); return self
    def __exit__(self, *a): print(f"✓ {self.label}: {time.time()-self.t:.0f}s")

def auroc(pos, neg):
    """Probability that a random positive scores above a random negative (ties count half)."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    pos, neg = pos[~np.isnan(pos)], neg[~np.isnan(neg)]
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    allv = np.concatenate([pos, neg]); r = rankdata(allv)
    return float((r[:len(pos)].sum() - len(pos)*(len(pos)+1)/2) / (len(pos)*len(neg)))
def rankdata(a):
    a = np.asarray(a, float); order = np.argsort(a, kind="mergesort"); r = np.empty(len(a))
    sa = a[order]; i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and sa[j+1] == sa[i]: j += 1
        r[order[i:j+1]] = (i + j) / 2 + 1; i = j + 1
    return r
def spearman(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float); m = ~(np.isnan(x) | np.isnan(y))
    if m.sum() < 3: return float("nan")
    return float(np.corrcoef(rankdata(x[m]), rankdata(y[m]))[0, 1])
def boot_ci(x, n=2000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed); m = rng.choice(x, (n, len(x))).mean(1)
    return (float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5)))

# Visual language shared with the earlier figures
EASY, HARD, LOAN, GREY, INK = "#2f7d6d", "#c4572e", "#5b4fc4", "#d9d6cf", "#2b2b2b"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
def savefig(fig, name):
    fig.savefig(os.path.join(FIG, name + ".png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG, name + ".svg"), bbox_inches="tight")
    plt.show(); plt.close(fig)
print("results ->", OUT)

LANG_NAME = {"es": "Spanish", "pt": "Portuguese", "it": "Italian", "de": "German", "hi": "Hindi", "en": "English"}

SENTENCES = [
    # --- untranslatable words in context ---
    dict(id="duende", lang="es", cat="untranslatable", text="Me gusta mucho el duende.",
         hot={"duende": "the mysterious, almost demonic power and emotional intensity of a great artistic performance, as in flamenco (Lorca's sense)",
              "gusta": "it pleases me: the thing is the subject and I am the one affected, like having a taste for it"}),
    dict(id="sobremesa", lang="es", cat="untranslatable", text="Después de comer nos quedamos dos horas de sobremesa.",
         hot={"sobremesa": "the relaxed time spent talking at the table after a meal has finished"}),
    dict(id="madrugada", lang="es", cat="untranslatable", text="Llegamos a casa de madrugada, muertos de sueño.",
         hot={"madrugada": "the small hours, between midnight and dawn"}),
    dict(id="estrenar", lang="es", cat="untranslatable", text="Hoy estreno los zapatos que me regalaste.",
         hot={"estreno": "I am using or wearing something for the very first time"}),
    dict(id="verguenza", lang="es", cat="untranslatable", text="Sentí vergüenza ajena durante todo su discurso.",
         hot={"vergüenza ajena": "embarrassment felt on behalf of someone else who is embarrassing themselves"}),
    dict(id="tertulia", lang="es", cat="untranslatable", text="Los jueves vamos a la tertulia del café de la esquina.",
         hot={"tertulia": "a regular informal gathering of people for conversation, often literary or political"}),
    # --- constructions ---
    dict(id="olvido", lang="es", cat="construction", text="Se me olvidó la llave en el coche.",
         hot={"Se me olvidó": "I forgot it, but framed as something that happened to me by accident, as if the key forgot itself on me"}),
    dict(id="estar_aburrido", lang="es", cat="construction", text="Juan está aburrido en la reunión.",
         hot={"está aburrido": "is bored, a temporary state (not 'is boring')"}),
    dict(id="ser_aburrido", lang="es", cat="construction", text="Juan es aburrido en las reuniones.",
         hot={"es aburrido": "is boring, a lasting characteristic (not 'is bored')"}),
    dict(id="ojala", lang="es", cat="construction", text="Ojalá llueva mañana por la tarde.",
         hot={"Ojalá": "a strong hope or wish, 'if only', originally 'may God will it'"}),
    # --- idioms ---
    dict(id="pelo", lang="es", cat="idiom", text="Creo que me estás tomando el pelo.",
         hot={"tomando el pelo": "teasing me, pulling my leg"}),
    dict(id="nubes", lang="es", cat="idiom", text="Mi hermana siempre está en las nubes.",
         hot={"en las nubes": "daydreaming, absent-minded"}),
    dict(id="pelos", lang="es", cat="idiom", text="Mi abuela no tiene pelos en la lengua.",
         hot={"no tiene pelos en la lengua": "speaks very frankly, without holding anything back"}),
    # --- controls ---
    dict(id="tren", lang="es", cat="control", text="El tren sale a las ocho de la mañana.", hot={}),
    dict(id="hermano", lang="es", cat="control", text="Mi hermano vive en un piso pequeño.", hot={}),
    dict(id="pan", lang="es", cat="control", text="Compré pan y leche en la tienda.", hot={}),
    dict(id="reunion", lang="es", cat="control", text="La reunión empieza a las diez en punto.", hot={}),
    dict(id="gatos", lang="es", cat="control", text="Tengo dos gatos negros y un perro.", hot={}),
    dict(id="invierno", lang="es", cat="control", text="En invierno hace mucho frío en el norte.", hot={}),
    # --- other European languages (Part B only) ---
    dict(id="saudade", lang="pt", cat="untranslatable", text="Tenho muita saudade da minha avó.",
         hot={"saudade": "a deep, bittersweet longing for someone or something absent, that may never return"}),
    dict(id="magari", lang="it", cat="construction", text="Magari fosse vero!",
         hot={"Magari": "if only (a wistful wish)"}),
    dict(id="fernweh", lang="de", cat="untranslatable", text="Im Winter habe ich immer Fernweh.",
         hot={"Fernweh": "an ache to travel to faraway places, the opposite of homesickness"}),
    # --- Hindi (DRAFTS: to be checked by a native speaker) ---
    dict(id="hi_pasand", lang="hi", cat="construction", draft=True, text="मुझे चाय बहुत पसंद है।",
         hot={"मुझे": "to me (dative experiencer: the tea is liked by me, like Spanish 'me gusta')"}),
    dict(id="hi_jugaad", lang="hi", cat="untranslatable", draft=True, text="उसने जुगाड़ से पुरानी गाड़ी ठीक कर दी।",
         hot={"जुगाड़": "a clever improvised fix made with whatever is at hand"}),
    dict(id="hi_mann", lang="hi", cat="untranslatable", draft=True, text="आज पढ़ाई में मेरा मन नहीं लग रहा।",
         hot={"मन": "heart-mind: the seat of feeling, will and attention together"}),
    dict(id="hi_apnapan", lang="hi", cat="untranslatable", draft=True, text="इस घर में मुझे बहुत अपनापन लगता है।",
         hot={"अपनापन": "a warm sense of belonging and intimate familiarity, of being among one's own"}),
    dict(id="hi_train", lang="hi", cat="control", draft=True, text="ट्रेन सुबह आठ बजे चलती है।", hot={}),
    dict(id="hi_bhai", lang="hi", cat="control", draft=True, text="मेरा भाई दिल्ली में रहता है।", hot={}),
]

# Sense minimal pairs for Part D: same word, two contexts, two senses.
# The English candidates differ only in the critical word, so their log-odds measures which sense the translation carries.
SENSE_PAIRS = [
    dict(id="cura", word="cura", a="Me preocupa el cura.", b="Me preocupa la cura.",
         en_a="I am worried about the {}.", en_b="I am worried about the {}.", sense_a="priest", sense_b="cure"),
    dict(id="cometa", word="cometa", a="Miramos el cometa.", b="Miramos la cometa.",
         en_a="We looked at the {}.", en_b="We looked at the {}.", sense_a="comet", sense_b="kite"),
    dict(id="pendiente", word="pendiente", a="Vi el pendiente.", b="Vi la pendiente.",
         en_a="I saw the {}.", en_b="I saw the {}.", sense_a="earring", sense_b="slope"),
    dict(id="corte", word="corte", a="Me sorprendió el corte.", b="Me sorprendió la corte.",
         en_a="The {} surprised me.", en_b="The {} surprised me.", sense_a="cut", sense_b="court"),
    dict(id="frente", word="frente", a="Señaló el frente.", b="Señaló la frente.",
         en_a="He pointed at the {}.", en_b="He pointed at the {}.", sense_a="front", sense_b="forehead"),
    dict(id="papa", word="papa", a="Me gusta el papa.", b="Me gusta la papa.",
         en_a="I like the {}.", en_b="I like the {}.", sense_a="pope", sense_b="potato"),
    dict(id="aburrido", word="aburrido", a="Juan es aburrido.", b="Juan está aburrido.",
         en_a="Juan is {}.", en_b="Juan is {}.", sense_a="boring", sense_b="bored"),
    dict(id="duende_ctx", word="duende", a="En ese cuento infantil, lo que más me gusta es el duende.",
         b="En ese cante flamenco, lo que más me gusta es el duende.",
         en_a="In that children's story, what I like most is the {}.",
         en_b="In that flamenco song, what I like most is the {}.", sense_a="goblin", sense_b="soul"),
]

SID = {S["id"]: S for S in SENTENCES}

# Part A: Spanish -> Italian word translation, with the English equivalent for the logit lens
DETOUR_SHOTS = [("perro", "cane"), ("ventana", "finestra"), ("mesa", "tavolo"), ("leche", "latte")]
DETOUR_WORDS = [("árbol", "albero", "tree"), ("pájaro", "uccello", "bird"), ("manzana", "mela", "apple"),
    ("cuchillo", "coltello", "knife"), ("calle", "strada", "street"), ("niño", "bambino", "child"),
    ("nube", "nuvola", "cloud"), ("queso", "formaggio", "cheese"), ("llave", "chiave", "key"),
    ("zapato", "scarpa", "shoe"), ("camisa", "camicia", "shirt"), ("mantequilla", "burro", "butter"),
    ("cerdo", "maiale", "pig"), ("caballo", "cavallo", "horse"), ("hermano", "fratello", "brother"),
    ("pescado", "pesce", "fish"), ("cuchara", "cucchiaio", "spoon"), ("tenedor", "forchetta", "fork"),
    ("silla", "sedia", "chair"), ("puerta", "porta", "door"), ("ciudad", "città", "city"),
    ("rodilla", "ginocchio", "knee"), ("hielo", "ghiaccio", "ice"), ("lluvia", "pioggia", "rain"),
    ("nieve", "neve", "snow"), ("hoja", "foglia", "leaf"), ("lengua", "lingua", "tongue"),
    ("oreja", "orecchio", "ear"), ("huevo", "uovo", "egg"), ("ajo", "aglio", "garlic"),
    ("cerebro", "cervello", "brain"), ("espejo", "specchio", "mirror")]

from transformers import AutoTokenizer, AutoModelForCausalLM
with Timer(f"load {MODEL_ID}"):
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    _dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=_dtype, device_map="auto")
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=_dtype, device_map="auto")
    model.eval()

def _get_attr_path(obj, path):
    for p in path.split("."): obj = getattr(obj, p)
    return obj
def find_decoder(m):
    for base in ["model", "model.model", "model.language_model", "transformer"]:
        try:
            b = _get_attr_path(m, base)
            layers = getattr(b, "layers", None) or getattr(b, "h", None)
            norm = getattr(b, "norm", None) or getattr(b, "ln_f", None)
            if layers is not None and norm is not None: return b, layers, norm
        except AttributeError: pass
    raise ValueError("could not find decoder layers")

BASE, LAYERS_MOD, FINAL_NORM = find_decoder(model)
LM_HEAD = model.get_output_embeddings()
N_LAYERS = len(LAYERS_MOD)
D_MODEL = model.config.hidden_size if hasattr(model.config, "hidden_size") else model.config.text_config.hidden_size
INPUT_DEV = model.get_input_embeddings().weight.device
BOS = [tok.bos_token_id] if tok.bos_token_id is not None else []
print(f"{N_LAYERS} layers, d={D_MODEL}, BOS={BOS}, input device {INPUT_DEV}")
if torch.cuda.is_available(): print(f"GPU memory in use: {torch.cuda.memory_allocated()/1e9:.1f} GB")

WORD_RE = re.compile(r"[\wऀ-ॣ०-ॿ]+(?:['’]\w+)?")
def words_with_spans(text): return [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(text)]
def enc(text, offsets=False): return tok(text, add_special_tokens=False, return_offsets_mapping=offsets)
def ntok(text): return len(enc(text)["input_ids"])

def chat_prompt(user, prefix=""):
    msgs = [{"role": "user", "content": user}]
    try: p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, date_string=CHAT_DATE)
    except Exception: p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return p + prefix

def tokens_in_span(offsets, a, b):
    return [i for i, (x, y) in enumerate(offsets) if y > x and y > a and x < b]

def find_phrase(text, phrase):
    m = re.search(r"(?<!\w)" + re.escape(phrase) + r"(?!\w)", text, flags=re.I)
    return (m.start(), m.end()) if m else None

def positions_at(prompt, a, b):
    return tokens_in_span(enc(prompt, offsets=True)["offset_mapping"], a, b)

def positions_in_prompt(prompt, sentence, phrase=None, span=None):
    """Token positions of `phrase` (or of the character span `span`) inside `sentence`, inside `prompt`."""
    base = prompt.rfind(sentence); assert base >= 0, "sentence not in prompt"
    sp = span or find_phrase(sentence, phrase); assert sp is not None, f"{phrase!r} not in {sentence!r}"
    return positions_at(prompt, base + sp[0], base + sp[1])

class _Stop(Exception): pass

def make_edit_hook(edits_by_b):
    """edits_by_b: {batch index: [(positions LongTensor, fn)]}; fn maps states [k,d] -> new states [k,d]."""
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        if h.shape[1] <= 1: return None              # cached decoding step: leave untouched
        for b, items in edits_by_b.items():
            for pos, fn in items:
                pos = pos.to(h.device)
                if int(pos.max()) < h.shape[1]:
                    h[b, pos] = fn(h[b, pos]).to(h.dtype)
        return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
    return hook

def _register_edits(edits_for_batch):
    """edits_for_batch: list (one per batch row) of lists of (layer, positions, fn)."""
    per_layer = defaultdict(lambda: defaultdict(list))
    for b, items in enumerate(edits_for_batch):
        for (layer, pos, fn) in (items or []):
            per_layer[layer][b].append((torch.as_tensor(pos, dtype=torch.long), fn))
    return [LAYERS_MOD[layer - 1].register_forward_hook(make_edit_hook(dict(eb))) for layer, eb in per_layer.items()]

@torch.no_grad()
def score(prompts, conts, batch=8, edits=None):
    """Teacher-forced log p of each continuation given its prompt.
    edits: optional list (per item) of [(layer, prompt token positions, fn)]; 'layer l' = hidden_states[l] = output of block l-1.
    Returns per item: dict(lp=array over continuation tokens, offsets=char offsets of those tokens in the continuation)."""
    res = [None] * len(prompts)
    for i in range(0, len(prompts), batch):
        idx = list(range(i, min(i + batch, len(prompts))))
        seqs, spans, offs = [], [], []
        for j in idx:
            p = enc(prompts[j])["input_ids"]; c = enc(conts[j], offsets=True)
            seqs.append(p + c["input_ids"]); spans.append((len(p), len(p) + len(c["input_ids"]))); offs.append(c["offset_mapping"])
        L = max(map(len, seqs))
        ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long); am = torch.zeros_like(ids)
        for b, s in enumerate(seqs): ids[b, :len(s)] = torch.tensor(s); am[b, :len(s)] = 1
        handles = _register_edits([edits[j] for j in idx]) if edits is not None else []
        try:
            out = BASE(input_ids=ids.to(INPUT_DEV), attention_mask=am.to(INPUT_DEV))
        finally:
            for h in handles: h.remove()
        H = out.last_hidden_state
        for b, j in enumerate(idx):
            s, e = spans[b]
            if e <= s: res[j] = dict(lp=np.zeros(0), offsets=[]); continue
            logits = LM_HEAD(H[b, s - 1:e - 1].to(LM_HEAD.weight.device)).float()
            tgt = ids[b, s:e].to(logits.device)
            res[j] = dict(lp=torch.log_softmax(logits, -1).gather(-1, tgt[:, None])[:, 0].cpu().numpy(), offsets=offs[b])
        del out, H
    return res

@torch.no_grad()
def generate(prompts, max_new_tokens=64, batch=16):
    tok.padding_side = "left"; outs = []
    for i in range(0, len(prompts), batch):
        e = tok(prompts[i:i + batch], add_special_tokens=False, return_tensors="pt", padding=True).to(INPUT_DEV)
        g = model.generate(**e, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                           top_k=None, pad_token_id=tok.pad_token_id)
        outs += [tok.decode(r, skip_special_tokens=True).strip() for r in g[:, e["input_ids"].shape[1]:]]
    tok.padding_side = "right"
    return outs

@torch.no_grad()
def generate_with_edits(prompt, edits, max_new_tokens=40):
    """Greedy generation for one prompt, with edits applied to the prompt positions (prefill only)."""
    e = tok(prompt, add_special_tokens=False, return_tensors="pt").to(INPUT_DEV)
    handles = _register_edits([edits])
    try:
        g = model.generate(**e, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                           top_k=None, pad_token_id=tok.pad_token_id)
    finally:
        for h in handles: h.remove()
    return tok.decode(g[0, e["input_ids"].shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def capture(ids, am, layers):
    """Residual states at `layers` (dict layer -> [B,T,d]); stops the forward pass after the deepest one."""
    store, deepest = {}, max(layers)
    def mk(l):
        def hook(m, i, o):
            store[l] = (o[0] if isinstance(o, tuple) else o).detach()
            if l == deepest: raise _Stop()
        return hook
    hs = [LAYERS_MOD[l - 1].register_forward_hook(mk(l)) for l in layers]
    try: BASE(input_ids=ids, attention_mask=am)
    except _Stop: pass
    finally:
        for h in hs: h.remove()
    return store

@torch.no_grad()
def capture_prompt_states(prompt, layer):
    e = tok(prompt, add_special_tokens=False, return_tensors="pt").to(INPUT_DEV)
    return capture(e["input_ids"], e["attention_mask"], [layer])[layer][0]

def encode_raw(text):
    e = enc(text, offsets=True)
    return BOS + e["input_ids"], [(0, 0)] * len(BOS) + list(e["offset_mapping"])

print("helpers ready")

RUNGS = ["B0", "B1", "B2", "B3", "B5", "B6"]
RUNG_NAME = {"B0": "word for word", "B1": "standard", "B2": "keep the word", "B3": "inline gloss",
             "B5": "footnote", "B6": "thick translation"}
NON_BORROW = ["B0", "B1", "B3", "B5", "B6"]
MAXTOK = {"B0": 64, "B1": 64, "B2": 72, "B3": 120, "B5": 150, "B6": 260}

def rung_user(r, s, L):
    n = LANG_NAME[L]
    return {
        "B0": f"Give a word-for-word English gloss of this {n} sentence: one English word for each {n} word, in the original order. Reply with the gloss only.\n\n{s}",
        "B1": f"Translate this {n} sentence into natural English. Reply with the translation only.\n\n{s}",
        "B2": f"Translate this {n} sentence into English, but keep any word that has no good English equivalent in the original {n}, in italics with *asterisks*. Reply with the translation only.\n\n{s}",
        "B3": f"Translate this {n} sentence into English. After each word or phrase that is hard to translate, add a short gloss in square brackets (at most 8 words). Do not quote the {n} words. Reply with the translation only.\n\n{s}",
        "B5": f"Translate this {n} sentence into English, then add one footnote (at most 40 words) explaining what the translation loses. Do not quote the {n} words. Format: the translation, a blank line, then 'Note: ...'.\n\n{s}",
        "B6": f"Translate this {n} sentence into English, then write one paragraph (at most 120 words) opening up the meanings, connotations and grammar of its hard parts, so that an English reader understands what the original says. Do not quote the {n} words.\n\n{s}",
    }[r]



def hot_word_index(S):
    idx = set()
    for span in S["hot"]:
        sp = find_phrase(S["text"], span)
        if sp: idx |= {k for k, (_, a, b) in enumerate(words_with_spans(S["text"])) if b > sp[0] and a < sp[1]}
    return idx




import tarfile
def pack():
    """Bundle everything under OUT into one archive for `colab download`."""
    p = f"{OUT_ROOT.rstrip('/')}_{slug(MODEL_ID)}.tar.gz"
    with tarfile.open(p, "w:gz") as t: t.add(OUT, arcname=slug(MODEL_ID))
    print("PACKED", p)


print("setup done:", MODEL_ID)


GPU: Tesla T4, 16 GB


results -> /content/ut_results/Qwen_Qwen2_5_0_5B_Instruct


▶ load Qwen/Qwen2.5-0.5B-Instruct ...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

... (4 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✓ load Qwen/Qwen2.5-0.5B-Instruct: 26s
24 layers, d=896, BOS=[], input device cuda:0
GPU memory in use: 1.0 GB
helpers ready
setup done: Qwen/Qwen2.5-0.5B-Instruct


In [ ]:
# Reload earlier stages' results, so this stage also works in a fresh kernel (after 00_setup.py).
A = load_json("A_detour.json") if have("A_detour.json") else None
ladder = load_json("B_ladder.json") if have("B_ladder.json") else None
B = load_json("B_roundtrip.json") if have("B_roundtrip.json") else None
summary = load_json("B_summary.json")["per_sentence"] if have("B_summary.json") else {}


if RUN_A and not have("A_detour.json"):
    with Timer("Part A: logit lens"):
        shots = "".join(f'Español: "{e}" - Italiano: "{i}"\n' for e, i in DETOUR_SHOTS)
        rows = []
        for es, it, en in DETOUR_WORDS:
            ti, te, ts = enc(it)["input_ids"][0], enc(en)["input_ids"][0], enc(es)["input_ids"][0]
            if len({ti, te, ts}) < 3: continue           # first tokens must differ
            rows.append((es, it, en, ti, te))
        P_it, P_en = [], []
        for i in range(0, len(rows), 16):
            chunk = rows[i:i + 16]
            seqs = [BOS + enc(shots + f'Español: "{es}" - Italiano: "')["input_ids"] for es, *_ in chunk]
            L = max(map(len, seqs))
            ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long); am = torch.zeros_like(ids)
            for b, s in enumerate(seqs): ids[b, :len(s)] = torch.tensor(s); am[b, :len(s)] = 1
            with torch.no_grad():
                out = BASE(input_ids=ids.to(INPUT_DEV), attention_mask=am.to(INPUT_DEV), output_hidden_states=True)
                last = am.sum(1) - 1
                for l in range(1, N_LAYERS + 1):
                    h = out.hidden_states[l][torch.arange(len(seqs)), last.to(out.hidden_states[l].device)]
                    if l < N_LAYERS: h = FINAL_NORM(h)      # hidden_states[N] already has the final norm
                    p = torch.softmax(LM_HEAD(h.to(LM_HEAD.weight.device)).float(), -1)
                    ti = torch.tensor([r[3] for r in chunk], device=p.device); te = torch.tensor([r[4] for r in chunk], device=p.device)
                    if l == 1: P_it.append([]); P_en.append([])
                    P_it[-1].append(p.gather(1, ti[:, None])[:, 0].cpu().numpy())
                    P_en[-1].append(p.gather(1, te[:, None])[:, 0].cpu().numpy())
            del out
        P_it = np.concatenate([np.stack(x, 1) for x in P_it], 0)     # [words, layers]
        P_en = np.concatenate([np.stack(x, 1) for x in P_en], 0)
        mi, me = P_it.mean(0), P_en.mean(0)
        cut = int(0.9 * N_LAYERS)
        detour_peak = int(np.argmax(me[:cut]) + 1)
        overtake = next((l + 1 for l in range(N_LAYERS) if mi[l] > me[l] and mi[l] > 0.05), None)
        save_json("A_detour.json", dict(words=[r[:3] for r in rows], P_it=P_it, P_en=P_en,
                                        detour_peak_layer=detour_peak, italian_overtakes_at=overtake))
A = load_json("A_detour.json") if have("A_detour.json") else None
if A:
    P_it, P_en = np.array(A["P_it"]), np.array(A["P_en"]); x = np.arange(1, N_LAYERS + 1) / N_LAYERS
    fig, ax = plt.subplots(figsize=(7, 3.4))
    for P, c, lab in [(P_en, HARD, "English word (never asked for)"), (P_it, EASY, "Italian answer")]:
        ax.plot(x, P.mean(0), color=c, lw=2, label=lab)
        ax.fill_between(x, np.percentile(P, 25, 0), np.percentile(P, 75, 0), color=c, alpha=0.15)
    ax.set_xlabel("depth through the model (0 = input, 1 = output)"); ax.set_ylabel("probability")
    ax.set_title(f"Translating Spanish into Italian, the model's working answer passes through English\n({MODEL_ID.split('/')[-1]}, {len(P_it)} words; band = middle half of words)", loc="left")
    ax.legend(frameon=False); savefig(fig, "A_english_detour")
    print(f"English peaks at layer {A['detour_peak_layer']}/{N_LAYERS}; Italian overtakes at {A['italian_overtakes_at']}; "
          f"peak mean P(English) = {P_en.mean(0).max():.3f}")

pack()


▶ Part A: logit lens ...


✓ Part A: logit lens: 2s


English peaks at layer 21/24; Italian overtakes at 24; peak mean P(English) = 0.051
PACKED /content/ut_results_Qwen_Qwen2_5_0_5B_Instruct.tar.gz


*File Operation*: `download` on `/content/ut_results_Qwen_Qwen2_5_0_5B_Instruct.tar.gz`

In [ ]:
# Reload earlier stages' results, so this stage also works in a fresh kernel (after 00_setup.py).
A = load_json("A_detour.json") if have("A_detour.json") else None
ladder = load_json("B_ladder.json") if have("B_ladder.json") else None
B = load_json("B_roundtrip.json") if have("B_roundtrip.json") else None
summary = load_json("B_summary.json")["per_sentence"] if have("B_summary.json") else {}


if RUN_B and not have("B_ladder.json"):
    with Timer("Part B: generating the ladder"):
        ladder = {S["id"]: {} for S in SENTENCES}
        for r in RUNGS:
            outs = generate([chat_prompt(rung_user(r, S["text"], S["lang"])) for S in SENTENCES], max_new_tokens=MAXTOK[r])
            for S, o in zip(SENTENCES, outs): ladder[S["id"]][r] = o
        save_json("B_ladder.json", ladder)
ladder = load_json("B_ladder.json") if have("B_ladder.json") else None
if ladder:
    for S in SENTENCES[:3] + [S for S in SENTENCES if S["id"] == "tren"]:
        print(f"\n{S['text']}"); [print(f"  {r:3s} {RUNG_NAME[r]:18s} {ladder[S['id']][r][:150]!r}") for r in RUNGS]

def mask_source_words(t, s):
    """Replace words in t that copy a source word of >= 4 letters (ignoring accents and a plural s) with '[·]'.
    Proper nouns are left alone. Genuine cognate translations (moment / momento) are NOT masked: they are translations."""
    import unicodedata
    def norm(w): return "".join(c for c in unicodedata.normalize("NFD", w.lower()) if unicodedata.category(c) != "Mn")
    src = set()
    for k, (w, a, b) in enumerate(words_with_spans(s)):
        if len(w) >= 4 and not (w[0].isupper() and k > 0): src |= {norm(w), norm(w) + "s", norm(w).rstrip("s")}
    n = [0]
    def rep(m):
        if norm(m.group(0)) in src: n[0] += 1; return "[·]"
        return m.group(0)
    return WORD_RE.sub(rep, t), n[0]

def bt_user(t, L): return f"Translate this English text into {LANG_NAME[L]}. Reply with the translation only.\n\n{t}"
def copy_user(s, L): return f"Translate this {LANG_NAME[L]} text into {LANG_NAME[L]}. Reply with the translation only.\n\n{s}"
def sense_user(ctx, span, L):
    return (f"Here is a text:\n\n{ctx}\n\nIn the original {LANG_NAME[L]} sentence, what did \"{span}\" mean in this context? "
            f"Answer in one line.")
EMPTY, DELTA = "(no text)", 0.5

def token_word_map(offsets, words):
    return [next((k for k, (_, wa, wb) in enumerate(words) if y > x and y > wa and x < wb), None) for (x, y) in offsets]

def drt_from(lp_t, lp_0, lp_s, offsets, words):
    R, Rs = lp_t - lp_0, lp_s - lp_0
    valid = Rs > DELTA
    w = np.where(valid, Rs, 0.0)
    ell = np.where(valid, 1 - np.clip(R / np.where(valid, Rs, 1.0), 0, 1), 0.0)
    d = float((w * ell).sum() / w.sum()) if w.sum() > 0 else float("nan")
    W, WL = np.zeros(len(words)), np.zeros(len(words))
    for i, k in enumerate(token_word_map(offsets, words)):
        if k is not None: W[k] += w[i]; WL[k] += w[i] * ell[i]
    return d, np.where(W > 0, WL / np.maximum(W, 1e-12), np.nan), W

if RUN_B and ladder and not have("B_roundtrip.json"):
    with Timer("Part B: round-trip scoring"):
        items, meta = [], []
        for S in SENTENCES:
            s, L = S["text"], S["lang"]
            items += [(chat_prompt(bt_user(EMPTY, L)), s), (chat_prompt(copy_user(s, L)), s)]
            meta += [(S["id"], "empty", None), (S["id"], "oracle", None)]
            for r in RUNGS:
                t = ladder[S["id"]][r]; tm, nm = mask_source_words(t, s)
                items += [(chat_prompt(bt_user(tm, L)), s), (chat_prompt(bt_user(t, L)), s)]
                meta += [(S["id"], r, "masked"), (S["id"], r, "raw")]
        sc = score([p for p, _ in items], [c for _, c in items])
        got = {(m[0], m[1], m[2]): r for m, r in zip(meta, sc)}

        # sense recovery
        s_items, s_meta = [], []
        for S in SENTENCES:
            s, L = S["text"], S["lang"]
            for span, gloss in S["hot"].items():
                ctxs = {"source": s, "empty": EMPTY}
                for r in RUNGS:
                    t = ladder[S["id"]][r]
                    ctxs[r] = t if r == "B2" else mask_source_words(t, s)[0]
                for k, ctx in ctxs.items():
                    s_items.append((chat_prompt(sense_user(ctx, span, L)), gloss)); s_meta.append((S["id"], span, k))
        ssc = score([p for p, _ in s_items], [c for _, c in s_items])
        sense_lp = defaultdict(dict)
        for (sid, span, k), r in zip(s_meta, ssc): sense_lp[(sid, span)][k] = float(r["lp"].mean())

        out = {}
        for S in SENTENCES:
            sid, s = S["id"], S["text"]; words = words_with_spans(s)
            lp0, lps = got[(sid, "empty", None)]["lp"], got[(sid, "oracle", None)]["lp"]
            offs = got[(sid, "empty", None)]["offsets"]
            rec = dict(words=[w for w, _, _ in words], rungs={})
            for r in RUNGS:
                t = ladder[sid][r]
                dm, wl_m, W = drt_from(got[(sid, r, "masked")]["lp"], lp0, lps, offs, words)
                du, wl_u, _ = drt_from(got[(sid, r, "raw")]["lp"], lp0, lps, offs, words)
                rec["rungs"][r] = dict(len=ntok(t), d_masked=dm, d_raw=du, word_left_masked=wl_m, word_left_raw=wl_u,
                                       n_masked=mask_source_words(t, s)[1])
            rec["word_weight"] = W
            sr = {}
            for span in S["hot"]:
                v = sense_lp[(sid, span)]; denom = v["source"] - v["empty"]
                sr[span] = dict(raw=v, norm={r: (v[r] - v["empty"]) / denom if abs(denom) > 1e-6 else float("nan") for r in RUNGS})
            rec["sense"] = sr
            out[sid] = rec
        save_json("B_roundtrip.json", out)

B = load_json("B_roundtrip.json") if have("B_roundtrip.json") else None
if B:
    ctrl = [B[S["id"]]["rungs"]["B1"]["d_masked"] for S in SENTENCES if S["cat"] == "control"]
    EPS = float(np.nanpercentile(ctrl, 90))
    def curve_stats(rec, rungs):
        L0 = rec["rungs"]["B1"]["len"]
        cands = [(rec["rungs"][r]["len"], rec["rungs"][r]["d_raw" if r == "B2" else "d_masked"]) for r in rungs]
        ok = [l for l, d in cands if d <= EPS]
        U = math.log2(max(min(ok), 1) / L0) if ok else math.log2(max(l for l, _ in cands) / L0)
        budgets = L0 * 2 ** np.linspace(0, 4, 9)
        Dl = [min([d for l, d in cands if l <= bgt] or [np.nan]) for bgt in budgets]
        return dict(U=max(U, 0.0), censored=not ok, A=float(np.nanmean(Dl)))
    summary = {}
    for sid, rec in B.items():
        summary[sid] = dict(cat=SID[sid]["cat"], lang=SID[sid]["lang"], **curve_stats(rec, NON_BORROW),
                            U_with_borrowing=curve_stats(rec, NON_BORROW + ["B2"])["U"],
                            d_B1=rec["rungs"]["B1"]["d_masked"],
                            borrow_gain=rec["rungs"]["B1"]["d_masked"] - rec["rungs"]["B2"]["d_raw"],
                            borrow_gain_masked=rec["rungs"]["B1"]["d_masked"] - rec["rungs"]["B2"]["d_masked"])
    # hot words vs other content words (standard translation, masked)
    hot_v, oth_v = [], []
    for S in SENTENCES:
        rec = B[S["id"]]; wl = np.array(rec["rungs"]["B1"]["word_left_masked"], float); W = np.array(rec["word_weight"])
        hot_idx = set()
        for span in S["hot"]:
            sp = find_phrase(S["text"], span)
            if sp: hot_idx |= {k for k, (_, a, b) in enumerate(words_with_spans(S["text"])) if b > sp[0] and a < sp[1]}
        for k, w in enumerate(rec["words"]):
            if W[k] <= 0 or np.isnan(wl[k]): continue
            if k in hot_idx: hot_v.append(wl[k])
            elif len(w) >= 4: oth_v.append(wl[k])
    B_auroc = auroc(hot_v, oth_v)
    print(f"tolerance ε (90th pct of control d at 'standard') = {EPS:.3f}")
    print(f"hot words vs other content words, leftover AUROC = {B_auroc:.3f} ({len(hot_v)} hot, {len(oth_v)} other)\n")
    print(f"{'category':15s} {'n':>2s} {'U (doublings)':>13s} {'cens':>4s} {'A':>6s} {'d std':>6s} {'borrow buys':>11s} {'…masked':>8s}")
    cats = ["untranslatable", "construction", "idiom", "control"]
    for c in cats:
        v = [x for x in summary.values() if x["cat"] == c]
        print(f"{c:15s} {len(v):2d} {np.mean([x['U'] for x in v]):13.2f} {sum(x['censored'] for x in v):4d} "
              f"{np.mean([x['A'] for x in v]):6.3f} {np.nanmean([x['d_B1'] for x in v]):6.3f} "
              f"{np.nanmean([x['borrow_gain'] for x in v]):11.3f} {np.nanmean([x['borrow_gain_masked'] for x in v]):8.3f}")
    fa = B["duende"]; wl = dict(zip(fa["words"], fa["rungs"]["B1"]["word_left_masked"]))
    print(f"\nMe gusta mucho el duende (standard translation: {ladder['duende']['B1']!r})")
    print("  leftover: " + ", ".join(f"{w} {wl.get(w, float('nan')):.2f}" for w in ["duende", "gusta", "mucho"])
          + "   [Llama-3.1-8B last time: duende 0.50, gusta 0.23, mucho 0.01]")
    print("  sense recovered (0 = none, 1 = as good as the source):")
    for span, v in fa["sense"].items():
        print(f"    {span:8s} " + "  ".join(f"{RUNG_NAME[r]} {v['norm'][r]:.2f}" for r in RUNGS))
    save_json("B_summary.json", dict(epsilon=EPS, auroc_hot_words=B_auroc, per_sentence=summary))

if B:
    def heat_row(ax, y, words, vals, title, sub):
        """Words set in a row, each shaded by its leftover; positions measured from the rendered text."""
        fig_ = ax.figure; r = fig_.canvas.get_renderer(); inv = ax.transData.inverted(); x = 0.0
        ax.text(0, y + 0.45, title, fontsize=9, color="#666", ha="left")
        for w, v in zip(words, vals):
            bad = v is None or np.isnan(v)
            c = GREY if bad else plt.cm.Oranges(0.08 + 0.85 * float(np.clip(v, 0, 1)))
            t = ax.text(x, y, w, fontsize=17, style="italic", family="serif", va="center", ha="left",
                        bbox=dict(boxstyle="round,pad=0.25", fc=c, ec="none"))
            bb = t.get_window_extent(renderer=r); (x0, _), (x1, _) = inv.transform([[bb.x0, bb.y0], [bb.x1, bb.y1]])
            ax.text((x0 + x1) / 2, y - 0.36, "" if bad else f"{v:.2f}", fontsize=7.5, color="#555", ha="center")
            x = x1 + 0.03
        ax.text(0, y - 0.66, sub, fontsize=9, color=INK, ha="left")
    show = [s for s in ["duende", "olvido", "pelo", "estrenar", "tren"] if s in B]
    fig, ax = plt.subplots(figsize=(9, 1.45 * len(show))); ax.axis("off")
    ax.set_xlim(-0.02, 1.25); ax.set_ylim(-1.5 * len(show) + 0.6, 0.9)
    for k, sid in enumerate(show):
        rec = B[sid]; S = SID[sid]
        wds = [w for w, _, _ in words_with_spans(S["text"])]
        heat_row(ax, -1.5 * k, wds, rec["rungs"]["B1"]["word_left_masked"], S["cat"], "→ " + ladder[sid]["B1"][:110])
    ax.set_xlim(-0.02, 1.25); ax.set_ylim(-1.5 * len(show) + 0.6, 0.9)
    ax.set_title("Where each sentence resists translation: darker words are harder to recover from the English\n"
                 f"(share of each word's information lost after a standard translation; {MODEL_ID.split('/')[-1]})", loc="left", fontsize=10)
    savefig(fig, "B1_heatmaps")

    show = [s for s in ["duende", "sobremesa", "pelo", "tren"] if s in B]
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    for sid in show:
        rec = B[sid]; col = EASY if SID[sid]["cat"] == "control" else HARD
        pts = sorted((rec["rungs"][r]["len"], rec["rungs"][r]["d_masked"], r) for r in NON_BORROW)
        env, best = [], 1.0
        for l, d, r in pts: best = min(best, d); env.append((l, best))
        ax.step([l for l, _ in env], [d for _, d in env], where="post", color=col, lw=2, alpha=0.9)
        ax.scatter([l for l, _, _ in pts], [d for _, d, _ in pts], color=col, s=18, zorder=3)
        ax.text(env[-1][0] * 1.05, env[-1][1], SID[sid]["text"][:28] + "…", fontsize=8, color=col, va="center")
    ax.axhline(EPS, color="#888", ls="--", lw=1); ax.text(ax.get_xlim()[0], EPS, "  gets across", fontsize=8, color="#888", va="bottom")
    ax.set_xscale("log"); ax.set_xlabel("length of the English (tokens, log scale): word for word → standard → gloss → footnote → thick")
    ax.set_ylabel("share of meaning not carried"); ax.set_title("How much English does it take?", loc="left")
    savefig(fig, "B2_distortion_vs_length")

    spans = [(sid, sp) for sid in B for sp in B[sid]["sense"]][:12]
    if spans:
        fig, ax = plt.subplots(figsize=(9, 3.6)); w = 0.15
        rr = ["B1", "B2", "B3", "B5", "B6"]; cols = [GREY, LOAN, "#e3a587", "#d07a55", HARD]
        for k, r in enumerate(rr):
            ax.bar(np.arange(len(spans)) + (k - 2) * w, np.clip([B[s]["sense"][sp]["norm"][r] for s, sp in spans], -0.5, 1.5), w, color=cols[k], label=RUNG_NAME[r])
        ax.set_xticks(np.arange(len(spans))); ax.set_xticklabels([sp for _, sp in spans], rotation=30, ha="right", style="italic")
        ax.axhline(1, color="#888", lw=0.8, ls=":"); ax.axhline(0, color="#888", lw=0.8)
        ax.set_ylabel("intended sense recovered\n(0 = none, 1 = as from the original)"); ax.set_ylim(-0.55, 1.55)
        ax.set_title("Does the English carry the sense the sentence means?", loc="left"); ax.legend(frameon=False, fontsize=8, ncol=5)
        savefig(fig, "B3_sense_recovery")

pack()


▶ Part B: generating the ladder ...


✓ Part B: generating the ladder: 49s

Me gusta mucho el duende.
  B0  word for word      'I like much the magic.'
  B1  standard           'I like much the spirit.'
  B2  keep the word      'I like much the spirit.'
  B3  inline gloss       'I love the spirit.'
  B5  footnote           'Translation: I love the spirit.\nFootnote:\nThis translation is accurate and conveys the meaning of "I love the spirit." However, it does not capture th'
  B6  thick translation  'The Spanish phrase "Me gusta mucho el duende" translates to "I love the charm" in English. The word "duende" means "charm," which is a term used in va'

Después de comer nos quedamos dos horas de sobremesa.
  B0  word for word      'After eating, we stayed for two hours of dessert.'
  B1  standard           'After eating, we spent two hours on dessert.'
  B2  keep the word      'After eating, we spent two hours of dessert.'
  B3  inline gloss       'After eating, we spent two hours on dessert.'
  B5  footnote           'Transl

✓ Part B: round-trip scoring: 42s
tolerance ε (90th pct of control d at 'standard') = 0.439
hot words vs other content words, leftover AUROC = 0.605 (33 hot, 81 other)

category         n U (doublings) cens      A  d std borrow buys  …masked
untranslatable  11          1.07    3  0.371  0.465       0.063    0.047
construction     6          2.49    1  0.369  0.373       0.141   -0.069
idiom            3          0.00    0  0.148  0.180      -0.013   -0.013
control          8          0.61    1  0.218  0.236       0.058   -0.009

Me gusta mucho el duende (standard translation: 'I like much the spirit.')
  leftover: duende 1.00, gusta 0.00, mucho 0.00   [Llama-3.1-8B last time: duende 0.50, gusta 0.23, mucho 0.01]
  sense recovered (0 = none, 1 = as good as the source):
    duende   word for word -0.35  standard -0.54  keep the word -0.54  inline gloss -0.76  footnote -0.65  thick translation -0.30
    gusta    word for word 0.09  standard -0.11  keep the word -0.11  inline gloss 0.59  f

PACKED /content/ut_results_Qwen_Qwen2_5_0_5B_Instruct.tar.gz


*File Operation*: `download` on `/content/ut_results_Qwen_Qwen2_5_0_5B_Instruct.tar.gz`

In [ ]:
# Reload earlier stages' results, so this stage also works in a fresh kernel (after 00_setup.py).
A = load_json("A_detour.json") if have("A_detour.json") else None
ladder = load_json("B_ladder.json") if have("B_ladder.json") else None
B = load_json("B_roundtrip.json") if have("B_roundtrip.json") else None
summary = load_json("B_summary.json")["per_sentence"] if have("B_summary.json") else {}


def pick_layers():
    f = [0.25, 0.4, 0.5, 0.6, 0.75]
    ls = {max(1, min(N_LAYERS - 1, round(x * N_LAYERS))) for x in f}
    if A and A.get("detour_peak_layer"): ls.add(int(min(A["detour_peak_layer"], N_LAYERS - 1)))
    return sorted(ls)
C_LAYERS = pick_layers(); print("Part C layers:", C_LAYERS)

def load_pairs(lang, n):
    """n parallel sentence pairs (x, en, source); a mix of domains so 'conventional practice' isn't one register."""
    from datasets import load_dataset
    # Europarl first: if it fails to load, OPUS-100 takes the whole quota
    sources = ([("Helsinki-NLP/europarl", "en-es")] if lang == "es" else []) + [("Helsinki-NLP/opus-100", f"en-{lang}")]
    want, pairs, seen = int(n * 1.3), [], set()
    for k, (name, cfg) in enumerate(sources):
        quota = (want - len(pairs)) // (len(sources) - k)
        try: ds = load_dataset(name, cfg, split="train", streaming=True)
        except Exception as e: print(f"  could not load {name}/{cfg}: {e}"); continue
        got = 0
        for ex in ds:
            tr = ex.get("translation", {}); en, xx = tr.get("en"), tr.get(lang)
            if not en or not xx: continue
            ne, nx = len(en.split()), len(xx.split())
            if not (8 <= ne <= 40 and 8 <= nx <= 40) or abs(ne - nx) > 0.5 * max(ne, nx): continue
            if sum(ch.isdigit() for ch in xx) > 6 or xx in seen: continue
            seen.add(xx); pairs.append((xx.strip(), en.strip(), name)); got += 1
            if got >= quota: break
        print(f"  {name}/{cfg}: {got} pairs")
    random.Random(SEED).shuffle(pairs)
    return pairs[:n]

def words_for(lang, texts):
    """Word tokens (text, start, end, morph, pos). Spanish uses spaCy when available, for morphology."""
    if lang == "es":
        try:
            import spacy
            nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
            out = []
            for doc in nlp.pipe(texts, batch_size=256):
                out.append([(t.text, t.idx, t.idx + len(t.text), str(t.morph), t.pos_) for t in doc
                            if WORD_RE.fullmatch(t.text)])
            return out
        except Exception as e:
            print("  spaCy unavailable, using regex words (no morphology probes):", e)
    return [[(w, a, b, "", "") for w, a, b in words_with_spans(t)] for t in texts]

_AL = None
def align_pairs(src_words, tgt_words, batch=64, layer=8):
    """1-to-1 word links (i, j) per sentence pair: mutual nearest neighbours of XLM-R (layer 8) word embeddings,
    i.e. SimAlign's high-precision 'argmax' method, implemented here to avoid a fragile dependency."""
    global _AL
    from transformers import AutoTokenizer, AutoModel
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if _AL is None:
        _AL = (AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base"),
               AutoModel.from_pretrained("FacebookAI/xlm-roberta-base").to(dev).eval())
    atok, amod = _AL
    def embed(word_lists):
        wl_ = [[w[0] for w in wl] or ["."] for wl in word_lists]
        e = atok(wl_, is_split_into_words=True, return_tensors="pt", padding=True, truncation=True, max_length=256)
        with torch.no_grad():
            H = amod(**{k: v.to(dev) for k, v in e.items()}, output_hidden_states=True).hidden_states[layer]
        out = []
        for b, wl in enumerate(wl_):
            M = torch.zeros(len(wl), H.shape[1], device=dev)
            for p, w in enumerate(e.word_ids(b)):
                if w is not None: M[w, p] = 1.0
            cnt = M.sum(1, keepdim=True)
            out.append((torch.nn.functional.normalize((M @ H[b].float()) / cnt.clamp_min(1), dim=-1), cnt[:, 0] > 0))
        return out
    links = []
    for i in range(0, len(src_words), batch):
        es, et = embed(src_words[i:i + batch]), embed(tgt_words[i:i + batch])
        for (a, va), (b, vb), sw, tw in zip(es, et, src_words[i:i + batch], tgt_words[i:i + batch]):
            if not sw or not tw: links.append([]); continue
            S = a @ b.T; S[~va] = -2; S[:, ~vb] = -2
            fwd, bwd = S.argmax(1), S.argmax(0)
            links.append([(k, int(fwd[k])) for k in range(len(sw)) if int(bwd[fwd[k]]) == k and float(S[k, fwd[k]]) > -1
                          and not any(ch.isdigit() for ch in sw[k][0] + tw[int(fwd[k])][0])])
    return links

# ---- collection ----

class CovAcc:
    def __init__(self, d, dev):
        z = lambda *s: torch.zeros(*s, dtype=torch.float64, device=dev)
        self.n, self.sx, self.sy = 0, z(d), z(d)
        self.Sxx, self.Syy, self.Sxy = z(d, d), z(d, d), z(d, d)
    def add(self, X, Y):
        X, Y = X.double(), Y.double()
        self.n += X.shape[0]; self.sx += X.sum(0); self.sy += Y.sum(0)
        self.Sxx += X.T @ X; self.Syy += Y.T @ Y; self.Sxy += X.T @ Y
    def finish(self):
        n = self.n; mx, my = self.sx / n, self.sy / n
        return dict(n=n, mx=mx.cpu(), my=my.cpu(),
                    Cxx=((self.Sxx - n * torch.outer(mx, mx)) / (n - 1)).cpu(),
                    Cyy=((self.Syy - n * torch.outer(my, my)) / (n - 1)).cpu(),
                    Cxy=((self.Sxy - n * torch.outer(mx, my)) / (n - 1)).cpu())

def word_states(texts, word_lists, layers, batch=32):
    """For each text: dict layer -> [n_words, d] float32 (word = mean of its tokens), plus a validity mask
    (words containing the first content token, an attention sink, are invalid)."""
    out = []
    for i in range(0, len(texts), batch):
        enc_ = [encode_raw(t) for t in texts[i:i + batch]]
        L = max(len(e[0]) for e in enc_)
        ids = torch.full((len(enc_), L), tok.pad_token_id, dtype=torch.long); am = torch.zeros_like(ids)
        for b, (s, _) in enumerate(enc_): ids[b, :len(s)] = torch.tensor(s); am[b, :len(s)] = 1
        st = capture(ids.to(INPUT_DEV), am.to(INPUT_DEV), layers)
        for b, (s, offs) in enumerate(enc_):
            wl = word_lists[i + b]; first = len(BOS)
            toks = [tokens_in_span(offs, w[1], w[2]) for w in wl]
            valid = np.array([len(t) > 0 and first not in t for t in toks], bool)
            M = torch.zeros(len(wl), len(s))
            for k, t in enumerate(toks):
                if t: M[k, t] = 1.0 / len(t)
            M = M.to(st[layers[0]].device)
            out.append(({l: M @ st[l][b, :len(s)].float() for l in layers}, valid))
        del st
    return out

def collect_pair(lang):
    tag = f"C_{lang}"
    cov_paths = {l: os.path.join(CACHE, f"{tag}_cov_L{l}.pt") for l in C_LAYERS}
    held_path = os.path.join(CACHE, f"{tag}_held.pt")
    if all(os.path.exists(p) for p in cov_paths.values()) and os.path.exists(held_path): return
    with Timer(f"Part C [{lang}]: corpus"):
        pairs = load_pairs(lang, N_SENT_FIT + N_SENT_HELD)
        xs, ens = [p[0] for p in pairs], [p[1] for p in pairs]
        wx, we = words_for(lang, xs), words_for("en", ens)
    with Timer(f"Part C [{lang}]: word alignment ({len(pairs)} pairs)"):
        links = align_pairs(wx, we)
    n_fit = len(pairs) - N_SENT_HELD
    acc = {l: CovAcc(D_MODEL, INPUT_DEV) for l in C_LAYERS}
    held = {l: ([], []) for l in C_LAYERS}; held_meta = []
    with Timer(f"Part C [{lang}]: collecting states at layers {C_LAYERS}"):
        B_ = 32
        for i in range(0, len(pairs), B_):
            sx = word_states(xs[i:i + B_], wx[i:i + B_], C_LAYERS)
            sy = word_states(ens[i:i + B_], we[i:i + B_], C_LAYERS)
            for b in range(len(sx)):
                k = i + b; (rx, vx), (ry, vy) = sx[b], sy[b]
                lk = [(a, c) for a, c in links[k] if vx[a] and vy[c]]
                if not lk: continue
                ia = torch.tensor([a for a, _ in lk]); ic = torch.tensor([c for _, c in lk])
                for l in C_LAYERS:
                    X, Y = rx[l][ia.to(rx[l].device)], ry[l][ic.to(ry[l].device)]
                    if k < n_fit: acc[l].add(X, Y)
                    else: held[l][0].append(X.half().cpu()); held[l][1].append(Y.half().cpu())
                if k >= n_fit:
                    held_meta += [dict(sent=k, x=wx[k][a][0], y=we[k][c][0], morph=wx[k][a][3], pos=wx[k][a][4]) for a, c in lk]
            if (i // B_) % 100 == 0: print(f"    {i}/{len(pairs)} sentences, {acc[C_LAYERS[0]].n} fit pairs")
    for l in C_LAYERS: torch.save(acc[l].finish(), cov_paths[l])
    torch.save(dict(X={l: torch.cat(held[l][0]) for l in C_LAYERS}, Y={l: torch.cat(held[l][1]) for l in C_LAYERS},
                    meta=held_meta, n_fit_pairs=acc[C_LAYERS[0]].n, n_sent=len(pairs)), held_path)
    del acc; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

if RUN_C:
    for lang in PAIRS: collect_pair(lang)

pack()


Part C layers: [6, 10, 12, 14, 18, 21]
▶ Part C [es]: corpus ...


README.md:   0%|          | 0.00/76.3k [00:00<?, ?B/s]

  Helsinki-NLP/europarl/en-es: 1430 pairs


README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

  Helsinki-NLP/opus-100/en-es: 1430 pairs


✓ Part C [es]: corpus: 11s
▶ Part C [es]: word alignment (2200 pairs) ...


... (13 outputs left out)

✓ Part C [hi]: word alignment (2200 pairs): 18s
▶ Part C [hi]: collecting states at layers [6, 10, 12, 14, 18, 21] ...


    0/2200 sentences, 245 fit pairs


✓ Part C [hi]: collecting states at layers [6, 10, 12, 14, 18, 21]: 148s


PACKED /content/ut_results_Qwen_Qwen2_5_0_5B_Instruct.tar.gz


*File Operation*: `download` on `/content/ut_results_Qwen_Qwen2_5_0_5B_Instruct.tar.gz`

In [ ]:
# Reload earlier stages' results, so this stage also works in a fresh kernel (after 00_setup.py).
A = load_json("A_detour.json") if have("A_detour.json") else None
ladder = load_json("B_ladder.json") if have("B_ladder.json") else None
B = load_json("B_roundtrip.json") if have("B_roundtrip.json") else None
summary = load_json("B_summary.json")["per_sentence"] if have("B_summary.json") else {}


CDEV = "cuda" if torch.cuda.is_available() else "cpu"

def inv_sqrt(C):
    e, V = torch.linalg.eigh(C); return (V * e.clamp_min(1e-12).rsqrt()) @ V.T

def cca_fit(cov, lam):
    Cxx, Cyy, Cxy = (cov[k].to(CDEV) for k in ["Cxx", "Cyy", "Cxy"]); d = Cxx.shape[0]
    I = torch.eye(d, dtype=Cxx.dtype, device=CDEV)
    Cxx_r = Cxx + lam * torch.trace(Cxx) / d * I; Cyy_r = Cyy + lam * torch.trace(Cyy) / d * I
    Wx, Wy = inv_sqrt(Cxx_r), inv_sqrt(Cyy_r)
    U, S, Vh = torch.linalg.svd(Wx @ Cxy @ Wy)
    return dict(lam=lam, rho_in=S, U=U, Wx=Wx, Wy=Wy, A=Wx @ U, B=Wy @ Vh.T, Cxx_r=Cxx_r, Cyy_r=Cyy_r,
                Cxx=Cxx, Cyy=Cyy, Cxy=Cxy, mx=cov["mx"].to(CDEV), my=cov["my"].to(CDEV))

def colcorr(P, Q):
    P = P - P.mean(0); Q = Q - Q.mean(0)
    return (P * Q).sum(0) / (P.norm(dim=0) * Q.norm(dim=0) + 1e-12)

def heldout_rho(fit, X, Y):
    return colcorr((X - fit["mx"]) @ fit["A"], (Y - fit["my"]) @ fit["B"])

def r2(Yc, Yp):
    raw = 1 - ((Yc - Yp) ** 2).sum() / (Yc ** 2).sum()
    per = 1 - ((Yc - Yp) ** 2).sum(0) / ((Yc - Yc.mean(0)) ** 2).sum(0).clamp_min(1e-12)
    return float(raw), float(per.mean())

def analyse_layer(lang, l):
    cov = torch.load(os.path.join(CACHE, f"C_{lang}_cov_L{l}.pt"), weights_only=False)
    held = torch.load(os.path.join(CACHE, f"C_{lang}_held.pt"), weights_only=False)
    X, Y = held["X"][l].to(CDEV).double(), held["Y"][l].to(CDEV).double()
    sent = torch.tensor([m["sent"] for m in held["meta"]])
    half = (sent % 2 == 0).to(CDEV)                       # split held-out by sentence
    best = None
    for lam in LAMBDAS:
        fit = cca_fit(cov, lam); rho = heldout_rho(fit, X[half], Y[half])
        s50 = float(rho[:50].sum())
        if best is None or s50 > best[0]: best = (s50, lam)
        del fit
    fit = cca_fit(cov, best[1])
    rho_h = heldout_rho(fit, X[~half], Y[~half])          # reported on the other half
    perm = torch.randperm(int((~half).sum()), generator=torch.Generator().manual_seed(SEED)).to(CDEV)
    rho_null = heldout_rho(fit, X[~half], Y[~half][perm])
    thr = float(torch.quantile(rho_null.abs().float(), 0.99))
    shared = torch.nonzero(rho_h > thr)[:, 0]
    # English bias: held-out share of each language's variance predictable from the other
    Xc, Yc = X[~half] - fit["mx"], Y[~half] - fit["my"]
    Gxy = torch.linalg.solve(fit["Cxx_r"], fit["Cxy"]); Gyx = torch.linalg.solve(fit["Cyy_r"], fit["Cxy"].T)
    E_en, E_en_std = r2(Yc, Xc @ Gxy)          # English predicted from the other language
    E_x, E_x_std = r2(Xc, Yc @ Gyx)            # other language predicted from English
    spec_x = float(torch.trace(fit["Cxy"] @ Gyx) / torch.trace(fit["Cxx"]))
    res = dict(layer=l, lam=best[1], n_fit_pairs=int(cov["n"]), n_held_pairs=int((~half).sum()),
               rho_in=fit["rho_in"][:1024].cpu().numpy(), rho_held=rho_h[:1024].cpu().numpy(),
               null_thr=thr, n_shared=int(len(shared)), top50_held=float(rho_h[:50].sum()), top50_in=float(fit["rho_in"][:50].sum()),
               E_en_from_x=E_en, E_x_from_en=E_x, E_en_from_x_std=E_en_std, E_x_from_en_std=E_x_std,
               bias=E_en - E_x, bias_std=E_en_std - E_x_std, E_x_spectral_in_sample=spec_x)
    return res, fit, rho_h, shared, Gxy, Gyx

C_RES, C_FITS, BEST_LAYER = {}, {}, {}
if RUN_C:
    for lang in PAIRS:
        with Timer(f"Part C [{lang}]: CCA at {len(C_LAYERS)} layers"):
            rows = []
            for l in C_LAYERS:
                out = analyse_layer(lang, l); res = out[0]; rows.append(res); del out
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                print(f"  L{l:3d}  λ={res['lam']:.0e}  shared dirs={res['n_shared']:5d}  top-50 ρ held/in-sample="
                      f"{res['top50_held']:.1f}/{res['top50_in']:.1f}  E_en←{lang}={res['E_en_from_x']:.3f}  "
                      f"E_{lang}←en={res['E_x_from_en']:.3f}  bias={res['bias']:+.3f}")
            save_json(f"C_spectrum_{lang}.json", rows); C_RES[lang] = rows
            BEST_LAYER[lang] = max(rows, key=lambda r: r["top50_held"])["layer"]
    print("best (most shared) layer:", BEST_LAYER)
    # refit and keep only what later cells need: the best layer per language, plus the detour layer for Spanish (Part D)
    keep = {(lang, BEST_LAYER[lang]) for lang in C_RES}
    if A and "es" in C_RES: keep.add(("es", min(C_LAYERS, key=lambda l: abs(l - A["detour_peak_layer"]))))
    for (lang, l) in sorted(keep):
        res, fit, rho_h, shared, Gxy, Gyx = analyse_layer(lang, l)
        C_FITS[(lang, l)] = dict(fit={k: fit[k] for k in ["Wx", "U", "A", "Cxx_r", "mx"]}, rho_h=rho_h, shared=shared, Gxy=Gxy, Gyx=Gyx)
        del fit; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

def span_scores(lang, l, sentences):
    """U (whitened), U_raw, half-life and #tokens for every word of the given test sentences."""
    F_ = C_FITS[(lang, l)]; fit = F_["fit"]; rho2 = F_["rho_h"].clamp(0, 1).double() ** 2
    texts = [S["text"] for S in sentences]
    wl = [[(w, a, b) for w, a, b in words_with_spans(t)] for t in texts]
    st = word_states(texts, wl, [l])
    ns = np.linspace(0, 50, 501); out = {}
    for S, words, (rec, valid) in zip(sentences, wl, st):
        X = rec[l].double().to(CDEV) - fit["mx"]
        c2 = ((X @ fit["Wx"]) @ fit["U"]) ** 2               # canonical coordinates squared
        Rn = torch.stack([(c2 * rho2 ** n).sum(1) / c2.sum(1) for n in ns], 1).cpu().numpy()
        Xrt = (X @ F_["Gxy"]) @ F_["Gyx"]
        U_raw = (((X - Xrt) ** 2).sum(1) / (X ** 2).sum(1)).cpu().numpy()
        half = [float(ns[np.argmax(r <= 0.5)]) if (r <= 0.5).any() else 50.0 for r in Rn]
        out[S["id"]] = dict(words=[w for w, _, _ in words], U=(1 - Rn[:, 10]).tolist(), U_raw=U_raw.tolist(),
                            half_life=half, R_curve=Rn[:, ::10][:, :11].tolist(),
                            ntok=[len(enc(w)["input_ids"]) for w, _, _ in words], valid=valid.tolist())
    return out

def hot_word_index(S):
    idx = set()
    for span in S["hot"]:
        sp = find_phrase(S["text"], span)
        if sp: idx |= {k for k, (_, a, b) in enumerate(words_with_spans(S["text"])) if b > sp[0] and a < sp[1]}
    return idx

C_SPANS = {}
if RUN_C and C_RES:
    for lang in C_RES:
        test = [S for S in SENTENCES if S["lang"] == lang]
        if not test: continue
        rows = {}
        for (lg, l) in C_FITS:
            if lg != lang: continue
            sc = span_scores(lang, l, test)
            hot, oth = defaultdict(list), defaultdict(list)
            for S in test:
                r = sc[S["id"]]; hi = hot_word_index(S)
                for k, w in enumerate(r["words"]):
                    if not r["valid"][k]: continue
                    tgt = hot if k in hi else (oth if len(w) >= 4 else None)
                    if tgt is None: continue
                    for m in ["U", "U_raw", "half_life", "ntok"]: tgt[m].append(r[m][k])
            au = {m: auroc(hot[m], oth[m]) for m in ["U", "U_raw", "half_life", "ntok"]}
            # agreement with Part B (sentence-level, content words weighted by round-trip information weight)
            agree = None
            if B:
                ui, ub = [], []
                for S in test:
                    r = sc[S["id"]]; W = np.array(B[S["id"]]["word_weight"]); u = np.array(r["U"]); v = np.array(r["valid"])
                    if (W[v] > 0).any(): ui.append(float((W[v] * u[v]).sum() / W[v].sum())); ub.append(summary[S["id"]]["A"])
                agree = spearman(ui, ub)
            rows[l] = dict(scores=sc, auroc=au, spearman_with_roundtrip_A=agree, n_hot=len(hot["U"]), n_other=len(oth["U"]))
            print(f"[{lang}] layer {l}: AUROC hot vs other content words — U {au['U']:.2f}, U_raw {au['U_raw']:.2f}, "
                  f"half-life {1-au['half_life']:.2f} (shorter = harder), #tokens baseline {au['ntok']:.2f}; "
                  f"agreement with round trip ρ_s={agree if agree is None else round(agree, 2)}")
            if lang == "es" and "duende" in sc:
                r = sc["duende"]; d = dict(zip(r["words"], zip(r["U"], r["half_life"])))
                print("   Me gusta mucho el duende: " + ", ".join(f"{w} U={d[w][0]:.3f} (half-life {d[w][1]:.1f})" for w in ["duende", "gusta", "mucho"] if w in d))
        C_SPANS[lang] = rows
        save_json(f"C_spans_{lang}.json", rows)

if RUN_C and "es" in C_RES and C_SPANS.get("es"):
    held = torch.load(os.path.join(CACHE, "C_es_held.pt"), weights_only=False); meta = held["meta"]
    l = BEST_LAYER["es"]; F_ = C_FITS[("es", l)]; fit = F_["fit"]
    if any(m["morph"] for m in meta):
        from sklearn.linear_model import RidgeClassifier
        from sklearn.model_selection import cross_val_score
        Z = (((held["X"][l].to(CDEV).double() - fit["mx"]) @ fit["Wx"]) @ fit["U"]).float().cpu().numpy()
        Z /= Z.std(0, keepdims=True) + 1e-8
        sh = np.zeros(Z.shape[1], bool); sh[F_["shared"].cpu().numpy()] = True
        def feat(m, key): return dict(kv.split("=") for kv in m["morph"].split("|") if "=" in kv).get(key)
        tasks = {
            "grammatical gender": [(i, feat(m, "Gender")) for i, m in enumerate(meta) if m["pos"] in ("NOUN", "ADJ") and feat(m, "Gender") in ("Masc", "Fem")],
            "number": [(i, feat(m, "Number")) for i, m in enumerate(meta) if m["pos"] in ("NOUN", "ADJ", "VERB") and feat(m, "Number") in ("Sing", "Plur")],
            "subjunctive vs indicative": [(i, feat(m, "Mood")) for i, m in enumerate(meta) if m["pos"] in ("VERB", "AUX") and feat(m, "Mood") in ("Sub", "Ind")],
            "imperfect vs preterite": [(i, feat(m, "Tense")) for i, m in enumerate(meta) if m["pos"] in ("VERB", "AUX") and feat(m, "Mood") == "Ind" and feat(m, "Tense") in ("Imp", "Past")],
            "noun vs verb (sanity)": [(i, m["pos"]) for i, m in enumerate(meta) if m["pos"] in ("NOUN", "VERB")],
        }
        probe = {}
        for name, items in tasks.items():
            labs = [b for _, b in items]
            if len(items) < 60 or min(labs.count(v) for v in set(labs)) < 20: print(f"  {name}: too few examples"); continue
            ii = np.array([i for i, _ in items]); y = np.array([b == sorted(set(labs))[0] for b in labs])
            clf = RidgeClassifier(alpha=float(len(ii)), class_weight="balanced")
            acc = {k: float(cross_val_score(clf, Z[ii][:, cols], y, cv=5, scoring="balanced_accuracy").mean())
                   for k, cols in [("all", np.ones_like(sh)), ("shared only", sh), ("private only", ~sh)] if cols.any()}
            clf.fit(Z[ii], y); wgt = clf.coef_[0] ** 2
            probe[name] = dict(n=len(ii), shared_share=float(wgt[sh].sum() / wgt.sum()), expected=float(sh.mean()), acc=acc)
            print(f"  {name:28s} n={len(ii):5d}  weight on shared directions {probe[name]['shared_share']:.2f} "
                  f"(chance {sh.mean():.2f})  accuracy all/shared/private = " + "/".join(f"{v:.2f}" for v in acc.values()))
        save_json("C_probes_es.json", dict(layer=l, probes=probe))

if RUN_C and C_RES:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    ax = axes[0]
    for lang, col in zip(C_RES, [HARD, LOAN, EASY]):
        r = next(x for x in C_RES[lang] if x["layer"] == BEST_LAYER[lang]); k = np.arange(1, len(r["rho_in"]) + 1)
        ax.plot(k, r["rho_in"], color=col, lw=1, alpha=0.5, ls="--")
        ax.plot(k, r["rho_held"], color=col, lw=1.6, label=f"{lang}–en, layer {r['layer']} (held-out)")
        ax.axhline(r["null_thr"], color=col, lw=0.8, ls=":")
    ax.set_xscale("log"); ax.set_xlabel("canonical direction k"); ax.set_ylabel("ρ_k")
    ax.set_title("Round-trip spectrum (dashed: in-sample; dotted: shuffled-pair null)", loc="left", fontsize=9); ax.legend(frameon=False, fontsize=8)
    ax = axes[1]
    for lang, col in zip(C_RES, [HARD, LOAN, EASY]):
        ax.plot([r["layer"] / N_LAYERS for r in C_RES[lang]], [r["top50_held"] / 50 for r in C_RES[lang]], "o-", color=col, label=f"{lang}–en")
    if A: ax.axvline(A["detour_peak_layer"] / N_LAYERS, color="#888", ls="--", lw=1); ax.text(A["detour_peak_layer"] / N_LAYERS, ax.get_ylim()[0], " English detour peak", fontsize=8, color="#888")
    ax.set_xlabel("depth"); ax.set_ylabel("mean held-out ρ, top 50 directions"); ax.legend(frameon=False)
    ax.set_title("Where the two languages share most", loc="left", fontsize=9)
    savefig(fig, "C1_spectrum")

    if C_SPANS.get("es"):
        sc = C_SPANS["es"][BEST_LAYER["es"]]["scores"]
        pick = [("duende", "duende"), ("duende", "gusta"), ("sobremesa", "sobremesa"), ("madrugada", "madrugada"),
                ("tren", "tren"), ("hermano", "hermano"), ("gatos", "gatos")]
        fig, ax = plt.subplots(figsize=(7, 3.6))
        for sid, w in pick:
            if sid not in sc or w not in sc[sid]["words"]: continue
            k = sc[sid]["words"].index(w); col = EASY if SID[sid]["cat"] == "control" else HARD
            ax.plot(range(11), sc[sid]["R_curve"][k], color=col, lw=2, alpha=0.85)
            ax.text(10.2, sc[sid]["R_curve"][k][-1], f"{w} (half-life {sc[sid]['half_life'][k]:.1f})", fontsize=8, style="italic", color=col, va="center")
        ax.axhline(0.5, color="#888", ls=":", lw=1)
        ax.set_xlabel("number of round trips Spanish → English → Spanish (inside the model)"); ax.set_ylabel("share of the word that survives")
        ax.set_title("What survives repeated round trips", loc="left"); ax.set_xlim(0, 13.5)
        savefig(fig, "C2_retention")

    fig, ax = plt.subplots(figsize=(5.5, 3.4)); xs = np.arange(len(C_RES)); w = 0.35
    r_ = [next(x for x in C_RES[lang] if x["layer"] == BEST_LAYER[lang]) for lang in C_RES]
    ax.bar(xs - w / 2, [r["E_en_from_x"] for r in r_], w, color=GREY, label="English, predicted from the other language")
    ax.bar(xs + w / 2, [r["E_x_from_en"] for r in r_], w, color=HARD, label="other language, predicted from English")
    ax.set_xticks(xs); ax.set_xticklabels([f"{LANG_NAME[l]}–English" for l in C_RES]); ax.set_ylabel("share of variation predictable")
    ax.set_title("How much of each language survives the crossing", loc="left"); ax.legend(frameon=False, fontsize=8)
    savefig(fig, "C3_english_bias")

pack()


▶ Part C [es]: CCA at 6 layers ...


  L  6  λ=1e-01  shared dirs=  891  top-50 ρ held/in-sample=40.4/40.9  E_en←es=0.426  E_es←en=0.434  bias=-0.008


  L 10  λ=1e-01  shared dirs=  896  top-50 ρ held/in-sample=40.6/40.9  E_en←es=0.454  E_es←en=0.457  bias=-0.003


  L 12  λ=1e-01  shared dirs=  895  top-50 ρ held/in-sample=40.5/40.8  E_en←es=0.462  E_es←en=0.466  bias=-0.004


  L 14  λ=1e-01  shared dirs=  894  top-50 ρ held/in-sample=40.5/40.9  E_en←es=0.470  E_es←en=0.474  bias=-0.004


  L 18  λ=1e-02  shared dirs=  894  top-50 ρ held/in-sample=41.2/43.0  E_en←es=0.452  E_es←en=0.454  bias=-0.003


... (7 outputs left out)

[es] layer 18: AUROC hot vs other content words — U 0.73, U_raw 0.60, half-life 0.68 (shorter = harder), #tokens baseline 0.45; agreement with round trip ρ_s=0.28
   Me gusta mucho el duende: duende U=0.803 (half-life 0.4), gusta U=0.792 (half-life 0.5), mucho U=0.798 (half-life 0.4)


[es] layer 21: AUROC hot vs other content words — U 0.61, U_raw 0.52, half-life 0.48 (shorter = harder), #tokens baseline 0.45; agreement with round trip ρ_s=0.3
   Me gusta mucho el duende: duende U=0.844 (half-life 0.4), gusta U=0.809 (half-life 0.4), mucho U=0.814 (half-life 0.4)


[hi] layer 6: AUROC hot vs other content words — U 0.62, U_raw 0.48, half-life 0.54 (shorter = harder), #tokens baseline 0.62; agreement with round trip ρ_s=0.09


IndexError: invalid index to scalar variable.

*File Operation*: `download` on `/content/ut_results_Qwen_Qwen2_5_0_5B_Instruct.tar.gz`

In [ ]:
# Reload earlier stages' results, so this stage also works in a fresh kernel (after 00_setup.py).
A = load_json("A_detour.json") if have("A_detour.json") else None
ladder = load_json("B_ladder.json") if have("B_ladder.json") else None
B = load_json("B_roundtrip.json") if have("B_roundtrip.json") else None
summary = load_json("B_summary.json")["per_sentence"] if have("B_summary.json") else {}


assert C_FITS, 'run C2_spectrum.py in this same kernel first (it keeps the fitted bases in GPU memory)'

def decomposer(lang, l):
    F_ = C_FITS[(lang, l)]; fit = F_["fit"]
    idx = F_["shared"]
    A_r = fit["A"][:, idx].float(); D_r = (fit["Cxx_r"] @ fit["A"][:, idx]).float(); mx = fit["mx"].float()
    def split(h):
        xc = h.float().to(A_r.device) - mx; sh = (xc @ A_r) @ D_r.T
        return sh, xc - sh
    test = torch.randn(4, D_MODEL, device=A_r.device) * 3 + mx
    sh, _ = split(test); sh2, _ = split(sh + mx)
    print(f"  layer {l}: shared dims {len(idx)} of {D_MODEL}; oblique projection idempotence error {float((sh2 - sh).norm() / sh.norm()):.1e}")
    return split, mx

def ablate_fn(split, mx, which):
    def fn(h):
        sh, pr = split(h); dev = h.device
        if which == "private": return (mx + sh).to(dev)
        if which == "shared": return (mx + pr).to(dev)
        return mx.expand_as(sh).to(dev)
    return fn

def interchange_fn(split, mx, donor, which):
    def fn(h):
        sh, pr = split(h); dsh, dpr = split(donor); xc = sh + pr
        if which == "private": new = xc - pr + dpr
        elif which == "shared": new = xc - sh + dsh
        else: new = dsh + dpr
        return (new + mx).to(h.device)
    return fn

if RUN_D and "es" in C_RES and ladder and not have("D_mediation.json"):
    D_out = {}
    med_layers = sorted({l for (lg, l) in C_FITS if lg == "es"})
    for l in med_layers:
        with Timer(f"Part D: layer {l}"):
            split, mx = decomposer("es", l)
            # --- 1. ablation dissociation ---
            es_sents = [S for S in SENTENCES if S["lang"] == "es"]
            targets = []
            for S in es_sents:
                hi = hot_word_index(S)
                for k, (w, a, b) in enumerate(words_with_spans(S["text"])):
                    if k in hi or (len(w) >= 4 and S["cat"] == "control"): targets.append((S, w, k in hi, (a, b)))
            ask = [chat_prompt(f'{S["text"]}\n\nIn the Spanish sentence above, what does the word "{w}" mean here? Answer in one line.')
                   for S, w, _, _ in targets]
            expl = generate(ask, max_new_tokens=40)
            P, Cn, E, meta = [], [], [], []
            for (S, w, is_hot, span), ap, ex in zip(targets, ask, expl):
                tp = chat_prompt(rung_user("B1", S["text"], "es"))
                for task, prompt, cont in [("translate", tp, ladder[S["id"]]["B1"]), ("explain", ap, ex)]:
                    pos = positions_in_prompt(prompt, S["text"], span=span)
                    if task == "explain":                      # also the word quoted in the question
                        q = prompt.rfind(f'"{w}"') + 1; pos = sorted(set(pos) | set(positions_at(prompt, q, q + len(w))))
                    for cond in ["none", "private", "shared", "full"]:
                        P.append(prompt); Cn.append(cont)
                        E.append(None if cond == "none" else [(l, pos, ablate_fn(split, mx, cond))])
                        meta.append((S["id"], w, is_hot, task, cond))
            sc = score(P, Cn, edits=E)
            lp = {m: float(r["lp"].mean()) for m, r in zip(meta, sc)}
            abl = []
            for (S, w, is_hot, _), ex in zip(targets, expl):
                row = dict(sid=S["id"], word=w, hot=is_hot, explanation=ex)
                for task in ["translate", "explain"]:
                    base = lp[(S["id"], w, is_hot, task, "none")]
                    for cond in ["private", "shared", "full"]:
                        row[f"{task}_{cond}"] = base - lp[(S["id"], w, is_hot, task, cond)]   # drop in log-prob per token
                abl.append(row)
            # --- 2. interchange on sense pairs ---
            inter = []
            for SP in SENSE_PAIRS:
                runs = {}
                for me, other in [("a", "b"), ("b", "a")]:
                    pm = chat_prompt(rung_user("B1", SP[me], "es")); po = chat_prompt(rung_user("B1", SP[other], "es"))
                    pos_m = positions_in_prompt(pm, SP[me], SP["word"]); pos_o = positions_in_prompt(po, SP[other], SP["word"])
                    if len(pos_m) != len(pos_o): print(f"  skip {SP['id']}: token counts differ"); runs = None; break
                    donor = capture_prompt_states(po, l)[pos_o]
                    tmpl = SP[f"en_{me}"]; cands = [tmpl.format(SP["sense_a"]), tmpl.format(SP["sense_b"])]
                    conds = {"none": None, "private": interchange_fn(split, mx, donor, "private"),
                             "shared": interchange_fn(split, mx, donor, "shared"), "full": interchange_fn(split, mx, donor, "full")}
                    Ps, Cs, Es, keys = [], [], [], []
                    for cname, fn in conds.items():
                        for ci, c in enumerate(cands):
                            Ps.append(pm); Cs.append(c); Es.append(None if fn is None else [(l, pos_m, fn)]); keys.append((cname, ci))
                    r = score(Ps, Cs, edits=Es); tot = {k: float(x["lp"].sum()) for k, x in zip(keys, r)}
                    lo = {c: tot[(c, 0)] - tot[(c, 1)] for c in conds}          # log-odds sense_a vs sense_b
                    gen = {c: generate_with_edits(pm, [] if fn is None else [(l, pos_m, fn)]) for c, fn in conds.items()}
                    runs[me] = dict(logodds=lo, generations=gen)
                if runs is None: continue
                base_a, base_b = runs["a"]["logodds"]["none"], runs["b"]["logodds"]["none"]
                gap = base_b - base_a
                flip = {}
                for c in ["private", "shared", "full"]:
                    fa = (runs["a"]["logodds"][c] - base_a) / gap if abs(gap) > 1e-6 else float("nan")
                    fb = (runs["b"]["logodds"][c] - base_b) / (-gap) if abs(gap) > 1e-6 else float("nan")
                    flip[c] = float(np.nanmean([fa, fb]))
                inter.append(dict(id=SP["id"], base_gap=gap, flip=flip, runs=runs))
                print(f"  {SP['id']:11s} base log-odds a={base_a:+.1f} b={base_b:+.1f}  flip private={flip['private']:+.2f} "
                      f"shared={flip['shared']:+.2f} full={flip['full']:+.2f}")
            D_out[l] = dict(ablation=abl, interchange=inter)
    save_json("D_mediation.json", D_out)

D = load_json("D_mediation.json") if have("D_mediation.json") else None
if D:
    for l, d in D.items():
        ab = d["ablation"]
        def ratio(r, task, cond): f = r[f"{task}_full"]; return r[f"{task}_{cond}"] / f if abs(f) > 1e-6 else np.nan
        print(f"\nlayer {l}: effect of removing the PRIVATE part, as a share of removing the whole state (mean, 95% CI)")
        for lab, grp in [("hot words", [r for r in ab if r["hot"]]), ("control words", [r for r in ab if not r["hot"]])]:
            t = [ratio(r, "translate", "private") for r in grp]; e = [ratio(r, "explain", "private") for r in grp]
            ct, ce = boot_ci(t), boot_ci(e)
            print(f"  {lab:14s} n={len(grp):3d}  on translation {np.nanmean(t):.2f} [{ct[0]:.2f}, {ct[1]:.2f}] (median {np.nanmedian(t):.2f})"
                  f"   on explanation {np.nanmean(e):.2f} [{ce[0]:.2f}, {ce[1]:.2f}] (median {np.nanmedian(e):.2f})")
        hot = [r for r in ab if r["hot"]]
        gap = [ratio(r, "explain", "private") - ratio(r, "translate", "private") for r in hot]
        cg = boot_ci(gap)
        print(f"  hot words: explanation minus translation = {np.nanmean(gap):+.2f} [{cg[0]:+.2f}, {cg[1]:+.2f}] (median {np.nanmedian(gap):+.2f}); positive = understood but not translated")
    l = list(D)[0]; ab = D[l]["ablation"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    ax = axes[0]
    for grp, col, lab in [([r for r in ab if not r["hot"]], EASY, "ordinary words"), ([r for r in ab if r["hot"]], HARD, "hard words")]:
        x = [r["translate_private"] / r["translate_full"] if abs(r["translate_full"]) > 1e-6 else np.nan for r in grp]
        y = [r["explain_private"] / r["explain_full"] if abs(r["explain_full"]) > 1e-6 else np.nan for r in grp]
        x, y = np.clip(x, -0.5, 1.5), np.clip(y, -0.5, 1.5)     # extreme ratios (tiny denominators) pinned to the edge
        ax.scatter(x, y, color=col, s=28, alpha=0.8, label=lab)
        for r, xi, yi in zip(grp, x, y):
            if r["hot"]: ax.text(xi, yi, " " + r["word"], fontsize=7, style="italic", color=col)
    ax.plot([-1, 2], [-1, 2], color="#bbb", lw=0.8); ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel("harm to the TRANSLATION\n(removing the language-private part, share of removing everything)")
    ax.set_ylabel("harm to EXPLAINING the word")
    ax.set_title("Above the diagonal: the model understands more than it translates", loc="left", fontsize=9); ax.legend(frameon=False, fontsize=8)
    ax = axes[1]; it = D[l]["interchange"]; w = 0.26
    for k, (c, col) in enumerate([("private", HARD), ("shared", EASY), ("full", GREY)]):
        ax.bar(np.arange(len(it)) + (k - 1) * w, [x["flip"][c] for x in it], w, color=col, label=f"swap the {c} part" if c != "full" else "swap everything")
    ax.set_xticks(np.arange(len(it))); ax.set_xticklabels([x["id"] for x in it], rotation=30, ha="right", style="italic")
    ax.axhline(0, color="#888", lw=0.8); ax.axhline(1, color="#888", lw=0.8, ls=":")
    ax.set_ylabel("how far the English moves to the other sense"); ax.legend(frameon=False, fontsize=8)
    ax.set_title("Swapping part of one word's state between two senses", loc="left", fontsize=9)
    savefig(fig, f"D_mediation_L{l}")
    print("\nExample generations under interchange (frame a, patched from frame b):")
    for x in D[l]["interchange"][:8]:
        g = x["runs"]["a"]["generations"]; print(f"  {x['id']:11s} none: {g['none'][:60]!r} | private: {g['private'][:60]!r} | full: {g['full'][:60]!r}")

pack()


▶ Part D: layer 18 ...
  layer 18: shared dims 894 of 896; oblique projection idempotence error 4.2e-07


  cura        base log-odds a=-15.8 b=-21.9  flip private=+0.00 shared=+0.25 full=+0.24


  cometa      base log-odds a=+17.5 b=+17.8  flip private=-0.12 shared=+1.62 full=+1.46


  pendiente   base log-odds a=-19.5 b=-21.5  flip private=+0.01 shared=+0.46 full=+0.43


  corte       base log-odds a=+6.7 b=-5.6  flip private=-0.01 shared=+0.55 full=+0.54


  frente      base log-odds a=+6.7 b=-1.5  flip private=+0.01 shared=+0.39 full=+0.39


... (9 outputs left out)

  aburrido    base log-odds a=+0.1 b=-4.9  flip private=-0.01 shared=+0.08 full=+0.08


  duende_ctx  base log-odds a=-4.7 b=-7.9  flip private=+0.01 shared=-0.09 full=-0.07
✓ Part D: layer 21: 40s

layer 18: effect of removing the PRIVATE part, as a share of removing the whole state (mean, 95% CI)
  hot words      n= 28  on translation -0.32 [-0.82, 0.05] (median -0.05)   on explanation 0.17 [-0.30, 0.74] (median 0.01)
  control words  n= 24  on translation -1.90 [-4.79, 0.02] (median 0.01)   on explanation 0.09 [-0.23, 0.58] (median -0.00)
  hot words: explanation minus translation = +0.49 [-0.23, +1.32] (median +0.09); positive = understood but not translated

layer 21: effect of removing the PRIVATE part, as a share of removing the whole state (mean, 95% CI)
  hot words      n= 28  on translation -0.08 [-2.25, 1.89] (median 0.04)   on explanation -0.01 [-0.23, 0.22] (median 0.03)
  control words  n= 24  on translation -0.36 [-1.61, 0.37] (median 0.05)   on explanation -0.16 [-0.82, 0.37] (median 0.05)
  hot words: explanation minus translation = +0.07 [-1.95, +2.39] (


Example generations under interchange (frame a, patched from frame b):
  cura        none: "I'm concerned about curing." | private: "I'm concerned about cure." | full: "I'm concerned about cure."
  cometa      none: 'We observe the comet.' | private: 'We observe the comet.' | full: 'We observe the comet.'
  pendiente   none: 'I saw the slope.' | private: 'I saw the slope.' | full: 'I saw the slope.'
  corte       none: 'I was surprised by the cut.' | private: 'I was surprised by the cut.' | full: 'I was surprised by the haircut.'
  frente      none: 'He signaled the front.' | private: 'He signaled the front.' | full: 'He signaled the front.'
  papa        none: 'I like the papaya.' | private: 'I like the papaya.' | full: 'I like the potato.'
  aburrido    none: 'Juan is boring.' | private: 'Juan is boring.' | full: 'Juan is bored.'
  duende_ctx  none: 'In that childish tale, what I most enjoy is the spirit.' | private: 'In that childish tale, what I most enjoy is the spirit.' | full: 

PACKED /content/ut_results_Qwen_Qwen2_5_0_5B_Instruct.tar.gz


*File Operation*: `download` on `/content/ut_results_Qwen_Qwen2_5_0_5B_Instruct.tar.gz`

In [ ]:
# Reload earlier stages' results, so this stage also works in a fresh kernel (after 00_setup.py).
A = load_json("A_detour.json") if have("A_detour.json") else None
ladder = load_json("B_ladder.json") if have("B_ladder.json") else None
B = load_json("B_roundtrip.json") if have("B_roundtrip.json") else None
summary = load_json("B_summary.json")["per_sentence"] if have("B_summary.json") else {}
D = load_json("D_mediation.json") if have("D_mediation.json") else None
C_RES = {lang: load_json(f"C_spectrum_{lang}.json") for lang in PAIRS if have(f"C_spectrum_{lang}.json")}
BEST_LAYER = {lang: max(rows, key=lambda r: r["top50_held"])["layer"] for lang, rows in C_RES.items()}
C_SPANS = {lang: {int(k): v for k, v in load_json(f"C_spans_{lang}.json").items()} for lang in C_RES if have(f"C_spans_{lang}.json")}


lines = [f"# Summary: {MODEL_ID}", "", f"{N_LAYERS} layers, d = {D_MODEL}. Every number is inside this one model.", ""]
if A:
    lines += ["## A. English detour", f"- English peaks at layer {A['detour_peak_layer']}/{N_LAYERS}; Italian overtakes at {A['italian_overtakes_at']}.",
              f"- Peak mean P(English) = {np.array(A['P_en']).mean(0).max():.3f} over {len(A['P_en'])} words.", ""]
if have("B_summary.json"):
    bs = load_json("B_summary.json"); ps = bs["per_sentence"]
    lines += ["## B. Round trip (masked)", f"- Tolerance ε = {bs['epsilon']:.3f}; hot-word AUROC = {bs['auroc_hot_words']:.3f}."]
    for c in ["untranslatable", "construction", "idiom", "control"]:
        v = [x for x in ps.values() if x["cat"] == c]
        if v: lines.append(f"- {c}: U = {np.mean([x['U'] for x in v]):.2f} doublings, borrowing buys {np.nanmean([x['borrow_gain'] for x in v]):.3f} (masked: {np.nanmean([x['borrow_gain_masked'] for x in v]):.3f}).")
    fa = B["duende"]; wl = dict(zip(fa["words"], fa["rungs"]["B1"]["word_left_masked"]))
    lines += [f"- *Me gusta mucho el duende* → {ladder['duende']['B1']!r}; leftover duende {wl.get('duende', np.nan):.2f}, gusta {wl.get('gusta', np.nan):.2f}, mucho {wl.get('mucho', np.nan):.2f}.",
              "- Sense recovered for *duende* (standard / keep / gloss / footnote / thick): " + " / ".join(f"{fa['sense']['duende']['norm'][r]:.2f}" for r in ["B1", "B2", "B3", "B5", "B6"]), ""]
for lang in C_RES:
    r = next(x for x in C_RES[lang] if x["layer"] == BEST_LAYER[lang])
    lines += [f"## C. Spectrum {lang}–en (layer {r['layer']})",
              f"- {r['n_shared']} shared directions above the null (of {D_MODEL}); fit on {r['n_fit_pairs']} word pairs.",
              f"- English predicted from {lang}: {r['E_en_from_x']:.3f}; {lang} predicted from English: {r['E_x_from_en']:.3f}; bias {r['bias']:+.3f} (standardised {r['bias_std']:+.3f})."]
    if C_SPANS.get(lang):
        au = C_SPANS[lang][BEST_LAYER[lang]]["auroc"]; ag = C_SPANS[lang][BEST_LAYER[lang]]["spearman_with_roundtrip_A"]
        lines.append(f"- AUROC hot vs other words: U {au['U']:.2f}, U_raw {au['U_raw']:.2f}, #tokens baseline {au['ntok']:.2f}; agreement with round trip {ag}.")
    lines.append("")
if D:
    for l, d in D.items():
        ab = d["ablation"]; hot = [r for r in ab if r["hot"]]
        f = lambda r, t: r[f"{t}_private"] / r[f"{t}_full"] if abs(r[f"{t}_full"]) > 1e-6 else np.nan
        lines += [f"## D. Mediation (layer {l})",
                  f"- Hot words, private-part share of harm: translation {np.nanmean([f(r, 'translate') for r in hot]):.2f}, explanation {np.nanmean([f(r, 'explain') for r in hot]):.2f}.",
                  "- Flip fractions (private / shared / full): " + "; ".join(f"{x['id']} {x['flip']['private']:+.2f}/{x['flip']['shared']:+.2f}/{x['flip']['full']:+.2f}" for x in d["interchange"]), ""]
open(os.path.join(OUT, "SUMMARY.md"), "w").write("\n".join(lines))
print("\n".join(lines))

pack()


# Summary: Qwen/Qwen2.5-0.5B-Instruct

24 layers, d = 896. Every number is inside this one model.

## A. English detour
- English peaks at layer 21/24; Italian overtakes at 24.
- Peak mean P(English) = 0.051 over 28 words.

## B. Round trip (masked)
- Tolerance ε = 0.439; hot-word AUROC = 0.605.
- untranslatable: U = 1.07 doublings, borrowing buys 0.063 (masked: 0.047).
- construction: U = 2.49 doublings, borrowing buys 0.141 (masked: -0.069).
- idiom: U = 0.00 doublings, borrowing buys -0.013 (masked: -0.013).
- control: U = 0.61 doublings, borrowing buys 0.058 (masked: -0.009).
- *Me gusta mucho el duende* → 'I like much the spirit.'; leftover duende 1.00, gusta 0.00, mucho 0.00.
- Sense recovered for *duende* (standard / keep / gloss / footnote / thick): -0.54 / -0.54 / -0.76 / -0.65 / -0.30

## C. Spectrum es–en (layer 18)
- 894 shared directions above the null (of 896); fit on 25191 word pairs.
- English predicted from es: 0.452; es predicted from English: 0.454; bias -0.003 (stan

PACKED /content/ut_results_Qwen_Qwen2_5_0_5B_Instruct.tar.gz


*File Operation*: `download` on `/content/ut_results_Qwen_Qwen2_5_0_5B_Instruct.tar.gz`

**Session Created**: 2026-11-21 18:02:10
- Endpoint: `gpu-a100-hm-euw4a0-3sdddqnc4gx2c`
- Hardware: `A100`

In [ ]:
import subprocess; print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],capture_output=True,text=True).stdout)


name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB



### Automation: install (2026-11-21 18:15:56)

In [ ]:

import subprocess, sys
def install():
    packages = ['-r', '/content/requirements.txt']
    try:
        subprocess.check_call(['uv', 'pip', 'install', '--system'] + packages)
        print('Installation Complete (via uv)!')
    except:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + packages)
        print('Installation Complete (via pip)!')
install()


In [ ]:
# Result of previous automation

Installation Complete (via uv)!


In [ ]:
# 00_setup.py: run once per Colab session (via `colab exec -f`), before any stage.
# Settings can be overridden by executing assignments first, e.g.  echo "MODEL_ID = '...'" | colab exec -s ut
g = globals()
MODEL_ID    = g.get("MODEL_ID", "unsloth/Llama-3.3-70B-Instruct-bnb-4bit")   # alt: unsloth/Qwen2.5-72B-Instruct-bnb-4bit; dry run: Qwen/Qwen2.5-0.5B-Instruct
DRY_RUN     = g.get("DRY_RUN", any(s in MODEL_ID for s in ["0.5B", "1.5B", "tiny"]))
OUT_ROOT    = g.get("OUT_ROOT", "/content/ut_results")       # on the VM; fetched with `colab download` (see pack())
LOCAL_CACHE = g.get("LOCAL_CACHE", "/content/cache")         # big covariance files; stay on the VM
RUN_A, RUN_B, RUN_C, RUN_D = g.get("RUN_A", True), g.get("RUN_B", True), g.get("RUN_C", True), g.get("RUN_D", True)
PAIRS       = g.get("PAIRS", ["es", "hi"])
N_SENT_FIT  = g.get("N_SENT_FIT", 2000 if DRY_RUN else 30000)
N_SENT_HELD = g.get("N_SENT_HELD", 200 if DRY_RUN else 1200)
LAMBDAS     = g.get("LAMBDAS", [1e-3, 1e-2, 1e-1])
SEED        = g.get("SEED", 0)
CHAT_DATE   = "26 Jul 2024"
MIN_GPU_GB  = g.get("MIN_GPU_GB", 0 if DRY_RUN else 75)

import subprocess, sys, matplotlib
matplotlib.use("Agg")
import torch
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0); print(f"GPU: {_p.name}, {_p.total_memory/1e9:.0f} GB")
    if _p.total_memory / 1e9 < MIN_GPU_GB:
        raise RuntimeError(f"This GPU has {_p.total_memory/1e9:.0f} GB; the 4-bit 70B model needs an 80 GB card. "
                           "Stop this session and create one with --gpu H100 (or an 80 GB A100).")
else:
    print("WARNING: no GPU visible")
try:
    import spacy; spacy.load("es_core_news_sm")
except Exception:
    subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm", "-q"], check=False)


import os, re, json, math, time, random, difflib, gc
from collections import defaultdict
import numpy as np
import torch
import matplotlib.pyplot as plt

def slug(s): return re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")
OUT = os.path.join(OUT_ROOT, slug(MODEL_ID)); os.makedirs(OUT, exist_ok=True)
FIG = os.path.join(OUT, "figures"); os.makedirs(FIG, exist_ok=True)
CACHE = os.path.join(LOCAL_CACHE, slug(MODEL_ID)); os.makedirs(CACHE, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def _np(o):
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, torch.Tensor): return o.tolist()
    raise TypeError(type(o))
def save_json(name, obj):
    with open(os.path.join(OUT, name), "w") as f: json.dump(obj, f, indent=1, default=_np, ensure_ascii=False)
def load_json(name):
    with open(os.path.join(OUT, name)) as f: return json.load(f)
def have(name): return os.path.exists(os.path.join(OUT, name))

class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); print(f"▶ {self.label} ..."); return self
    def __exit__(self, *a): print(f"✓ {self.label}: {time.time()-self.t:.0f}s")

def auroc(pos, neg):
    """Probability that a random positive scores above a random negative (ties count half)."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    pos, neg = pos[~np.isnan(pos)], neg[~np.isnan(neg)]
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    allv = np.concatenate([pos, neg]); r = rankdata(allv)
    return float((r[:len(pos)].sum() - len(pos)*(len(pos)+1)/2) / (len(pos)*len(neg)))
def rankdata(a):
    a = np.asarray(a, float); order = np.argsort(a, kind="mergesort"); r = np.empty(len(a))
    sa = a[order]; i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and sa[j+1] == sa[i]: j += 1
        r[order[i:j+1]] = (i + j) / 2 + 1; i = j + 1
    return r
def spearman(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float); m = ~(np.isnan(x) | np.isnan(y))
    if m.sum() < 3: return float("nan")
    return float(np.corrcoef(rankdata(x[m]), rankdata(y[m]))[0, 1])
def boot_ci(x, n=2000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed); m = rng.choice(x, (n, len(x))).mean(1)
    return (float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5)))

# Visual language shared with the earlier figures
EASY, HARD, LOAN, GREY, INK = "#2f7d6d", "#c4572e", "#5b4fc4", "#d9d6cf", "#2b2b2b"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
def savefig(fig, name):
    fig.savefig(os.path.join(FIG, name + ".png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG, name + ".svg"), bbox_inches="tight")
    plt.show(); plt.close(fig)
print("results ->", OUT)

LANG_NAME = {"es": "Spanish", "pt": "Portuguese", "it": "Italian", "de": "German", "hi": "Hindi", "en": "English"}

SENTENCES = [
    # --- untranslatable words in context ---
    dict(id="duende", lang="es", cat="untranslatable", text="Me gusta mucho el duende.",
         hot={"duende": "the mysterious, almost demonic power and emotional intensity of a great artistic performance, as in flamenco (Lorca's sense)",
              "gusta": "it pleases me: the thing is the subject and I am the one affected, like having a taste for it"}),
    dict(id="sobremesa", lang="es", cat="untranslatable", text="Después de comer nos quedamos dos horas de sobremesa.",
         hot={"sobremesa": "the relaxed time spent talking at the table after a meal has finished"}),
    dict(id="madrugada", lang="es", cat="untranslatable", text="Llegamos a casa de madrugada, muertos de sueño.",
         hot={"madrugada": "the small hours, between midnight and dawn"}),
    dict(id="estrenar", lang="es", cat="untranslatable", text="Hoy estreno los zapatos que me regalaste.",
         hot={"estreno": "I am using or wearing something for the very first time"}),
    dict(id="verguenza", lang="es", cat="untranslatable", text="Sentí vergüenza ajena durante todo su discurso.",
         hot={"vergüenza ajena": "embarrassment felt on behalf of someone else who is embarrassing themselves"}),
    dict(id="tertulia", lang="es", cat="untranslatable", text="Los jueves vamos a la tertulia del café de la esquina.",
         hot={"tertulia": "a regular informal gathering of people for conversation, often literary or political"}),
    # --- constructions ---
    dict(id="olvido", lang="es", cat="construction", text="Se me olvidó la llave en el coche.",
         hot={"Se me olvidó": "I forgot it, but framed as something that happened to me by accident, as if the key forgot itself on me"}),
    dict(id="estar_aburrido", lang="es", cat="construction", text="Juan está aburrido en la reunión.",
         hot={"está aburrido": "is bored, a temporary state (not 'is boring')"}),
    dict(id="ser_aburrido", lang="es", cat="construction", text="Juan es aburrido en las reuniones.",
         hot={"es aburrido": "is boring, a lasting characteristic (not 'is bored')"}),
    dict(id="ojala", lang="es", cat="construction", text="Ojalá llueva mañana por la tarde.",
         hot={"Ojalá": "a strong hope or wish, 'if only', originally 'may God will it'"}),
    # --- idioms ---
    dict(id="pelo", lang="es", cat="idiom", text="Creo que me estás tomando el pelo.",
         hot={"tomando el pelo": "teasing me, pulling my leg"}),
    dict(id="nubes", lang="es", cat="idiom", text="Mi hermana siempre está en las nubes.",
         hot={"en las nubes": "daydreaming, absent-minded"}),
    dict(id="pelos", lang="es", cat="idiom", text="Mi abuela no tiene pelos en la lengua.",
         hot={"no tiene pelos en la lengua": "speaks very frankly, without holding anything back"}),
    # --- controls ---
    dict(id="tren", lang="es", cat="control", text="El tren sale a las ocho de la mañana.", hot={}),
    dict(id="hermano", lang="es", cat="control", text="Mi hermano vive en un piso pequeño.", hot={}),
    dict(id="pan", lang="es", cat="control", text="Compré pan y leche en la tienda.", hot={}),
    dict(id="reunion", lang="es", cat="control", text="La reunión empieza a las diez en punto.", hot={}),
    dict(id="gatos", lang="es", cat="control", text="Tengo dos gatos negros y un perro.", hot={}),
    dict(id="invierno", lang="es", cat="control", text="En invierno hace mucho frío en el norte.", hot={}),
    # --- other European languages (Part B only) ---
    dict(id="saudade", lang="pt", cat="untranslatable", text="Tenho muita saudade da minha avó.",
         hot={"saudade": "a deep, bittersweet longing for someone or something absent, that may never return"}),
    dict(id="magari", lang="it", cat="construction", text="Magari fosse vero!",
         hot={"Magari": "if only (a wistful wish)"}),
    dict(id="fernweh", lang="de", cat="untranslatable", text="Im Winter habe ich immer Fernweh.",
         hot={"Fernweh": "an ache to travel to faraway places, the opposite of homesickness"}),
    # --- Hindi (DRAFTS: to be checked by a native speaker) ---
    dict(id="hi_pasand", lang="hi", cat="construction", draft=True, text="मुझे चाय बहुत पसंद है।",
         hot={"मुझे": "to me (dative experiencer: the tea is liked by me, like Spanish 'me gusta')"}),
    dict(id="hi_jugaad", lang="hi", cat="untranslatable", draft=True, text="उसने जुगाड़ से पुरानी गाड़ी ठीक कर दी।",
         hot={"जुगाड़": "a clever improvised fix made with whatever is at hand"}),
    dict(id="hi_mann", lang="hi", cat="untranslatable", draft=True, text="आज पढ़ाई में मेरा मन नहीं लग रहा।",
         hot={"मन": "heart-mind: the seat of feeling, will and attention together"}),
    dict(id="hi_apnapan", lang="hi", cat="untranslatable", draft=True, text="इस घर में मुझे बहुत अपनापन लगता है।",
         hot={"अपनापन": "a warm sense of belonging and intimate familiarity, of being among one's own"}),
    dict(id="hi_train", lang="hi", cat="control", draft=True, text="ट्रेन सुबह आठ बजे चलती है।", hot={}),
    dict(id="hi_bhai", lang="hi", cat="control", draft=True, text="मेरा भाई दिल्ली में रहता है।", hot={}),
]

# Sense minimal pairs for Part D: same word, two contexts, two senses.
# The English candidates differ only in the critical word, so their log-odds measures which sense the translation carries.
SENSE_PAIRS = [
    dict(id="cura", word="cura", a="Me preocupa el cura.", b="Me preocupa la cura.",
         en_a="I am worried about the {}.", en_b="I am worried about the {}.", sense_a="priest", sense_b="cure"),
    dict(id="cometa", word="cometa", a="Miramos el cometa.", b="Miramos la cometa.",
         en_a="We looked at the {}.", en_b="We looked at the {}.", sense_a="comet", sense_b="kite"),
    dict(id="pendiente", word="pendiente", a="Vi el pendiente.", b="Vi la pendiente.",
         en_a="I saw the {}.", en_b="I saw the {}.", sense_a="earring", sense_b="slope"),
    dict(id="corte", word="corte", a="Me sorprendió el corte.", b="Me sorprendió la corte.",
         en_a="The {} surprised me.", en_b="The {} surprised me.", sense_a="cut", sense_b="court"),
    dict(id="frente", word="frente", a="Señaló el frente.", b="Señaló la frente.",
         en_a="He pointed at the {}.", en_b="He pointed at the {}.", sense_a="front", sense_b="forehead"),
    dict(id="papa", word="papa", a="Me gusta el papa.", b="Me gusta la papa.",
         en_a="I like the {}.", en_b="I like the {}.", sense_a="pope", sense_b="potato"),
    dict(id="aburrido", word="aburrido", a="Juan es aburrido.", b="Juan está aburrido.",
         en_a="Juan is {}.", en_b="Juan is {}.", sense_a="boring", sense_b="bored"),
    dict(id="duende_ctx", word="duende", a="En ese cuento infantil, lo que más me gusta es el duende.",
         b="En ese cante flamenco, lo que más me gusta es el duende.",
         en_a="In that children's story, what I like most is the {}.",
         en_b="In that flamenco song, what I like most is the {}.", sense_a="goblin", sense_b="soul"),
]

SID = {S["id"]: S for S in SENTENCES}

# Part A: Spanish -> Italian word translation, with the English equivalent for the logit lens
DETOUR_SHOTS = [("perro", "cane"), ("ventana", "finestra"), ("mesa", "tavolo"), ("leche", "latte")]
DETOUR_WORDS = [("árbol", "albero", "tree"), ("pájaro", "uccello", "bird"), ("manzana", "mela", "apple"),
    ("cuchillo", "coltello", "knife"), ("calle", "strada", "street"), ("niño", "bambino", "child"),
    ("nube", "nuvola", "cloud"), ("queso", "formaggio", "cheese"), ("llave", "chiave", "key"),
    ("zapato", "scarpa", "shoe"), ("camisa", "camicia", "shirt"), ("mantequilla", "burro", "butter"),
    ("cerdo", "maiale", "pig"), ("caballo", "cavallo", "horse"), ("hermano", "fratello", "brother"),
    ("pescado", "pesce", "fish"), ("cuchara", "cucchiaio", "spoon"), ("tenedor", "forchetta", "fork"),
    ("silla", "sedia", "chair"), ("puerta", "porta", "door"), ("ciudad", "città", "city"),
    ("rodilla", "ginocchio", "knee"), ("hielo", "ghiaccio", "ice"), ("lluvia", "pioggia", "rain"),
    ("nieve", "neve", "snow"), ("hoja", "foglia", "leaf"), ("lengua", "lingua", "tongue"),
    ("oreja", "orecchio", "ear"), ("huevo", "uovo", "egg"), ("ajo", "aglio", "garlic"),
    ("cerebro", "cervello", "brain"), ("espejo", "specchio", "mirror")]

from transformers import AutoTokenizer, AutoModelForCausalLM
with Timer(f"load {MODEL_ID}"):
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    _dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=_dtype, device_map="auto")
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=_dtype, device_map="auto")
    model.eval()

def _get_attr_path(obj, path):
    for p in path.split("."): obj = getattr(obj, p)
    return obj
def find_decoder(m):
    for base in ["model", "model.model", "model.language_model", "transformer"]:
        try:
            b = _get_attr_path(m, base)
            layers = getattr(b, "layers", None) or getattr(b, "h", None)
            norm = getattr(b, "norm", None) or getattr(b, "ln_f", None)
            if layers is not None and norm is not None: return b, layers, norm
        except AttributeError: pass
    raise ValueError("could not find decoder layers")

BASE, LAYERS_MOD, FINAL_NORM = find_decoder(model)
LM_HEAD = model.get_output_embeddings()
N_LAYERS = len(LAYERS_MOD)
D_MODEL = model.config.hidden_size if hasattr(model.config, "hidden_size") else model.config.text_config.hidden_size
INPUT_DEV = model.get_input_embeddings().weight.device
BOS = [tok.bos_token_id] if tok.bos_token_id is not None else []
print(f"{N_LAYERS} layers, d={D_MODEL}, BOS={BOS}, input device {INPUT_DEV}")
if torch.cuda.is_available(): print(f"GPU memory in use: {torch.cuda.memory_allocated()/1e9:.1f} GB")

WORD_RE = re.compile(r"[\wऀ-ॣ०-ॿ]+(?:['’]\w+)?")
def words_with_spans(text): return [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(text)]
def enc(text, offsets=False): return tok(text, add_special_tokens=False, return_offsets_mapping=offsets)
def ntok(text): return len(enc(text)["input_ids"])

def chat_prompt(user, prefix=""):
    msgs = [{"role": "user", "content": user}]
    try: p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, date_string=CHAT_DATE)
    except Exception: p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return p + prefix

def tokens_in_span(offsets, a, b):
    return [i for i, (x, y) in enumerate(offsets) if y > x and y > a and x < b]

def find_phrase(text, phrase):
    m = re.search(r"(?<!\w)" + re.escape(phrase) + r"(?!\w)", text, flags=re.I)
    return (m.start(), m.end()) if m else None

def positions_at(prompt, a, b):
    return tokens_in_span(enc(prompt, offsets=True)["offset_mapping"], a, b)

def positions_in_prompt(prompt, sentence, phrase=None, span=None):
    """Token positions of `phrase` (or of the character span `span`) inside `sentence`, inside `prompt`."""
    base = prompt.rfind(sentence); assert base >= 0, "sentence not in prompt"
    sp = span or find_phrase(sentence, phrase); assert sp is not None, f"{phrase!r} not in {sentence!r}"
    return positions_at(prompt, base + sp[0], base + sp[1])

class _Stop(Exception): pass

def make_edit_hook(edits_by_b):
    """edits_by_b: {batch index: [(positions LongTensor, fn)]}; fn maps states [k,d] -> new states [k,d]."""
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        if h.shape[1] <= 1: return None              # cached decoding step: leave untouched
        for b, items in edits_by_b.items():
            for pos, fn in items:
                pos = pos.to(h.device)
                if int(pos.max()) < h.shape[1]:
                    h[b, pos] = fn(h[b, pos]).to(h.dtype)
        return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
    return hook

def _register_edits(edits_for_batch):
    """edits_for_batch: list (one per batch row) of lists of (layer, positions, fn)."""
    per_layer = defaultdict(lambda: defaultdict(list))
    for b, items in enumerate(edits_for_batch):
        for (layer, pos, fn) in (items or []):
            per_layer[layer][b].append((torch.as_tensor(pos, dtype=torch.long), fn))
    return [LAYERS_MOD[layer - 1].register_forward_hook(make_edit_hook(dict(eb))) for layer, eb in per_layer.items()]

@torch.no_grad()
def score(prompts, conts, batch=8, edits=None):
    """Teacher-forced log p of each continuation given its prompt.
    edits: optional list (per item) of [(layer, prompt token positions, fn)]; 'layer l' = hidden_states[l] = output of block l-1.
    Returns per item: dict(lp=array over continuation tokens, offsets=char offsets of those tokens in the continuation)."""
    res = [None] * len(prompts)
    for i in range(0, len(prompts), batch):
        idx = list(range(i, min(i + batch, len(prompts))))
        seqs, spans, offs = [], [], []
        for j in idx:
            p = enc(prompts[j])["input_ids"]; c = enc(conts[j], offsets=True)
            seqs.append(p + c["input_ids"]); spans.append((len(p), len(p) + len(c["input_ids"]))); offs.append(c["offset_mapping"])
        L = max(map(len, seqs))
        ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long); am = torch.zeros_like(ids)
        for b, s in enumerate(seqs): ids[b, :len(s)] = torch.tensor(s); am[b, :len(s)] = 1
        handles = _register_edits([edits[j] for j in idx]) if edits is not None else []
        try:
            out = BASE(input_ids=ids.to(INPUT_DEV), attention_mask=am.to(INPUT_DEV))
        finally:
            for h in handles: h.remove()
        H = out.last_hidden_state
        for b, j in enumerate(idx):
            s, e = spans[b]
            if e <= s: res[j] = dict(lp=np.zeros(0), offsets=[]); continue
            logits = LM_HEAD(H[b, s - 1:e - 1].to(LM_HEAD.weight.device)).float()
            tgt = ids[b, s:e].to(logits.device)
            res[j] = dict(lp=torch.log_softmax(logits, -1).gather(-1, tgt[:, None])[:, 0].cpu().numpy(), offsets=offs[b])
        del out, H
    return res

@torch.no_grad()
def generate(prompts, max_new_tokens=64, batch=16):
    tok.padding_side = "left"; outs = []
    for i in range(0, len(prompts), batch):
        e = tok(prompts[i:i + batch], add_special_tokens=False, return_tensors="pt", padding=True).to(INPUT_DEV)
        g = model.generate(**e, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                           top_k=None, pad_token_id=tok.pad_token_id)
        outs += [tok.decode(r, skip_special_tokens=True).strip() for r in g[:, e["input_ids"].shape[1]:]]
    tok.padding_side = "right"
    return outs

@torch.no_grad()
def generate_with_edits(prompt, edits, max_new_tokens=40):
    """Greedy generation for one prompt, with edits applied to the prompt positions (prefill only)."""
    e = tok(prompt, add_special_tokens=False, return_tensors="pt").to(INPUT_DEV)
    handles = _register_edits([edits])
    try:
        g = model.generate(**e, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                           top_k=None, pad_token_id=tok.pad_token_id)
    finally:
        for h in handles: h.remove()
    return tok.decode(g[0, e["input_ids"].shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def capture(ids, am, layers):
    """Residual states at `layers` (dict layer -> [B,T,d]); stops the forward pass after the deepest one."""
    store, deepest = {}, max(layers)
    def mk(l):
        def hook(m, i, o):
            store[l] = (o[0] if isinstance(o, tuple) else o).detach()
            if l == deepest: raise _Stop()
        return hook
    hs = [LAYERS_MOD[l - 1].register_forward_hook(mk(l)) for l in layers]
    try: BASE(input_ids=ids, attention_mask=am)
    except _Stop: pass
    finally:
        for h in hs: h.remove()
    return store

@torch.no_grad()
def capture_prompt_states(prompt, layer):
    e = tok(prompt, add_special_tokens=False, return_tensors="pt").to(INPUT_DEV)
    return capture(e["input_ids"], e["attention_mask"], [layer])[layer][0]

def encode_raw(text):
    e = enc(text, offsets=True)
    return BOS + e["input_ids"], [(0, 0)] * len(BOS) + list(e["offset_mapping"])

print("helpers ready")

RUNGS = ["B0", "B1", "B2", "B3", "B5", "B6"]
RUNG_NAME = {"B0": "word for word", "B1": "standard", "B2": "keep the word", "B3": "inline gloss",
             "B5": "footnote", "B6": "thick translation"}
NON_BORROW = ["B0", "B1", "B3", "B5", "B6"]
MAXTOK = {"B0": 64, "B1": 64, "B2": 72, "B3": 120, "B5": 150, "B6": 260}

def rung_user(r, s, L):
    n = LANG_NAME[L]
    return {
        "B0": f"Give a word-for-word English gloss of this {n} sentence: one English word for each {n} word, in the original order. Reply with the gloss only.\n\n{s}",
        "B1": f"Translate this {n} sentence into natural English. Reply with the translation only.\n\n{s}",
        "B2": f"Translate this {n} sentence into English, but keep any word that has no good English equivalent in the original {n}, in italics with *asterisks*. Reply with the translation only.\n\n{s}",
        "B3": f"Translate this {n} sentence into English. After each word or phrase that is hard to translate, add a short gloss in square brackets (at most 8 words). Do not quote the {n} words. Reply with the translation only.\n\n{s}",
        "B5": f"Translate this {n} sentence into English, then add one footnote (at most 40 words) explaining what the translation loses. Do not quote the {n} words. Format: the translation, a blank line, then 'Note: ...'.\n\n{s}",
        "B6": f"Translate this {n} sentence into English, then write one paragraph (at most 120 words) opening up the meanings, connotations and grammar of its hard parts, so that an English reader understands what the original says. Do not quote the {n} words.\n\n{s}",
    }[r]



def hot_word_index(S):
    idx = set()
    for span in S["hot"]:
        sp = find_phrase(S["text"], span)
        if sp: idx |= {k for k, (_, a, b) in enumerate(words_with_spans(S["text"])) if b > sp[0] and a < sp[1]}
    return idx




import tarfile
def pack():
    """Bundle everything under OUT into one archive for `colab download`."""
    p = f"{OUT_ROOT.rstrip('/')}_{slug(MODEL_ID)}.tar.gz"
    with tarfile.open(p, "w:gz") as t: t.add(OUT, arcname=slug(MODEL_ID))
    print("PACKED", p)


print("setup done:", MODEL_ID)


GPU: NVIDIA A100-SXM4-80GB, 85 GB


results -> /content/ut_results/unsloth_Llama_3_3_70B_Instruct_bnb_4bit


▶ load unsloth/Llama-3.3-70B-Instruct-bnb-4bit ...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

... (6 outputs left out)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

✓ load unsloth/Llama-3.3-70B-Instruct-bnb-4bit: 151s
80 layers, d=8192, BOS=[128000], input device cuda:0
GPU memory in use: 39.5 GB
helpers ready
setup done: unsloth/Llama-3.3-70B-Instruct-bnb-4bit


In [ ]:
# Reload earlier stages' results, so this stage also works in a fresh kernel (after 00_setup.py).
A = load_json("A_detour.json") if have("A_detour.json") else None
ladder = load_json("B_ladder.json") if have("B_ladder.json") else None
B = load_json("B_roundtrip.json") if have("B_roundtrip.json") else None
summary = load_json("B_summary.json")["per_sentence"] if have("B_summary.json") else {}


if RUN_A and not have("A_detour.json"):
    with Timer("Part A: logit lens"):
        shots = "".join(f'Español: "{e}" - Italiano: "{i}"\n' for e, i in DETOUR_SHOTS)
        rows = []
        for es, it, en in DETOUR_WORDS:
            ti, te, ts = enc(it)["input_ids"][0], enc(en)["input_ids"][0], enc(es)["input_ids"][0]
            if len({ti, te, ts}) < 3: continue           # first tokens must differ
            rows.append((es, it, en, ti, te))
        P_it, P_en = [], []
        for i in range(0, len(rows), 16):
            chunk = rows[i:i + 16]
            seqs = [BOS + enc(shots + f'Español: "{es}" - Italiano: "')["input_ids"] for es, *_ in chunk]
            L = max(map(len, seqs))
            ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long); am = torch.zeros_like(ids)
            for b, s in enumerate(seqs): ids[b, :len(s)] = torch.tensor(s); am[b, :len(s)] = 1
            with torch.no_grad():
                out = BASE(input_ids=ids.to(INPUT_DEV), attention_mask=am.to(INPUT_DEV), output_hidden_states=True)
                last = am.sum(1) - 1
                for l in range(1, N_LAYERS + 1):
                    h = out.hidden_states[l][torch.arange(len(seqs)), last.to(out.hidden_states[l].device)]
                    if l < N_LAYERS: h = FINAL_NORM(h)      # hidden_states[N] already has the final norm
                    p = torch.softmax(LM_HEAD(h.to(LM_HEAD.weight.device)).float(), -1)
                    ti = torch.tensor([r[3] for r in chunk], device=p.device); te = torch.tensor([r[4] for r in chunk], device=p.device)
                    if l == 1: P_it.append([]); P_en.append([])
                    P_it[-1].append(p.gather(1, ti[:, None])[:, 0].cpu().numpy())
                    P_en[-1].append(p.gather(1, te[:, None])[:, 0].cpu().numpy())
            del out
        P_it = np.concatenate([np.stack(x, 1) for x in P_it], 0)     # [words, layers]
        P_en = np.concatenate([np.stack(x, 1) for x in P_en], 0)
        mi, me = P_it.mean(0), P_en.mean(0)
        cut = int(0.9 * N_LAYERS)
        detour_peak = int(np.argmax(me[:cut]) + 1)
        overtake = next((l + 1 for l in range(N_LAYERS) if mi[l] > me[l] and mi[l] > 0.05), None)
        save_json("A_detour.json", dict(words=[r[:3] for r in rows], P_it=P_it, P_en=P_en,
                                        detour_peak_layer=detour_peak, italian_overtakes_at=overtake))
A = load_json("A_detour.json") if have("A_detour.json") else None
if A:
    P_it, P_en = np.array(A["P_it"]), np.array(A["P_en"]); x = np.arange(1, N_LAYERS + 1) / N_LAYERS
    fig, ax = plt.subplots(figsize=(7, 3.4))
    for P, c, lab in [(P_en, HARD, "English word (never asked for)"), (P_it, EASY, "Italian answer")]:
        ax.plot(x, P.mean(0), color=c, lw=2, label=lab)
        ax.fill_between(x, np.percentile(P, 25, 0), np.percentile(P, 75, 0), color=c, alpha=0.15)
    ax.set_xlabel("depth through the model (0 = input, 1 = output)"); ax.set_ylabel("probability")
    ax.set_title(f"Translating Spanish into Italian, the model's working answer passes through English\n({MODEL_ID.split('/')[-1]}, {len(P_it)} words; band = middle half of words)", loc="left")
    ax.legend(frameon=False); savefig(fig, "A_english_detour")
    print(f"English peaks at layer {A['detour_peak_layer']}/{N_LAYERS}; Italian overtakes at {A['italian_overtakes_at']}; "
          f"peak mean P(English) = {P_en.mean(0).max():.3f}")

pack()


▶ Part A: logit lens ...


✓ Part A: logit lens: 4s


English peaks at layer 60/80; Italian overtakes at 69; peak mean P(English) = 0.039
PACKED /content/ut_results_unsloth_Llama_3_3_70B_Instruct_bnb_4bit.tar.gz


*File Operation*: `download` on `/content/ut_results_unsloth_Llama_3_3_70B_Instruct_bnb_4bit.tar.gz`

In [ ]:
# Reload earlier stages' results, so this stage also works in a fresh kernel (after 00_setup.py).
A = load_json("A_detour.json") if have("A_detour.json") else None
ladder = load_json("B_ladder.json") if have("B_ladder.json") else None
B = load_json("B_roundtrip.json") if have("B_roundtrip.json") else None
summary = load_json("B_summary.json")["per_sentence"] if have("B_summary.json") else {}


if RUN_B and not have("B_ladder.json"):
    with Timer("Part B: generating the ladder"):
        ladder = {S["id"]: {} for S in SENTENCES}
        for r in RUNGS:
            outs = generate([chat_prompt(rung_user(r, S["text"], S["lang"])) for S in SENTENCES], max_new_tokens=MAXTOK[r])
            for S, o in zip(SENTENCES, outs): ladder[S["id"]][r] = o
        save_json("B_ladder.json", ladder)
ladder = load_json("B_ladder.json") if have("B_ladder.json") else None
if ladder:
    for S in SENTENCES[:3] + [S for S in SENTENCES if S["id"] == "tren"]:
        print(f"\n{S['text']}"); [print(f"  {r:3s} {RUNG_NAME[r]:18s} {ladder[S['id']][r][:150]!r}") for r in RUNGS]

def mask_source_words(t, s):
    """Replace words in t that copy a source word of >= 4 letters (ignoring accents and a plural s) with '[·]'.
    Proper nouns are left alone. Genuine cognate translations (moment / momento) are NOT masked: they are translations."""
    import unicodedata
    def norm(w): return "".join(c for c in unicodedata.normalize("NFD", w.lower()) if unicodedata.category(c) != "Mn")
    src = set()
    for k, (w, a, b) in enumerate(words_with_spans(s)):
        if len(w) >= 4 and not (w[0].isupper() and k > 0): src |= {norm(w), norm(w) + "s", norm(w).rstrip("s")}
    n = [0]
    def rep(m):
        if norm(m.group(0)) in src: n[0] += 1; return "[·]"
        return m.group(0)
    return WORD_RE.sub(rep, t), n[0]

def bt_user(t, L): return f"Translate this English text into {LANG_NAME[L]}. Reply with the translation only.\n\n{t}"
def copy_user(s, L): return f"Translate this {LANG_NAME[L]} text into {LANG_NAME[L]}. Reply with the translation only.\n\n{s}"
def sense_user(ctx, span, L):
    return (f"Here is a text:\n\n{ctx}\n\nIn the original {LANG_NAME[L]} sentence, what did \"{span}\" mean in this context? "
            f"Answer in one line.")
EMPTY, DELTA = "(no text)", 0.5

def token_word_map(offsets, words):
    return [next((k for k, (_, wa, wb) in enumerate(words) if y > x and y > wa and x < wb), None) for (x, y) in offsets]

def drt_from(lp_t, lp_0, lp_s, offsets, words):
    R, Rs = lp_t - lp_0, lp_s - lp_0
    valid = Rs > DELTA
    w = np.where(valid, Rs, 0.0)
    ell = np.where(valid, 1 - np.clip(R / np.where(valid, Rs, 1.0), 0, 1), 0.0)
    d = float((w * ell).sum() / w.sum()) if w.sum() > 0 else float("nan")
    W, WL = np.zeros(len(words)), np.zeros(len(words))
    for i, k in enumerate(token_word_map(offsets, words)):
        if k is not None: W[k] += w[i]; WL[k] += w[i] * ell[i]
    return d, np.where(W > 0, WL / np.maximum(W, 1e-12), np.nan), W

if RUN_B and ladder and not have("B_roundtrip.json"):
    with Timer("Part B: round-trip scoring"):
        items, meta = [], []
        for S in SENTENCES:
            s, L = S["text"], S["lang"]
            items += [(chat_prompt(bt_user(EMPTY, L)), s), (chat_prompt(copy_user(s, L)), s)]
            meta += [(S["id"], "empty", None), (S["id"], "oracle", None)]
            for r in RUNGS:
                t = ladder[S["id"]][r]; tm, nm = mask_source_words(t, s)
                items += [(chat_prompt(bt_user(tm, L)), s), (chat_prompt(bt_user(t, L)), s)]
                meta += [(S["id"], r, "masked"), (S["id"], r, "raw")]
        sc = score([p for p, _ in items], [c for _, c in items])
        got = {(m[0], m[1], m[2]): r for m, r in zip(meta, sc)}

        # sense recovery
        s_items, s_meta = [], []
        for S in SENTENCES:
            s, L = S["text"], S["lang"]
            for span, gloss in S["hot"].items():
                ctxs = {"source": s, "empty": EMPTY}
                for r in RUNGS:
                    t = ladder[S["id"]][r]
                    ctxs[r] = t if r == "B2" else mask_source_words(t, s)[0]
                for k, ctx in ctxs.items():
                    s_items.append((chat_prompt(sense_user(ctx, span, L)), gloss)); s_meta.append((S["id"], span, k))
        ssc = score([p for p, _ in s_items], [c for _, c in s_items])
        sense_lp = defaultdict(dict)
        for (sid, span, k), r in zip(s_meta, ssc): sense_lp[(sid, span)][k] = float(r["lp"].mean())

        out = {}
        for S in SENTENCES:
            sid, s = S["id"], S["text"]; words = words_with_spans(s)
            lp0, lps = got[(sid, "empty", None)]["lp"], got[(sid, "oracle", None)]["lp"]
            offs = got[(sid, "empty", None)]["offsets"]
            rec = dict(words=[w for w, _, _ in words], rungs={})
            for r in RUNGS:
                t = ladder[sid][r]
                dm, wl_m, W = drt_from(got[(sid, r, "masked")]["lp"], lp0, lps, offs, words)
                du, wl_u, _ = drt_from(got[(sid, r, "raw")]["lp"], lp0, lps, offs, words)
                rec["rungs"][r] = dict(len=ntok(t), d_masked=dm, d_raw=du, word_left_masked=wl_m, word_left_raw=wl_u,
                                       n_masked=mask_source_words(t, s)[1])
            rec["word_weight"] = W
            sr = {}
            for span in S["hot"]:
                v = sense_lp[(sid, span)]; denom = v["source"] - v["empty"]
                sr[span] = dict(raw=v, norm={r: (v[r] - v["empty"]) / denom if abs(denom) > 1e-6 else float("nan") for r in RUNGS})
            rec["sense"] = sr
            out[sid] = rec
        save_json("B_roundtrip.json", out)

B = load_json("B_roundtrip.json") if have("B_roundtrip.json") else None
if B:
    ctrl = [B[S["id"]]["rungs"]["B1"]["d_masked"] for S in SENTENCES if S["cat"] == "control"]
    EPS = float(np.nanpercentile(ctrl, 90))
    def curve_stats(rec, rungs):
        L0 = rec["rungs"]["B1"]["len"]
        cands = [(rec["rungs"][r]["len"], rec["rungs"][r]["d_raw" if r == "B2" else "d_masked"]) for r in rungs]
        ok = [l for l, d in cands if d <= EPS]
        U = math.log2(max(min(ok), 1) / L0) if ok else math.log2(max(l for l, _ in cands) / L0)
        budgets = L0 * 2 ** np.linspace(0, 4, 9)
        Dl = [min([d for l, d in cands if l <= bgt] or [np.nan]) for bgt in budgets]
        return dict(U=max(U, 0.0), censored=not ok, A=float(np.nanmean(Dl)))
    summary = {}
    for sid, rec in B.items():
        summary[sid] = dict(cat=SID[sid]["cat"], lang=SID[sid]["lang"], **curve_stats(rec, NON_BORROW),
                            U_with_borrowing=curve_stats(rec, NON_BORROW + ["B2"])["U"],
                            d_B1=rec["rungs"]["B1"]["d_masked"],
                            borrow_gain=rec["rungs"]["B1"]["d_masked"] - rec["rungs"]["B2"]["d_raw"],
                            borrow_gain_masked=rec["rungs"]["B1"]["d_masked"] - rec["rungs"]["B2"]["d_masked"])
    # hot words vs other content words (standard translation, masked)
    hot_v, oth_v = [], []
    for S in SENTENCES:
        rec = B[S["id"]]; wl = np.array(rec["rungs"]["B1"]["word_left_masked"], float); W = np.array(rec["word_weight"])
        hot_idx = set()
        for span in S["hot"]:
            sp = find_phrase(S["text"], span)
            if sp: hot_idx |= {k for k, (_, a, b) in enumerate(words_with_spans(S["text"])) if b > sp[0] and a < sp[1]}
        for k, w in enumerate(rec["words"]):
            if W[k] <= 0 or np.isnan(wl[k]): continue
            if k in hot_idx: hot_v.append(wl[k])
            elif len(w) >= 4: oth_v.append(wl[k])
    B_auroc = auroc(hot_v, oth_v)
    print(f"tolerance ε (90th pct of control d at 'standard') = {EPS:.3f}")
    print(f"hot words vs other content words, leftover AUROC = {B_auroc:.3f} ({len(hot_v)} hot, {len(oth_v)} other)\n")
    print(f"{'category':15s} {'n':>2s} {'U (doublings)':>13s} {'cens':>4s} {'A':>6s} {'d std':>6s} {'borrow buys':>11s} {'…masked':>8s}")
    cats = ["untranslatable", "construction", "idiom", "control"]
    for c in cats:
        v = [x for x in summary.values() if x["cat"] == c]
        print(f"{c:15s} {len(v):2d} {np.mean([x['U'] for x in v]):13.2f} {sum(x['censored'] for x in v):4d} "
              f"{np.mean([x['A'] for x in v]):6.3f} {np.nanmean([x['d_B1'] for x in v]):6.3f} "
              f"{np.nanmean([x['borrow_gain'] for x in v]):11.3f} {np.nanmean([x['borrow_gain_masked'] for x in v]):8.3f}")
    fa = B["duende"]; wl = dict(zip(fa["words"], fa["rungs"]["B1"]["word_left_masked"]))
    print(f"\nMe gusta mucho el duende (standard translation: {ladder['duende']['B1']!r})")
    print("  leftover: " + ", ".join(f"{w} {wl.get(w, float('nan')):.2f}" for w in ["duende", "gusta", "mucho"])
          + "   [Llama-3.1-8B last time: duende 0.50, gusta 0.23, mucho 0.01]")
    print("  sense recovered (0 = none, 1 = as good as the source):")
    for span, v in fa["sense"].items():
        print(f"    {span:8s} " + "  ".join(f"{RUNG_NAME[r]} {v['norm'][r]:.2f}" for r in RUNGS))
    save_json("B_summary.json", dict(epsilon=EPS, auroc_hot_words=B_auroc, per_sentence=summary))

if B:
    def heat_row(ax, y, words, vals, title, sub):
        """Words set in a row, each shaded by its leftover; positions measured from the rendered text."""
        fig_ = ax.figure; r = fig_.canvas.get_renderer(); inv = ax.transData.inverted(); x = 0.0
        ax.text(0, y + 0.45, title, fontsize=9, color="#666", ha="left")
        for w, v in zip(words, vals):
            bad = v is None or np.isnan(v)
            c = GREY if bad else plt.cm.Oranges(0.08 + 0.85 * float(np.clip(v, 0, 1)))
            t = ax.text(x, y, w, fontsize=17, style="italic", family="serif", va="center", ha="left",
                        bbox=dict(boxstyle="round,pad=0.25", fc=c, ec="none"))
            bb = t.get_window_extent(renderer=r); (x0, _), (x1, _) = inv.transform([[bb.x0, bb.y0], [bb.x1, bb.y1]])
            ax.text((x0 + x1) / 2, y - 0.36, "" if bad else f"{v:.2f}", fontsize=7.5, color="#555", ha="center")
            x = x1 + 0.03
        ax.text(0, y - 0.66, sub, fontsize=9, color=INK, ha="left")
    show = [s for s in ["duende", "olvido", "pelo", "estrenar", "tren"] if s in B]
    fig, ax = plt.subplots(figsize=(9, 1.45 * len(show))); ax.axis("off")
    ax.set_xlim(-0.02, 1.25); ax.set_ylim(-1.5 * len(show) + 0.6, 0.9)
    for k, sid in enumerate(show):
        rec = B[sid]; S = SID[sid]
        wds = [w for w, _, _ in words_with_spans(S["text"])]
        heat_row(ax, -1.5 * k, wds, rec["rungs"]["B1"]["word_left_masked"], S["cat"], "→ " + ladder[sid]["B1"][:110])
    ax.set_xlim(-0.02, 1.25); ax.set_ylim(-1.5 * len(show) + 0.6, 0.9)
    ax.set_title("Where each sentence resists translation: darker words are harder to recover from the English\n"
                 f"(share of each word's information lost after a standard translation; {MODEL_ID.split('/')[-1]})", loc="left", fontsize=10)
    savefig(fig, "B1_heatmaps")

    show = [s for s in ["duende", "sobremesa", "pelo", "tren"] if s in B]
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    for sid in show:
        rec = B[sid]; col = EASY if SID[sid]["cat"] == "control" else HARD
        pts = sorted((rec["rungs"][r]["len"], rec["rungs"][r]["d_masked"], r) for r in NON_BORROW)
        env, best = [], 1.0
        for l, d, r in pts: best = min(best, d); env.append((l, best))
        ax.step([l for l, _ in env], [d for _, d in env], where="post", color=col, lw=2, alpha=0.9)
        ax.scatter([l for l, _, _ in pts], [d for _, d, _ in pts], color=col, s=18, zorder=3)
        ax.text(env[-1][0] * 1.05, env[-1][1], SID[sid]["text"][:28] + "…", fontsize=8, color=col, va="center")
    ax.axhline(EPS, color="#888", ls="--", lw=1); ax.text(ax.get_xlim()[0], EPS, "  gets across", fontsize=8, color="#888", va="bottom")
    ax.set_xscale("log"); ax.set_xlabel("length of the English (tokens, log scale): word for word → standard → gloss → footnote → thick")
    ax.set_ylabel("share of meaning not carried"); ax.set_title("How much English does it take?", loc="left")
    savefig(fig, "B2_distortion_vs_length")

    spans = [(sid, sp) for sid in B for sp in B[sid]["sense"]][:12]
    if spans:
        fig, ax = plt.subplots(figsize=(9, 3.6)); w = 0.15
        rr = ["B1", "B2", "B3", "B5", "B6"]; cols = [GREY, LOAN, "#e3a587", "#d07a55", HARD]
        for k, r in enumerate(rr):
            ax.bar(np.arange(len(spans)) + (k - 2) * w, np.clip([B[s]["sense"][sp]["norm"][r] for s, sp in spans], -0.5, 1.5), w, color=cols[k], label=RUNG_NAME[r])
        ax.set_xticks(np.arange(len(spans))); ax.set_xticklabels([sp for _, sp in spans], rotation=30, ha="right", style="italic")
        ax.axhline(1, color="#888", lw=0.8, ls=":"); ax.axhline(0, color="#888", lw=0.8)
        ax.set_ylabel("intended sense recovered\n(0 = none, 1 = as from the original)"); ax.set_ylim(-0.55, 1.55)
        ax.set_title("Does the English carry the sense the sentence means?", loc="left"); ax.legend(frameon=False, fontsize=8, ncol=5)
        savefig(fig, "B3_sense_recovery")

pack()


[transformers] Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


▶ Part B: generating the ladder ...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[transformers] Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


... (7 outputs left out)

[transformers] Both `max_new_tokens` (=260) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ Part B: generating the ladder: 98s

Me gusta mucho el duende.
  B0  word for word      'I like much the goblin.'
  B1  standard           'I love the elf.'
  B2  keep the word      'I really like the *duende*.'
  B3  inline gloss       'I like very much the goblin [mythical creature].'
  B5  footnote           'I love the duende very much.\n\n \nNote: The translation loses cultural nuance and poetic connotation.'
  B6  thick translation  'The translation is: I really like the elf.\n\nThis sentence expresses a strong affection for a mythical creature, evoking a sense of enchantment and fan'

Después de comer nos quedamos dos horas de sobremesa.
  B0  word for word      'After of eat we remained two hours of after-lunch.'
  B1  standard           'After lunch, we lingered at the table for two hours.'
  B2  keep the word      'After eating, we stayed two hours of *sobremesa*.'
  B3  inline gloss       'After eating we stayed two hours of overtime [work after hours].'
  B5  footnote     

✓ Part B: round-trip scoring: 62s
tolerance ε (90th pct of control d at 'standard') = 0.342
hot words vs other content words, leftover AUROC = 0.611 (29 hot, 70 other)

category         n U (doublings) cens      A  d std borrow buys  …masked
untranslatable  11          0.00    0  0.063  0.217       0.101    0.013
construction     6          0.33    0  0.080  0.206       0.120   -0.046
idiom            3          0.00    0  0.010  0.050       0.039    0.038
control          8          0.00    0  0.022  0.108       0.079    0.076

Me gusta mucho el duende (standard translation: 'I love the elf.')
  leftover: duende 0.07, gusta 1.00, mucho 1.00   [Llama-3.1-8B last time: duende 0.50, gusta 0.23, mucho 0.01]
  sense recovered (0 = none, 1 = as good as the source):
    duende   word for word 0.09  standard 0.30  keep the word -0.04  inline gloss 0.67  footnote -0.72  thick translation -0.46
    gusta    word for word 1.08  standard 0.74  keep the word 1.18  inline gloss 1.36  footnote 2.23 

PACKED /content/ut_results_unsloth_Llama_3_3_70B_Instruct_bnb_4bit.tar.gz


*File Operation*: `download` on `/content/ut_results_unsloth_Llama_3_3_70B_Instruct_bnb_4bit.tar.gz`

**Session Created**: 2026-11-22 12:12:29
- Endpoint: `gpu-a100-hm-ass1c0-35n6m1zfn9j2v`
- Hardware: `A100`

### Automation: install (2026-11-22 12:16:43)

In [ ]:

import subprocess, sys
def install():
    packages = ['-r', '/content/requirements.txt']
    try:
        subprocess.check_call(['uv', 'pip', 'install', '--system'] + packages)
        print('Installation Complete (via uv)!')
    except:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + packages)
        print('Installation Complete (via pip)!')
install()


In [ ]:
# Result of previous automation

Installation Complete (via uv)!


In [ ]:
import base64, os; os.makedirs('/content/ut', exist_ok=True)
open('/content/ut/00_setup.py','wb').write(base64.b64decode('...'))
open('/content/ut/A_detour.py','wb').write(base64.b64decode('IyBSZWxvYWQgZWFybGllciBzdGFnZXMnIHJlc3VsdHMsIHNvIHRoaXMgc3RhZ2UgYWxzbyB3b3JrcyBpbiBhIGZyZXNoIGtlcm5lbCAoYWZ0ZXIgMDBfc2V0dXAucHkpLgpBID0gbG9hZF9qc29uKCJBX2RldG91ci5qc29uIikgaWYgaGF2ZSgiQV9kZXRvdXIuanNvbiIpIGVsc2UgTm9uZQpsYWRkZXIgPSBsb2FkX2pzb24oIkJfbGFkZGVyLmpzb24iKSBpZiBoYXZlKCJCX2xhZGRlci5qc29uIikgZWxzZSBOb25lCkIgPSBsb2FkX2pzb24oIkJfcm91bmR0cmlwLmpzb24iKSBpZiBoYXZlKCJCX3JvdW5kdHJpcC5qc29uIikgZWxzZSBOb25lCnN1bW1hcnkgPSBsb2FkX2pzb24oIkJfc3VtbWFyeS5qc29uIilbInBlcl9zZW50ZW5jZSJdIGlmIGhhdmUoIkJfc3VtbWFyeS5qc29uIikgZWxzZSB7fQoKCmlmIFJVTl9BIGFuZCBub3QgaGF2ZSgiQV9kZXRvdXIuanNvbiIpOgogICAgd2l0aCBUaW1lcigiUGFydCBBOiBsb2dpdCBsZW5zIik6CiAgICAgICAgc2hvdHMgPSAiIi5qb2luKGYnRXNwYcOxb2w6ICJ7ZX0iIC0gSXRhbGlhbm86ICJ7aX0iXG4nIGZvciBlLCBpIGluIERFVE9VUl9TSE9UUykKICAgICAgICByb3dzID0gW10KICAgICAgICBmb3IgZXMsIGl0LCBlbiBpbiBERVRPVVJfV09SRFM6CiAgICAgICAgICAgIHRpLCB0ZSwgdHMgPSBlbmMoaXQpWyJpbnB1dF9pZHMiXVswXSwgZW5jKGVuKVsiaW5wdXRfaWRzIl1bMF0sIGVuYyhlcylbImlucHV0X2lkcyJdWzBdCiAgICAgICAgICAgIGlmIGxlbih7dGksIHRlLCB0c30pIDwgMzogY29udGludWUgICAgICAgICAgICMgZmlyc3QgdG9rZW5zIG11c3QgZGlmZmVyCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKChlcywgaXQsIGVuLCB0aSwgdGUpKQogICAgICAgIFBfaXQsIFBfZW4gPSBbXSwgW10KICAgICAgICBmb3IgaSBpbiByYW5nZSgwLCBsZW4ocm93cyksIDE2KToKICAgICAgICAgICAgY2h1bmsgPSByb3dzW2k6aSArIDE2XQogICAgICAgICAgICBzZXFzID0gW0JPUyArIGVuYyhzaG90cyArIGYnRXNwYcOxb2w6ICJ7ZXN9IiAtIEl0YWxpYW5vOiAiJylbImlucHV0X2lkcyJdIGZvciBlcywgKl8gaW4gY2h1bmtdCiAgICAgICAgICAgIEwgPSBtYXgobWFwKGxlbiwgc2VxcykpCiAgICAgICAgICAgIGlkcyA9IHRvcmNoLmZ1bGwoKGxlbihzZXFzKSwgTCksIHRvay5wYWRfdG9rZW5faWQsIGR0eXBlPXRvcmNoLmxvbmcpOyBhbSA9IHRvcmNoLnplcm9zX2xpa2UoaWRzKQogICAgICAgICAgICBmb3IgYiwgcyBpbiBlbnVtZXJhdGUoc2Vxcyk6IGlkc1tiLCA6bGVuKHMpXSA9IHRvcmNoLnRlbnNvcihzKTsgYW1bYiwgOmxlbihzKV0gPSAxCiAgICAgICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICAgICAgb3V0ID0gQkFTRShpbnB1dF9pZHM9aWRzLnRvKElOUFVUX0RFViksIGF0dGVudGlvbl9tYXNrPWFtLnRvKElOUFVUX0RFViksIG91dHB1dF9oaWRkZW5fc3RhdGVzPVRydWUpCiAgICAgICAgICAgICAgICBsYXN0ID0gYW0uc3VtKDEpIC0gMQogICAgICAgICAgICAgICAgZm9yIGwgaW4gcmFuZ2UoMSwgTl9MQVlFUlMgKyAxKToKICAgICAgICAgICAgICAgICAgICBoID0gb3V0LmhpZGRlbl9zdGF0ZXNbbF1bdG9yY2guYXJhbmdlKGxlbihzZXFzKSksIGxhc3QudG8ob3V0LmhpZGRlbl9zdGF0ZXNbbF0uZGV2aWNlKV0KICAgICAgICAgICAgICAgICAgICBpZiBsIDwgTl9MQVlFUlM6IGggPSBGSU5BTF9OT1JNKGgpICAgICAgIyBoaWRkZW5fc3RhdGVzW05dIGFscmVhZHkgaGFzIHRoZSBmaW5hbCBub3JtCiAgICAgICAgICAgICAgICAgICAgcCA9IHRvcmNoLnNvZnRtYXgoTE1fSEVBRChoLnRvKExNX0hFQUQud2VpZ2h0LmRldmljZSkpLmZsb2F0KCksIC0xKQogICAgICAgICAgICAgICAgICAgIHRpID0gdG9yY2gudGVuc29yKFtyWzNdIGZvciByIGluIGNodW5rXSwgZGV2aWNlPXAuZGV2aWNlKTsgdGUgPSB0b3JjaC50ZW5zb3IoW3JbNF0gZm9yIHIgaW4gY2h1bmtdLCBkZXZpY2U9cC5kZXZpY2UpCiAgICAgICAgICAgICAgICAgICAgaWYgbCA9PSAxOiBQX2l0LmFwcGVuZChbXSk7IFBfZW4uYXBwZW5kKFtdKQogICAgICAgICAgICAgICAgICAgIFBfaXRbLTFdLmFwcGVuZChwLmdhdGhlcigxLCB0aVs6LCBOb25lXSlbOiwgMF0uY3B1KCkubnVtcHkoKSkKICAgICAgICAgICAgICAgICAgICBQX2VuWy0xXS5hcHBlbmQocC5nYXRoZXIoMSwgdGVbOiwgTm9uZV0pWzosIDBdLmNwdSgpLm51bXB5KCkpCiAgICAgICAgICAgIGRlbCBvdXQKICAgICAgICBQX2l0ID0gbnAuY29uY2F0ZW5hdGUoW25wLnN0YWNrKHgsIDEpIGZvciB4IGluIFBfaXRdLCAwKSAgICAgIyBbd29yZHMsIGxheWVyc10KICAgICAgICBQX2VuID0gbnAuY29uY2F0ZW5hdGUoW25wLnN0YWNrKHgsIDEpIGZvciB4IGluIFBfZW5dLCAwKQogICAgICAgIG1pLCBtZSA9IFBfaXQubWVhbigwKSwgUF9lbi5tZWFuKDApCiAgICAgICAgY3V0ID0gaW50KDAuOSAqIE5fTEFZRVJTKQogICAgICAgIGRldG91cl9wZWFrID0gaW50KG5wLmFyZ21heChtZVs6Y3V0XSkgKyAxKQogICAgICAgIG92ZXJ0YWtlID0gbmV4dCgobCArIDEgZm9yIGwgaW4gcmFuZ2UoTl9MQVlFUlMpIGlmIG1pW2xdID4gbWVbbF0gYW5kIG1pW2xdID4gMC4wNSksIE5vbmUpCiAgICAgICAgc2F2ZV9qc29uKCJBX2RldG91ci5qc29uIiwgZGljdCh3b3Jkcz1bcls6M10gZm9yIHIgaW4gcm93c10sIFBfaXQ9UF9pdCwgUF9lbj1QX2VuLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZGV0b3VyX3BlYWtfbGF5ZXI9ZGV0b3VyX3BlYWssIGl0YWxpYW5fb3ZlcnRha2VzX2F0PW92ZXJ0YWtlKSkKQSA9IGxvYWRfanNvbigiQV9kZXRvdXIuanNvbiIpIGlmIGhhdmUoIkFfZGV0b3VyLmpzb24iKSBlbHNlIE5vbmUKaWYgQToKICAgIFBfaXQsIFBfZW4gPSBucC5hcnJheShBWyJQX2l0Il0pLCBucC5hcnJheShBWyJQX2VuIl0pOyB4ID0gbnAuYXJhbmdlKDEsIE5fTEFZRVJTICsgMSkgLyBOX0xBWUVSUwogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg3LCAzLjQpKQogICAgZm9yIFAsIGMsIGxhYiBpbiBbKFBfZW4sIEhBUkQsICJFbmdsaXNoIHdvcmQgKG5ldmVyIGFza2VkIGZvcikiKSwgKFBfaXQsIEVBU1ksICJJdGFsaWFuIGFuc3dlciIpXToKICAgICAgICBheC5wbG90KHgsIFAubWVhbigwKSwgY29sb3I9YywgbHc9MiwgbGFiZWw9bGFiKQogICAgICAgIGF4LmZpbGxfYmV0d2Vlbih4LCBucC5wZXJjZW50aWxlKFAsIDI1LCAwKSwgbnAucGVyY2VudGlsZShQLCA3NSwgMCksIGNvbG9yPWMsIGFscGhhPTAuMTUpCiAgICBheC5zZXRfeGxhYmVsKCJkZXB0aCB0aHJvdWdoIHRoZSBtb2RlbCAoMCA9IGlucHV0LCAxID0gb3V0cHV0KSIpOyBheC5zZXRfeWxhYmVsKCJwcm9iYWJpbGl0eSIpCiAgICBheC5zZXRfdGl0bGUoZiJUcmFuc2xhdGluZyBTcGFuaXNoIGludG8gSXRhbGlhbiwgdGhlIG1vZGVsJ3Mgd29ya2luZyBhbnN3ZXIgcGFzc2VzIHRocm91Z2ggRW5nbGlzaFxuKHtNT0RFTF9JRC5zcGxpdCgnLycpWy0xXX0sIHtsZW4oUF9pdCl9IHdvcmRzOyBiYW5kID0gbWlkZGxlIGhhbGYgb2Ygd29yZHMpIiwgbG9jPSJsZWZ0IikKICAgIGF4LmxlZ2VuZChmcmFtZW9uPUZhbHNlKTsgc2F2ZWZpZyhmaWcsICJBX2VuZ2xpc2hfZGV0b3VyIikKICAgIHByaW50KGYiRW5nbGlzaCBwZWFrcyBhdCBsYXllciB7QVsnZGV0b3VyX3BlYWtfbGF5ZXInXX0ve05fTEFZRVJTfTsgSXRhbGlhbiBvdmVydGFrZXMgYXQge0FbJ2l0YWxpYW5fb3ZlcnRha2VzX2F0J119OyAiCiAgICAgICAgICBmInBlYWsgbWVhbiBQKEVuZ2xpc2gpID0ge1BfZW4ubWVhbigwKS5tYXgoKTouM2Z9IikKCnBhY2soKQo='))
open('/content/ut/B_roundtrip.py','wb').write(base64.b64decode('IyBSZWxvYWQgZWFybGllciBzdGFnZXMnIHJlc3VsdHMsIHNvIHRoaXMgc3RhZ2UgYWxzbyB3b3JrcyBpbiBhIGZyZXNoIGtlcm5lbCAoYWZ0ZXIgMDBfc2V0dXAucHkpLgpBID0gbG9hZF9qc29uKCJBX2RldG91ci5qc29uIikgaWYgaGF2ZSgiQV9kZXRvdXIuanNvbiIpIGVsc2UgTm9uZQpsYWRkZXIgPSBsb2FkX2pzb24oIkJfbGFkZGVyLmpzb24iKSBpZiBoYXZlKCJCX2xhZGRlci5qc29uIikgZWxzZSBOb25lCkIgPSBsb2FkX2pzb24oIkJfcm91bmR0cmlwLmpzb24iKSBpZiBoYXZlKCJCX3JvdW5kdHJpcC5qc29uIikgZWxzZSBOb25lCnN1bW1hcnkgPSBsb2FkX2pzb24oIkJfc3VtbWFyeS5qc29uIilbInBlcl9zZW50ZW5jZSJdIGlmIGhhdmUoIkJfc3VtbWFyeS5qc29uIikgZWxzZSB7fQoKCmlmIFJVTl9CIGFuZCBub3QgaGF2ZSgiQl9sYWRkZXIuanNvbiIpOgogICAgd2l0aCBUaW1lcigiUGFydCBCOiBnZW5lcmF0aW5nIHRoZSBsYWRkZXIiKToKICAgICAgICBsYWRkZXIgPSB7U1siaWQiXToge30gZm9yIFMgaW4gU0VOVEVOQ0VTfQogICAgICAgIGZvciByIGluIFJVTkdTOgogICAgICAgICAgICBvdXRzID0gZ2VuZXJhdGUoW2NoYXRfcHJvbXB0KHJ1bmdfdXNlcihyLCBTWyJ0ZXh0Il0sIFNbImxhbmciXSkpIGZvciBTIGluIFNFTlRFTkNFU10sIG1heF9uZXdfdG9rZW5zPU1BWFRPS1tyXSkKICAgICAgICAgICAgZm9yIFMsIG8gaW4gemlwKFNFTlRFTkNFUywgb3V0cyk6IGxhZGRlcltTWyJpZCJdXVtyXSA9IG8KICAgICAgICBzYXZlX2pzb24oIkJfbGFkZGVyLmpzb24iLCBsYWRkZXIpCmxhZGRlciA9IGxvYWRfanNvbigiQl9sYWRkZXIuanNvbiIpIGlmIGhhdmUoIkJfbGFkZGVyLmpzb24iKSBlbHNlIE5vbmUKaWYgbGFkZGVyOgogICAgZm9yIFMgaW4gU0VOVEVOQ0VTWzozXSArIFtTIGZvciBTIGluIFNFTlRFTkNFUyBpZiBTWyJpZCJdID09ICJ0cmVuIl06CiAgICAgICAgcHJpbnQoZiJcbntTWyd0ZXh0J119Iik7IFtwcmludChmIiAge3I6M3N9IHtSVU5HX05BTUVbcl06MThzfSB7bGFkZGVyW1NbJ2lkJ11dW3JdWzoxNTBdIXJ9IikgZm9yIHIgaW4gUlVOR1NdCgpkZWYgbWFza19zb3VyY2Vfd29yZHModCwgcyk6CiAgICAiIiJSZXBsYWNlIHdvcmRzIGluIHQgdGhhdCBjb3B5IGEgc291cmNlIHdvcmQgb2YgPj0gNCBsZXR0ZXJzIChpZ25vcmluZyBhY2NlbnRzIGFuZCBhIHBsdXJhbCBzKSB3aXRoICdbwrddJy4KICAgIFByb3BlciBub3VucyBhcmUgbGVmdCBhbG9uZS4gR2VudWluZSBjb2duYXRlIHRyYW5zbGF0aW9ucyAobW9tZW50IC8gbW9tZW50bykgYXJlIE5PVCBtYXNrZWQ6IHRoZXkgYXJlIHRyYW5zbGF0aW9ucy4iIiIKICAgIGltcG9ydCB1bmljb2RlZGF0YQogICAgZGVmIG5vcm0odyk6IHJldHVybiAiIi5qb2luKGMgZm9yIGMgaW4gdW5pY29kZWRhdGEubm9ybWFsaXplKCJORkQiLCB3Lmxvd2VyKCkpIGlmIHVuaWNvZGVkYXRhLmNhdGVnb3J5KGMpICE9ICJNbiIpCiAgICBzcmMgPSBzZXQoKQogICAgZm9yIGssICh3LCBhLCBiKSBpbiBlbnVtZXJhdGUod29yZHNfd2l0aF9zcGFucyhzKSk6CiAgICAgICAgaWYgbGVuKHcpID49IDQgYW5kIG5vdCAod1swXS5pc3VwcGVyKCkgYW5kIGsgPiAwKTogc3JjIHw9IHtub3JtKHcpLCBub3JtKHcpICsgInMiLCBub3JtKHcpLnJzdHJpcCgicyIpfQogICAgbiA9IFswXQogICAgZGVmIHJlcChtKToKICAgICAgICBpZiBub3JtKG0uZ3JvdXAoMCkpIGluIHNyYzogblswXSArPSAxOyByZXR1cm4gIlvCt10iCiAgICAgICAgcmV0dXJuIG0uZ3JvdXAoMCkKICAgIHJldHVybiBXT1JEX1JFLnN1YihyZXAsIHQpLCBuWzBdCgpkZWYgYnRfdXNlcih0LCBMKTogcmV0dXJuIGYiVHJhbnNsYXRlIHRoaXMgRW5nbGlzaCB0ZXh0IGludG8ge0xBTkdfTkFNRVtMXX0uIFJlcGx5IHdpdGggdGhlIHRyYW5zbGF0aW9uIG9ubHkuXG5cbnt0fSIKZGVmIGNvcHlfdXNlcihzLCBMKTogcmV0dXJuIGYiVHJhbnNsYXRlIHRoaXMge0xBTkdfTkFNRVtMXX0gdGV4dCBpbnRvIHtMQU5HX05BTUVbTF19LiBSZXBseSB3aXRoIHRoZSB0cmFuc2xhdGlvbiBvbmx5LlxuXG57c30iCmRlZiBzZW5zZV91c2VyKGN0eCwgc3BhbiwgTCk6CiAgICByZXR1cm4gKGYiSGVyZSBpcyBhIHRleHQ6XG5cbntjdHh9XG5cbkluIHRoZSBvcmlnaW5hbCB7TEFOR19OQU1FW0xdfSBzZW50ZW5jZSwgd2hhdCBkaWQgXCJ7c3Bhbn1cIiBtZWFuIGluIHRoaXMgY29udGV4dD8gIgogICAgICAgICAgICBmIkFuc3dlciBpbiBvbmUgbGluZS4iKQpFTVBUWSwgREVMVEEgPSAiKG5vIHRleHQpIiwgMC41CgpkZWYgdG9rZW5fd29yZF9tYXAob2Zmc2V0cywgd29yZHMpOgogICAgcmV0dXJuIFtuZXh0KChrIGZvciBrLCAoXywgd2EsIHdiKSBpbiBlbnVtZXJhdGUod29yZHMpIGlmIHkgPiB4IGFuZCB5ID4gd2EgYW5kIHggPCB3YiksIE5vbmUpIGZvciAoeCwgeSkgaW4gb2Zmc2V0c10KCmRlZiBkcnRfZnJvbShscF90LCBscF8wLCBscF9zLCBvZmZzZXRzLCB3b3Jkcyk6CiAgICBSLCBScyA9IGxwX3QgLSBscF8wLCBscF9zIC0gbHBfMAogICAgdmFsaWQgPSBScyA+IERFTFRBCiAgICB3ID0gbnAud2hlcmUodmFsaWQsIFJzLCAwLjApCiAgICBlbGwgPSBucC53aGVyZSh2YWxpZCwgMSAtIG5wLmNsaXAoUiAvIG5wLndoZXJlKHZhbGlkLCBScywgMS4wKSwgMCwgMSksIDAuMCkKICAgIGQgPSBmbG9hdCgodyAqIGVsbCkuc3VtKCkgLyB3LnN1bSgpKSBpZiB3LnN1bSgpID4gMCBlbHNlIGZsb2F0KCJuYW4iKQogICAgVywgV0wgPSBucC56ZXJvcyhsZW4od29yZHMpKSwgbnAuemVyb3MobGVuKHdvcmRzKSkKICAgIGZvciBpLCBrIGluIGVudW1lcmF0ZSh0b2tlbl93b3JkX21hcChvZmZzZXRzLCB3b3JkcykpOgogICAgICAgIGlmIGsgaXMgbm90IE5vbmU6IFdba10gKz0gd1tpXTsgV0xba10gKz0gd1tpXSAqIGVsbFtpXQogICAgcmV0dXJuIGQsIG5wLndoZXJlKFcgPiAwLCBXTCAvIG5wLm1heGltdW0oVywgMWUtMTIpLCBucC5uYW4pLCBXCgppZiBSVU5fQiBhbmQgbGFkZGVyIGFuZCBub3QgaGF2ZSgiQl9yb3VuZHRyaXAuanNvbiIpOgogICAgd2l0aCBUaW1lcigiUGFydCBCOiByb3VuZC10cmlwIHNjb3JpbmciKToKICAgICAgICBpdGVtcywgbWV0YSA9IFtdLCBbXQogICAgICAgIGZvciBTIGluIFNFTlRFTkNFUzoKICAgICAgICAgICAgcywgTCA9IFNbInRleHQiXSwgU1sibGFuZyJdCiAgICAgICAgICAgIGl0ZW1zICs9IFsoY2hhdF9wcm9tcHQoYnRfdXNlcihFTVBUWSwgTCkpLCBzKSwgKGNoYXRfcHJvbXB0KGNvcHlfdXNlcihzLCBMKSksIHMpXQogICAgICAgICAgICBtZXRhICs9IFsoU1siaWQiXSwgImVtcHR5IiwgTm9uZSksIChTWyJpZCJdLCAib3JhY2xlIiwgTm9uZSldCiAgICAgICAgICAgIGZvciByIGluIFJVTkdTOgogICAgICAgICAgICAgICAgdCA9IGxhZGRlcltTWyJpZCJdXVtyXTsgdG0sIG5tID0gbWFza19zb3VyY2Vfd29yZHModCwgcykKICAgICAgICAgICAgICAgIGl0ZW1zICs9IFsoY2hhdF9wcm9tcHQoYnRfdXNlcih0bSwgTCkpLCBzKSwgKGNoYXRfcHJvbXB0KGJ0X3VzZXIodCwgTCkpLCBzKV0KICAgICAgICAgICAgICAgIG1ldGEgKz0gWyhTWyJpZCJdLCByLCAibWFza2VkIiksIChTWyJpZCJdLCByLCAicmF3IildCiAgICAgICAgc2MgPSBzY29yZShbcCBmb3IgcCwgXyBpbiBpdGVtc10sIFtjIGZvciBfLCBjIGluIGl0ZW1zXSkKICAgICAgICBnb3QgPSB7KG1bMF0sIG1bMV0sIG1bMl0pOiByIGZvciBtLCByIGluIHppcChtZXRhLCBzYyl9CgogICAgICAgICMgc2Vuc2UgcmVjb3ZlcnkKICAgICAgICBzX2l0ZW1zLCBzX21ldGEgPSBbXSwgW10KICAgICAgICBmb3IgUyBpbiBTRU5URU5DRVM6CiAgICAgICAgICAgIHMsIEwgPSBTWyJ0ZXh0Il0sIFNbImxhbmciXQogICAgICAgICAgICBmb3Igc3BhbiwgZ2xvc3MgaW4gU1siaG90Il0uaXRlbXMoKToKICAgICAgICAgICAgICAgIGN0eHMgPSB7InNvdXJjZSI6IHMsICJlbXB0eSI6IEVNUFRZfQogICAgICAgICAgICAgICAgZm9yIHIgaW4gUlVOR1M6CiAgICAgICAgICAgICAgICAgICAgdCA9IGxhZGRlcltTWyJpZCJdXVtyXQogICAgICAgICAgICAgICAgICAgIGN0eHNbcl0gPSB0IGlmIHIgPT0gIkIyIiBlbHNlIG1hc2tfc291cmNlX3dvcmRzKHQsIHMpWzBdCiAgICAgICAgICAgICAgICBmb3IgaywgY3R4IGluIGN0eHMuaXRlbXMoKToKICAgICAgICAgICAgICAgICAgICBzX2l0ZW1zLmFwcGVuZCgoY2hhdF9wcm9tcHQoc2Vuc2VfdXNlcihjdHgsIHNwYW4sIEwpKSwgZ2xvc3MpKTsgc19tZXRhLmFwcGVuZCgoU1siaWQiXSwgc3BhbiwgaykpCiAgICAgICAgc3NjID0gc2NvcmUoW3AgZm9yIHAsIF8gaW4gc19pdGVtc10sIFtjIGZvciBfLCBjIGluIHNfaXRlbXNdKQogICAgICAgIHNlbnNlX2xwID0gZGVmYXVsdGRpY3QoZGljdCkKICAgICAgICBmb3IgKHNpZCwgc3BhbiwgayksIHIgaW4gemlwKHNfbWV0YSwgc3NjKTogc2Vuc2VfbHBbKHNpZCwgc3BhbildW2tdID0gZmxvYXQoclsibHAiXS5tZWFuKCkpCgogICAgICAgIG91dCA9IHt9CiAgICAgICAgZm9yIFMgaW4gU0VOVEVOQ0VTOgogICAgICAgICAgICBzaWQsIHMgPSBTWyJpZCJdLCBTWyJ0ZXh0Il07IHdvcmRzID0gd29yZHNfd2l0aF9zcGFucyhzKQogICAgICAgICAgICBscDAsIGxwcyA9IGdvdFsoc2lkLCAiZW1wdHkiLCBOb25lKV1bImxwIl0sIGdvdFsoc2lkLCAib3JhY2xlIiwgTm9uZSldWyJscCJdCiAgICAgICAgICAgIG9mZnMgPSBnb3RbKHNpZCwgImVtcHR5IiwgTm9uZSldWyJvZmZzZXRzIl0KICAgICAgICAgICAgcmVjID0gZGljdCh3b3Jkcz1bdyBmb3IgdywgXywgXyBpbiB3b3Jkc10sIHJ1bmdzPXt9KQogICAgICAgICAgICBmb3IgciBpbiBSVU5HUzoKICAgICAgICAgICAgICAgIHQgPSBsYWRkZXJbc2lkXVtyXQogICAgICAgICAgICAgICAgZG0sIHdsX20sIFcgPSBkcnRfZnJvbShnb3RbKHNpZCwgciwgIm1hc2tlZCIpXVsibHAiXSwgbHAwLCBscHMsIG9mZnMsIHdvcmRzKQogICAgICAgICAgICAgICAgZHUsIHdsX3UsIF8gPSBkcnRfZnJvbShnb3RbKHNpZCwgciwgInJhdyIpXVsibHAiXSwgbHAwLCBscHMsIG9mZnMsIHdvcmRzKQogICAgICAgICAgICAgICAgcmVjWyJydW5ncyJdW3JdID0gZGljdChsZW49bnRvayh0KSwgZF9tYXNrZWQ9ZG0sIGRfcmF3PWR1LCB3b3JkX2xlZnRfbWFza2VkPXdsX20sIHdvcmRfbGVmdF9yYXc9d2xfdSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbl9tYXNrZWQ9bWFza19zb3VyY2Vfd29yZHModCwgcylbMV0pCiAgICAgICAgICAgIHJlY1sid29yZF93ZWlnaHQiXSA9IFcKICAgICAgICAgICAgc3IgPSB7fQogICAgICAgICAgICBmb3Igc3BhbiBpbiBTWyJob3QiXToKICAgICAgICAgICAgICAgIHYgPSBzZW5zZV9scFsoc2lkLCBzcGFuKV07IGRlbm9tID0gdlsic291cmNlIl0gLSB2WyJlbXB0eSJdCiAgICAgICAgICAgICAgICBzcltzcGFuXSA9IGRpY3QocmF3PXYsIG5vcm09e3I6ICh2W3JdIC0gdlsiZW1wdHkiXSkgLyBkZW5vbSBpZiBhYnMoZGVub20pID4gMWUtNiBlbHNlIGZsb2F0KCJuYW4iKSBmb3IgciBpbiBSVU5HU30pCiAgICAgICAgICAgIHJlY1sic2Vuc2UiXSA9IHNyCiAgICAgICAgICAgIG91dFtzaWRdID0gcmVjCiAgICAgICAgc2F2ZV9qc29uKCJCX3JvdW5kdHJpcC5qc29uIiwgb3V0KQoKQiA9IGxvYWRfanNvbigiQl9yb3VuZHRyaXAuanNvbiIpIGlmIGhhdmUoIkJfcm91bmR0cmlwLmpzb24iKSBlbHNlIE5vbmUKaWYgQjoKICAgIGN0cmwgPSBbQltTWyJpZCJdXVsicnVuZ3MiXVsiQjEiXVsiZF9tYXNrZWQiXSBmb3IgUyBpbiBTRU5URU5DRVMgaWYgU1siY2F0Il0gPT0gImNvbnRyb2wiXQogICAgRVBTID0gZmxvYXQobnAubmFucGVyY2VudGlsZShjdHJsLCA5MCkpCiAgICBkZWYgY3VydmVfc3RhdHMocmVjLCBydW5ncyk6CiAgICAgICAgTDAgPSByZWNbInJ1bmdzIl1bIkIxIl1bImxlbiJdCiAgICAgICAgY2FuZHMgPSBbKHJlY1sicnVuZ3MiXVtyXVsibGVuIl0sIHJlY1sicnVuZ3MiXVtyXVsiZF9yYXciIGlmIHIgPT0gIkIyIiBlbHNlICJkX21hc2tlZCJdKSBmb3IgciBpbiBydW5nc10KICAgICAgICBvayA9IFtsIGZvciBsLCBkIGluIGNhbmRzIGlmIGQgPD0gRVBTXQogICAgICAgIFUgPSBtYXRoLmxvZzIobWF4KG1pbihvayksIDEpIC8gTDApIGlmIG9rIGVsc2UgbWF0aC5sb2cyKG1heChsIGZvciBsLCBfIGluIGNhbmRzKSAvIEwwKQogICAgICAgIGJ1ZGdldHMgPSBMMCAqIDIgKiogbnAubGluc3BhY2UoMCwgNCwgOSkKICAgICAgICBEbCA9IFttaW4oW2QgZm9yIGwsIGQgaW4gY2FuZHMgaWYgbCA8PSBiZ3RdIG9yIFtucC5uYW5dKSBmb3IgYmd0IGluIGJ1ZGdldHNdCiAgICAgICAgcmV0dXJuIGRpY3QoVT1tYXgoVSwgMC4wKSwgY2Vuc29yZWQ9bm90IG9rLCBBPWZsb2F0KG5wLm5hbm1lYW4oRGwpKSkKICAgIHN1bW1hcnkgPSB7fQogICAgZm9yIHNpZCwgcmVjIGluIEIuaXRlbXMoKToKICAgICAgICBzdW1tYXJ5W3NpZF0gPSBkaWN0KGNhdD1TSURbc2lkXVsiY2F0Il0sIGxhbmc9U0lEW3NpZF1bImxhbmciXSwgKipjdXJ2ZV9zdGF0cyhyZWMsIE5PTl9CT1JST1cpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgVV93aXRoX2JvcnJvd2luZz1jdXJ2ZV9zdGF0cyhyZWMsIE5PTl9CT1JST1cgKyBbIkIyIl0pWyJVIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBkX0IxPXJlY1sicnVuZ3MiXVsiQjEiXVsiZF9tYXNrZWQiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGJvcnJvd19nYWluPXJlY1sicnVuZ3MiXVsiQjEiXVsiZF9tYXNrZWQiXSAtIHJlY1sicnVuZ3MiXVsiQjIiXVsiZF9yYXciXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGJvcnJvd19nYWluX21hc2tlZD1yZWNbInJ1bmdzIl1bIkIxIl1bImRfbWFza2VkIl0gLSByZWNbInJ1bmdzIl1bIkIyIl1bImRfbWFza2VkIl0pCiAgICAjIGhvdCB3b3JkcyB2cyBvdGhlciBjb250ZW50IHdvcmRzIChzdGFuZGFyZCB0cmFuc2xhdGlvbiwgbWFza2VkKQogICAgaG90X3YsIG90aF92ID0gW10sIFtdCiAgICBmb3IgUyBpbiBTRU5URU5DRVM6CiAgICAgICAgcmVjID0gQltTWyJpZCJdXTsgd2wgPSBucC5hcnJheShyZWNbInJ1bmdzIl1bIkIxIl1bIndvcmRfbGVmdF9tYXNrZWQiXSwgZmxvYXQpOyBXID0gbnAuYXJyYXkocmVjWyJ3b3JkX3dlaWdodCJdKQogICAgICAgIGhvdF9pZHggPSBzZXQoKQogICAgICAgIGZvciBzcGFuIGluIFNbImhvdCJdOgogICAgICAgICAgICBzcCA9IGZpbmRfcGhyYXNlKFNbInRleHQiXSwgc3BhbikKICAgICAgICAgICAgaWYgc3A6IGhvdF9pZHggfD0ge2sgZm9yIGssIChfLCBhLCBiKSBpbiBlbnVtZXJhdGUod29yZHNfd2l0aF9zcGFucyhTWyJ0ZXh0Il0pKSBpZiBiID4gc3BbMF0gYW5kIGEgPCBzcFsxXX0KICAgICAgICBmb3IgaywgdyBpbiBlbnVtZXJhdGUocmVjWyJ3b3JkcyJdKToKICAgICAgICAgICAgaWYgV1trXSA8PSAwIG9yIG5wLmlzbmFuKHdsW2tdKTogY29udGludWUKICAgICAgICAgICAgaWYgayBpbiBob3RfaWR4OiBob3Rfdi5hcHBlbmQod2xba10pCiAgICAgICAgICAgIGVsaWYgbGVuKHcpID49IDQ6IG90aF92LmFwcGVuZCh3bFtrXSkKICAgIEJfYXVyb2MgPSBhdXJvYyhob3Rfdiwgb3RoX3YpCiAgICBwcmludChmInRvbGVyYW5jZSDOtSAoOTB0aCBwY3Qgb2YgY29udHJvbCBkIGF0ICdzdGFuZGFyZCcpID0ge0VQUzouM2Z9IikKICAgIHByaW50KGYiaG90IHdvcmRzIHZzIG90aGVyIGNvbnRlbnQgd29yZHMsIGxlZnRvdmVyIEFVUk9DID0ge0JfYXVyb2M6LjNmfSAoe2xlbihob3Rfdil9IGhvdCwge2xlbihvdGhfdil9IG90aGVyKVxuIikKICAgIHByaW50KGYieydjYXRlZ29yeSc6MTVzfSB7J24nOj4yc30geydVIChkb3VibGluZ3MpJzo+MTNzfSB7J2NlbnMnOj40c30geydBJzo+NnN9IHsnZCBzdGQnOj42c30geydib3Jyb3cgYnV5cyc6PjExc30geyfigKZtYXNrZWQnOj44c30iKQogICAgY2F0cyA9IFsidW50cmFuc2xhdGFibGUiLCAiY29uc3RydWN0aW9uIiwgImlkaW9tIiwgImNvbnRyb2wiXQogICAgZm9yIGMgaW4gY2F0czoKICAgICAgICB2ID0gW3ggZm9yIHggaW4gc3VtbWFyeS52YWx1ZXMoKSBpZiB4WyJjYXQiXSA9PSBjXQogICAgICAgIHByaW50KGYie2M6MTVzfSB7bGVuKHYpOjJkfSB7bnAubWVhbihbeFsnVSddIGZvciB4IGluIHZdKToxMy4yZn0ge3N1bSh4WydjZW5zb3JlZCddIGZvciB4IGluIHYpOjRkfSAiCiAgICAgICAgICAgICAgZiJ7bnAubWVhbihbeFsnQSddIGZvciB4IGluIHZdKTo2LjNmfSB7bnAubmFubWVhbihbeFsnZF9CMSddIGZvciB4IGluIHZdKTo2LjNmfSAiCiAgICAgICAgICAgICAgZiJ7bnAubmFubWVhbihbeFsnYm9ycm93X2dhaW4nXSBmb3IgeCBpbiB2XSk6MTEuM2Z9IHtucC5uYW5tZWFuKFt4Wydib3Jyb3dfZ2Fpbl9tYXNrZWQnXSBmb3IgeCBpbiB2XSk6OC4zZn0iKQogICAgZmEgPSBCWyJkdWVuZGUiXTsgd2wgPSBkaWN0KHppcChmYVsid29yZHMiXSwgZmFbInJ1bmdzIl1bIkIxIl1bIndvcmRfbGVmdF9tYXNrZWQiXSkpCiAgICBwcmludChmIlxuTWUgZ3VzdGEgbXVjaG8gZWwgZHVlbmRlIChzdGFuZGFyZCB0cmFuc2xhdGlvbjoge2xhZGRlclsnZHVlbmRlJ11bJ0IxJ10hcn0pIikKICAgIHByaW50KCIgIGxlZnRvdmVyOiAiICsgIiwgIi5qb2luKGYie3d9IHt3bC5nZXQodywgZmxvYXQoJ25hbicpKTouMmZ9IiBmb3IgdyBpbiBbImR1ZW5kZSIsICJndXN0YSIsICJtdWNobyJdKQogICAgICAgICAgKyAiICAgW0xsYW1hLTMuMS04QiBsYXN0IHRpbWU6IGR1ZW5kZSAwLjUwLCBndXN0YSAwLjIzLCBtdWNobyAwLjAxXSIpCiAgICBwcmludCgiICBzZW5zZSByZWNvdmVyZWQgKDAgPSBub25lLCAxID0gYXMgZ29vZCBhcyB0aGUgc291cmNlKToiKQogICAgZm9yIHNwYW4sIHYgaW4gZmFbInNlbnNlIl0uaXRlbXMoKToKICAgICAgICBwcmludChmIiAgICB7c3Bhbjo4c30gIiArICIgICIuam9pbihmIntSVU5HX05BTUVbcl19IHt2Wydub3JtJ11bcl06LjJmfSIgZm9yIHIgaW4gUlVOR1MpKQogICAgc2F2ZV9qc29uKCJCX3N1bW1hcnkuanNvbiIsIGRpY3QoZXBzaWxvbj1FUFMsIGF1cm9jX2hvdF93b3Jkcz1CX2F1cm9jLCBwZXJfc2VudGVuY2U9c3VtbWFyeSkpCgppZiBCOgogICAgZGVmIGhlYXRfcm93KGF4LCB5LCB3b3JkcywgdmFscywgdGl0bGUsIHN1Yik6CiAgICAgICAgIiIiV29yZHMgc2V0IGluIGEgcm93LCBlYWNoIHNoYWRlZCBieSBpdHMgbGVmdG92ZXI7IHBvc2l0aW9ucyBtZWFzdXJlZCBmcm9tIHRoZSByZW5kZXJlZCB0ZXh0LiIiIgogICAgICAgIGZpZ18gPSBheC5maWd1cmU7IHIgPSBmaWdfLmNhbnZhcy5nZXRfcmVuZGVyZXIoKTsgaW52ID0gYXgudHJhbnNEYXRhLmludmVydGVkKCk7IHggPSAwLjAKICAgICAgICBheC50ZXh0KDAsIHkgKyAwLjQ1LCB0aXRsZSwgZm9udHNpemU9OSwgY29sb3I9IiM2NjYiLCBoYT0ibGVmdCIpCiAgICAgICAgZm9yIHcsIHYgaW4gemlwKHdvcmRzLCB2YWxzKToKICAgICAgICAgICAgYmFkID0gdiBpcyBOb25lIG9yIG5wLmlzbmFuKHYpCiAgICAgICAgICAgIGMgPSBHUkVZIGlmIGJhZCBlbHNlIHBsdC5jbS5PcmFuZ2VzKDAuMDggKyAwLjg1ICogZmxvYXQobnAuY2xpcCh2LCAwLCAxKSkpCiAgICAgICAgICAgIHQgPSBheC50ZXh0KHgsIHksIHcsIGZvbnRzaXplPTE3LCBzdHlsZT0iaXRhbGljIiwgZmFtaWx5PSJzZXJpZiIsIHZhPSJjZW50ZXIiLCBoYT0ibGVmdCIsCiAgICAgICAgICAgICAgICAgICAgICAgIGJib3g9ZGljdChib3hzdHlsZT0icm91bmQscGFkPTAuMjUiLCBmYz1jLCBlYz0ibm9uZSIpKQogICAgICAgICAgICBiYiA9IHQuZ2V0X3dpbmRvd19leHRlbnQocmVuZGVyZXI9cik7ICh4MCwgXyksICh4MSwgXykgPSBpbnYudHJhbnNmb3JtKFtbYmIueDAsIGJiLnkwXSwgW2JiLngxLCBiYi55MV1dKQogICAgICAgICAgICBheC50ZXh0KCh4MCArIHgxKSAvIDIsIHkgLSAwLjM2LCAiIiBpZiBiYWQgZWxzZSBmInt2Oi4yZn0iLCBmb250c2l6ZT03LjUsIGNvbG9yPSIjNTU1IiwgaGE9ImNlbnRlciIpCiAgICAgICAgICAgIHggPSB4MSArIDAuMDMKICAgICAgICBheC50ZXh0KDAsIHkgLSAwLjY2LCBzdWIsIGZvbnRzaXplPTksIGNvbG9yPUlOSywgaGE9ImxlZnQiKQogICAgc2hvdyA9IFtzIGZvciBzIGluIFsiZHVlbmRlIiwgIm9sdmlkbyIsICJwZWxvIiwgImVzdHJlbmFyIiwgInRyZW4iXSBpZiBzIGluIEJdCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDksIDEuNDUgKiBsZW4oc2hvdykpKTsgYXguYXhpcygib2ZmIikKICAgIGF4LnNldF94bGltKC0wLjAyLCAxLjI1KTsgYXguc2V0X3lsaW0oLTEuNSAqIGxlbihzaG93KSArIDAuNiwgMC45KQogICAgZm9yIGssIHNpZCBpbiBlbnVtZXJhdGUoc2hvdyk6CiAgICAgICAgcmVjID0gQltzaWRdOyBTID0gU0lEW3NpZF0KICAgICAgICB3ZHMgPSBbdyBmb3IgdywgXywgXyBpbiB3b3Jkc193aXRoX3NwYW5zKFNbInRleHQiXSldCiAgICAgICAgaGVhdF9yb3coYXgsIC0xLjUgKiBrLCB3ZHMsIHJlY1sicnVuZ3MiXVsiQjEiXVsid29yZF9sZWZ0X21hc2tlZCJdLCBTWyJjYXQiXSwgIuKGkiAiICsgbGFkZGVyW3NpZF1bIkIxIl1bOjExMF0pCiAgICBheC5zZXRfeGxpbSgtMC4wMiwgMS4yNSk7IGF4LnNldF95bGltKC0xLjUgKiBsZW4oc2hvdykgKyAwLjYsIDAuOSkKICAgIGF4LnNldF90aXRsZSgiV2hlcmUgZWFjaCBzZW50ZW5jZSByZXNpc3RzIHRyYW5zbGF0aW9uOiBkYXJrZXIgd29yZHMgYXJlIGhhcmRlciB0byByZWNvdmVyIGZyb20gdGhlIEVuZ2xpc2hcbiIKICAgICAgICAgICAgICAgICBmIihzaGFyZSBvZiBlYWNoIHdvcmQncyBpbmZvcm1hdGlvbiBsb3N0IGFmdGVyIGEgc3RhbmRhcmQgdHJhbnNsYXRpb247IHtNT0RFTF9JRC5zcGxpdCgnLycpWy0xXX0pIiwgbG9jPSJsZWZ0IiwgZm9udHNpemU9MTApCiAgICBzYXZlZmlnKGZpZywgIkIxX2hlYXRtYXBzIikKCiAgICBzaG93ID0gW3MgZm9yIHMgaW4gWyJkdWVuZGUiLCAic29icmVtZXNhIiwgInBlbG8iLCAidHJlbiJdIGlmIHMgaW4gQl0KICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oNy41LCAzLjgpKQogICAgZm9yIHNpZCBpbiBzaG93OgogICAgICAgIHJlYyA9IEJbc2lkXTsgY29sID0gRUFTWSBpZiBTSURbc2lkXVsiY2F0Il0gPT0gImNvbnRyb2wiIGVsc2UgSEFSRAogICAgICAgIHB0cyA9IHNvcnRlZCgocmVjWyJydW5ncyJdW3JdWyJsZW4iXSwgcmVjWyJydW5ncyJdW3JdWyJkX21hc2tlZCJdLCByKSBmb3IgciBpbiBOT05fQk9SUk9XKQogICAgICAgIGVudiwgYmVzdCA9IFtdLCAxLjAKICAgICAgICBmb3IgbCwgZCwgciBpbiBwdHM6IGJlc3QgPSBtaW4oYmVzdCwgZCk7IGVudi5hcHBlbmQoKGwsIGJlc3QpKQogICAgICAgIGF4LnN0ZXAoW2wgZm9yIGwsIF8gaW4gZW52XSwgW2QgZm9yIF8sIGQgaW4gZW52XSwgd2hlcmU9InBvc3QiLCBjb2xvcj1jb2wsIGx3PTIsIGFscGhhPTAuOSkKICAgICAgICBheC5zY2F0dGVyKFtsIGZvciBsLCBfLCBfIGluIHB0c10sIFtkIGZvciBfLCBkLCBfIGluIHB0c10sIGNvbG9yPWNvbCwgcz0xOCwgem9yZGVyPTMpCiAgICAgICAgYXgudGV4dChlbnZbLTFdWzBdICogMS4wNSwgZW52Wy0xXVsxXSwgU0lEW3NpZF1bInRleHQiXVs6MjhdICsgIuKApiIsIGZvbnRzaXplPTgsIGNvbG9yPWNvbCwgdmE9ImNlbnRlciIpCiAgICBheC5heGhsaW5lKEVQUywgY29sb3I9IiM4ODgiLCBscz0iLS0iLCBsdz0xKTsgYXgudGV4dChheC5nZXRfeGxpbSgpWzBdLCBFUFMsICIgIGdldHMgYWNyb3NzIiwgZm9udHNpemU9OCwgY29sb3I9IiM4ODgiLCB2YT0iYm90dG9tIikKICAgIGF4LnNldF94c2NhbGUoImxvZyIpOyBheC5zZXRfeGxhYmVsKCJsZW5ndGggb2YgdGhlIEVuZ2xpc2ggKHRva2VucywgbG9nIHNjYWxlKTogd29yZCBmb3Igd29yZCDihpIgc3RhbmRhcmQg4oaSIGdsb3NzIOKGkiBmb290bm90ZSDihpIgdGhpY2siKQogICAgYXguc2V0X3lsYWJlbCgic2hhcmUgb2YgbWVhbmluZyBub3QgY2FycmllZCIpOyBheC5zZXRfdGl0bGUoIkhvdyBtdWNoIEVuZ2xpc2ggZG9lcyBpdCB0YWtlPyIsIGxvYz0ibGVmdCIpCiAgICBzYXZlZmlnKGZpZywgIkIyX2Rpc3RvcnRpb25fdnNfbGVuZ3RoIikKCiAgICBzcGFucyA9IFsoc2lkLCBzcCkgZm9yIHNpZCBpbiBCIGZvciBzcCBpbiBCW3NpZF1bInNlbnNlIl1dWzoxMl0KICAgIGlmIHNwYW5zOgogICAgICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oOSwgMy42KSk7IHcgPSAwLjE1CiAgICAgICAgcnIgPSBbIkIxIiwgIkIyIiwgIkIzIiwgIkI1IiwgIkI2Il07IGNvbHMgPSBbR1JFWSwgTE9BTiwgIiNlM2E1ODciLCAiI2QwN2E1NSIsIEhBUkRdCiAgICAgICAgZm9yIGssIHIgaW4gZW51bWVyYXRlKHJyKToKICAgICAgICAgICAgYXguYmFyKG5wLmFyYW5nZShsZW4oc3BhbnMpKSArIChrIC0gMikgKiB3LCBucC5jbGlwKFtCW3NdWyJzZW5zZSJdW3NwXVsibm9ybSJdW3JdIGZvciBzLCBzcCBpbiBzcGFuc10sIC0wLjUsIDEuNSksIHcsIGNvbG9yPWNvbHNba10sIGxhYmVsPVJVTkdfTkFNRVtyXSkKICAgICAgICBheC5zZXRfeHRpY2tzKG5wLmFyYW5nZShsZW4oc3BhbnMpKSk7IGF4LnNldF94dGlja2xhYmVscyhbc3AgZm9yIF8sIHNwIGluIHNwYW5zXSwgcm90YXRpb249MzAsIGhhPSJyaWdodCIsIHN0eWxlPSJpdGFsaWMiKQogICAgICAgIGF4LmF4aGxpbmUoMSwgY29sb3I9IiM4ODgiLCBsdz0wLjgsIGxzPSI6Iik7IGF4LmF4aGxpbmUoMCwgY29sb3I9IiM4ODgiLCBsdz0wLjgpCiAgICAgICAgYXguc2V0X3lsYWJlbCgiaW50ZW5kZWQgc2Vuc2UgcmVjb3ZlcmVkXG4oMCA9IG5vbmUsIDEgPSBhcyBmcm9tIHRoZSBvcmlnaW5hbCkiKTsgYXguc2V0X3lsaW0oLTAuNTUsIDEuNTUpCiAgICAgICAgYXguc2V0X3RpdGxlKCJEb2VzIHRoZSBFbmdsaXNoIGNhcnJ5IHRoZSBzZW5zZSB0aGUgc2VudGVuY2UgbWVhbnM/IiwgbG9jPSJsZWZ0Iik7IGF4LmxlZ2VuZChmcmFtZW9uPUZhbHNlLCBmb250c2l6ZT04LCBuY29sPTUpCiAgICAgICAgc2F2ZWZpZyhmaWcsICJCM19zZW5zZV9yZWNvdmVyeSIpCgpwYWNrKCkK'))
open('/content/ut/C1_collect.py','wb').write(base64.b64decode('IyBSZWxvYWQgZWFybGllciBzdGFnZXMnIHJlc3VsdHMsIHNvIHRoaXMgc3RhZ2UgYWxzbyB3b3JrcyBpbiBhIGZyZXNoIGtlcm5lbCAoYWZ0ZXIgMDBfc2V0dXAucHkpLgpBID0gbG9hZF9qc29uKCJBX2RldG91ci5qc29uIikgaWYgaGF2ZSgiQV9kZXRvdXIuanNvbiIpIGVsc2UgTm9uZQpsYWRkZXIgPSBsb2FkX2pzb24oIkJfbGFkZGVyLmpzb24iKSBpZiBoYXZlKCJCX2xhZGRlci5qc29uIikgZWxzZSBOb25lCkIgPSBsb2FkX2pzb24oIkJfcm91bmR0cmlwLmpzb24iKSBpZiBoYXZlKCJCX3JvdW5kdHJpcC5qc29uIikgZWxzZSBOb25lCnN1bW1hcnkgPSBsb2FkX2pzb24oIkJfc3VtbWFyeS5qc29uIilbInBlcl9zZW50ZW5jZSJdIGlmIGhhdmUoIkJfc3VtbWFyeS5qc29uIikgZWxzZSB7fQoKCmRlZiBwaWNrX2xheWVycygpOgogICAgZiA9IFswLjI1LCAwLjQsIDAuNSwgMC42LCAwLjc1XQogICAgbHMgPSB7bWF4KDEsIG1pbihOX0xBWUVSUyAtIDEsIHJvdW5kKHggKiBOX0xBWUVSUykpKSBmb3IgeCBpbiBmfQogICAgaWYgQSBhbmQgQS5nZXQoImRldG91cl9wZWFrX2xheWVyIik6IGxzLmFkZChpbnQobWluKEFbImRldG91cl9wZWFrX2xheWVyIl0sIE5fTEFZRVJTIC0gMSkpKQogICAgcmV0dXJuIHNvcnRlZChscykKQ19MQVlFUlMgPSBwaWNrX2xheWVycygpOyBwcmludCgiUGFydCBDIGxheWVyczoiLCBDX0xBWUVSUykKCmRlZiBsb2FkX3BhaXJzKGxhbmcsIG4pOgogICAgIiIibiBwYXJhbGxlbCBzZW50ZW5jZSBwYWlycyAoeCwgZW4sIHNvdXJjZSk7IGEgbWl4IG9mIGRvbWFpbnMgc28gJ2NvbnZlbnRpb25hbCBwcmFjdGljZScgaXNuJ3Qgb25lIHJlZ2lzdGVyLiIiIgogICAgZnJvbSBkYXRhc2V0cyBpbXBvcnQgbG9hZF9kYXRhc2V0CiAgICAjIEV1cm9wYXJsIGZpcnN0OiBpZiBpdCBmYWlscyB0byBsb2FkLCBPUFVTLTEwMCB0YWtlcyB0aGUgd2hvbGUgcXVvdGEKICAgIHNvdXJjZXMgPSAoWygiSGVsc2lua2ktTkxQL2V1cm9wYXJsIiwgImVuLWVzIildIGlmIGxhbmcgPT0gImVzIiBlbHNlIFtdKSArIFsoIkhlbHNpbmtpLU5MUC9vcHVzLTEwMCIsIGYiZW4te2xhbmd9IildCiAgICB3YW50LCBwYWlycywgc2VlbiA9IGludChuICogMS4zKSwgW10sIHNldCgpCiAgICBmb3IgaywgKG5hbWUsIGNmZykgaW4gZW51bWVyYXRlKHNvdXJjZXMpOgogICAgICAgIHF1b3RhID0gKHdhbnQgLSBsZW4ocGFpcnMpKSAvLyAobGVuKHNvdXJjZXMpIC0gaykKICAgICAgICB0cnk6IGRzID0gbG9hZF9kYXRhc2V0KG5hbWUsIGNmZywgc3BsaXQ9InRyYWluIiwgc3RyZWFtaW5nPVRydWUpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiBwcmludChmIiAgY291bGQgbm90IGxvYWQge25hbWV9L3tjZmd9OiB7ZX0iKTsgY29udGludWUKICAgICAgICBnb3QgPSAwCiAgICAgICAgZm9yIGV4IGluIGRzOgogICAgICAgICAgICB0ciA9IGV4LmdldCgidHJhbnNsYXRpb24iLCB7fSk7IGVuLCB4eCA9IHRyLmdldCgiZW4iKSwgdHIuZ2V0KGxhbmcpCiAgICAgICAgICAgIGlmIG5vdCBlbiBvciBub3QgeHg6IGNvbnRpbnVlCiAgICAgICAgICAgIG5lLCBueCA9IGxlbihlbi5zcGxpdCgpKSwgbGVuKHh4LnNwbGl0KCkpCiAgICAgICAgICAgIGlmIG5vdCAoOCA8PSBuZSA8PSA0MCBhbmQgOCA8PSBueCA8PSA0MCkgb3IgYWJzKG5lIC0gbngpID4gMC41ICogbWF4KG5lLCBueCk6IGNvbnRpbnVlCiAgICAgICAgICAgIGlmIHN1bShjaC5pc2RpZ2l0KCkgZm9yIGNoIGluIHh4KSA+IDYgb3IgeHggaW4gc2VlbjogY29udGludWUKICAgICAgICAgICAgc2Vlbi5hZGQoeHgpOyBwYWlycy5hcHBlbmQoKHh4LnN0cmlwKCksIGVuLnN0cmlwKCksIG5hbWUpKTsgZ290ICs9IDEKICAgICAgICAgICAgaWYgZ290ID49IHF1b3RhOiBicmVhawogICAgICAgIHByaW50KGYiICB7bmFtZX0ve2NmZ306IHtnb3R9IHBhaXJzIikKICAgIHJhbmRvbS5SYW5kb20oU0VFRCkuc2h1ZmZsZShwYWlycykKICAgIHJldHVybiBwYWlyc1s6bl0KCmRlZiB3b3Jkc19mb3IobGFuZywgdGV4dHMpOgogICAgIiIiV29yZCB0b2tlbnMgKHRleHQsIHN0YXJ0LCBlbmQsIG1vcnBoLCBwb3MpLiBTcGFuaXNoIHVzZXMgc3BhQ3kgd2hlbiBhdmFpbGFibGUsIGZvciBtb3JwaG9sb2d5LiIiIgogICAgaWYgbGFuZyA9PSAiZXMiOgogICAgICAgIHRyeToKICAgICAgICAgICAgaW1wb3J0IHNwYWN5CiAgICAgICAgICAgIG5scCA9IHNwYWN5LmxvYWQoImVzX2NvcmVfbmV3c19zbSIsIGRpc2FibGU9WyJwYXJzZXIiLCAibmVyIl0pCiAgICAgICAgICAgIG91dCA9IFtdCiAgICAgICAgICAgIGZvciBkb2MgaW4gbmxwLnBpcGUodGV4dHMsIGJhdGNoX3NpemU9MjU2KToKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoWyh0LnRleHQsIHQuaWR4LCB0LmlkeCArIGxlbih0LnRleHQpLCBzdHIodC5tb3JwaCksIHQucG9zXykgZm9yIHQgaW4gZG9jCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBXT1JEX1JFLmZ1bGxtYXRjaCh0LnRleHQpXSkKICAgICAgICAgICAgcmV0dXJuIG91dAogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgcHJpbnQoIiAgc3BhQ3kgdW5hdmFpbGFibGUsIHVzaW5nIHJlZ2V4IHdvcmRzIChubyBtb3JwaG9sb2d5IHByb2Jlcyk6IiwgZSkKICAgIHJldHVybiBbWyh3LCBhLCBiLCAiIiwgIiIpIGZvciB3LCBhLCBiIGluIHdvcmRzX3dpdGhfc3BhbnModCldIGZvciB0IGluIHRleHRzXQoKX0FMID0gTm9uZQpkZWYgYWxpZ25fcGFpcnMoc3JjX3dvcmRzLCB0Z3Rfd29yZHMsIGJhdGNoPTY0LCBsYXllcj04KToKICAgICIiIjEtdG8tMSB3b3JkIGxpbmtzIChpLCBqKSBwZXIgc2VudGVuY2UgcGFpcjogbXV0dWFsIG5lYXJlc3QgbmVpZ2hib3VycyBvZiBYTE0tUiAobGF5ZXIgOCkgd29yZCBlbWJlZGRpbmdzLAogICAgaS5lLiBTaW1BbGlnbidzIGhpZ2gtcHJlY2lzaW9uICdhcmdtYXgnIG1ldGhvZCwgaW1wbGVtZW50ZWQgaGVyZSB0byBhdm9pZCBhIGZyYWdpbGUgZGVwZW5kZW5jeS4iIiIKICAgIGdsb2JhbCBfQUwKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvVG9rZW5pemVyLCBBdXRvTW9kZWwKICAgIGRldiA9ICJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIKICAgIGlmIF9BTCBpcyBOb25lOgogICAgICAgIF9BTCA9IChBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZCgiRmFjZWJvb2tBSS94bG0tcm9iZXJ0YS1iYXNlIiksCiAgICAgICAgICAgICAgIEF1dG9Nb2RlbC5mcm9tX3ByZXRyYWluZWQoIkZhY2Vib29rQUkveGxtLXJvYmVydGEtYmFzZSIpLnRvKGRldikuZXZhbCgpKQogICAgYXRvaywgYW1vZCA9IF9BTAogICAgZGVmIGVtYmVkKHdvcmRfbGlzdHMpOgogICAgICAgIHdsXyA9IFtbd1swXSBmb3IgdyBpbiB3bF0gb3IgWyIuIl0gZm9yIHdsIGluIHdvcmRfbGlzdHNdCiAgICAgICAgZSA9IGF0b2sod2xfLCBpc19zcGxpdF9pbnRvX3dvcmRzPVRydWUsIHJldHVybl90ZW5zb3JzPSJwdCIsIHBhZGRpbmc9VHJ1ZSwgdHJ1bmNhdGlvbj1UcnVlLCBtYXhfbGVuZ3RoPTI1NikKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgSCA9IGFtb2QoKip7azogdi50byhkZXYpIGZvciBrLCB2IGluIGUuaXRlbXMoKX0sIG91dHB1dF9oaWRkZW5fc3RhdGVzPVRydWUpLmhpZGRlbl9zdGF0ZXNbbGF5ZXJdCiAgICAgICAgb3V0ID0gW10KICAgICAgICBmb3IgYiwgd2wgaW4gZW51bWVyYXRlKHdsXyk6CiAgICAgICAgICAgIE0gPSB0b3JjaC56ZXJvcyhsZW4od2wpLCBILnNoYXBlWzFdLCBkZXZpY2U9ZGV2KQogICAgICAgICAgICBmb3IgcCwgdyBpbiBlbnVtZXJhdGUoZS53b3JkX2lkcyhiKSk6CiAgICAgICAgICAgICAgICBpZiB3IGlzIG5vdCBOb25lOiBNW3csIHBdID0gMS4wCiAgICAgICAgICAgIGNudCA9IE0uc3VtKDEsIGtlZXBkaW09VHJ1ZSkKICAgICAgICAgICAgb3V0LmFwcGVuZCgodG9yY2gubm4uZnVuY3Rpb25hbC5ub3JtYWxpemUoKE0gQCBIW2JdLmZsb2F0KCkpIC8gY250LmNsYW1wX21pbigxKSwgZGltPS0xKSwgY250WzosIDBdID4gMCkpCiAgICAgICAgcmV0dXJuIG91dAogICAgbGlua3MgPSBbXQogICAgZm9yIGkgaW4gcmFuZ2UoMCwgbGVuKHNyY193b3JkcyksIGJhdGNoKToKICAgICAgICBlcywgZXQgPSBlbWJlZChzcmNfd29yZHNbaTppICsgYmF0Y2hdKSwgZW1iZWQodGd0X3dvcmRzW2k6aSArIGJhdGNoXSkKICAgICAgICBmb3IgKGEsIHZhKSwgKGIsIHZiKSwgc3csIHR3IGluIHppcChlcywgZXQsIHNyY193b3Jkc1tpOmkgKyBiYXRjaF0sIHRndF93b3Jkc1tpOmkgKyBiYXRjaF0pOgogICAgICAgICAgICBpZiBub3Qgc3cgb3Igbm90IHR3OiBsaW5rcy5hcHBlbmQoW10pOyBjb250aW51ZQogICAgICAgICAgICBTID0gYSBAIGIuVDsgU1t+dmFdID0gLTI7IFNbOiwgfnZiXSA9IC0yCiAgICAgICAgICAgIGZ3ZCwgYndkID0gUy5hcmdtYXgoMSksIFMuYXJnbWF4KDApCiAgICAgICAgICAgIGxpbmtzLmFwcGVuZChbKGssIGludChmd2Rba10pKSBmb3IgayBpbiByYW5nZShsZW4oc3cpKSBpZiBpbnQoYndkW2Z3ZFtrXV0pID09IGsgYW5kIGZsb2F0KFNbaywgZndkW2tdXSkgPiAtMQogICAgICAgICAgICAgICAgICAgICAgICAgIGFuZCBub3QgYW55KGNoLmlzZGlnaXQoKSBmb3IgY2ggaW4gc3dba11bMF0gKyB0d1tpbnQoZndkW2tdKV1bMF0pXSkKICAgIHJldHVybiBsaW5rcwoKIyAtLS0tIGNvbGxlY3Rpb24gLS0tLQoKY2xhc3MgQ292QWNjOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGQsIGRldik6CiAgICAgICAgeiA9IGxhbWJkYSAqczogdG9yY2guemVyb3MoKnMsIGR0eXBlPXRvcmNoLmZsb2F0NjQsIGRldmljZT1kZXYpCiAgICAgICAgc2VsZi5uLCBzZWxmLnN4LCBzZWxmLnN5ID0gMCwgeihkKSwgeihkKQogICAgICAgIHNlbGYuU3h4LCBzZWxmLlN5eSwgc2VsZi5TeHkgPSB6KGQsIGQpLCB6KGQsIGQpLCB6KGQsIGQpCiAgICBkZWYgYWRkKHNlbGYsIFgsIFkpOgogICAgICAgIFgsIFkgPSBYLmRvdWJsZSgpLCBZLmRvdWJsZSgpCiAgICAgICAgc2VsZi5uICs9IFguc2hhcGVbMF07IHNlbGYuc3ggKz0gWC5zdW0oMCk7IHNlbGYuc3kgKz0gWS5zdW0oMCkKICAgICAgICBzZWxmLlN4eCArPSBYLlQgQCBYOyBzZWxmLlN5eSArPSBZLlQgQCBZOyBzZWxmLlN4eSArPSBYLlQgQCBZCiAgICBkZWYgZmluaXNoKHNlbGYpOgogICAgICAgIG4gPSBzZWxmLm47IG14LCBteSA9IHNlbGYuc3ggLyBuLCBzZWxmLnN5IC8gbgogICAgICAgIHJldHVybiBkaWN0KG49biwgbXg9bXguY3B1KCksIG15PW15LmNwdSgpLAogICAgICAgICAgICAgICAgICAgIEN4eD0oKHNlbGYuU3h4IC0gbiAqIHRvcmNoLm91dGVyKG14LCBteCkpIC8gKG4gLSAxKSkuY3B1KCksCiAgICAgICAgICAgICAgICAgICAgQ3l5PSgoc2VsZi5TeXkgLSBuICogdG9yY2gub3V0ZXIobXksIG15KSkgLyAobiAtIDEpKS5jcHUoKSwKICAgICAgICAgICAgICAgICAgICBDeHk9KChzZWxmLlN4eSAtIG4gKiB0b3JjaC5vdXRlcihteCwgbXkpKSAvIChuIC0gMSkpLmNwdSgpKQoKZGVmIHdvcmRfc3RhdGVzKHRleHRzLCB3b3JkX2xpc3RzLCBsYXllcnMsIGJhdGNoPTMyKToKICAgICIiIkZvciBlYWNoIHRleHQ6IGRpY3QgbGF5ZXIgLT4gW25fd29yZHMsIGRdIGZsb2F0MzIgKHdvcmQgPSBtZWFuIG9mIGl0cyB0b2tlbnMpLCBwbHVzIGEgdmFsaWRpdHkgbWFzawogICAgKHdvcmRzIGNvbnRhaW5pbmcgdGhlIGZpcnN0IGNvbnRlbnQgdG9rZW4sIGFuIGF0dGVudGlvbiBzaW5rLCBhcmUgaW52YWxpZCkuIiIiCiAgICBvdXQgPSBbXQogICAgZm9yIGkgaW4gcmFuZ2UoMCwgbGVuKHRleHRzKSwgYmF0Y2gpOgogICAgICAgIGVuY18gPSBbZW5jb2RlX3Jhdyh0KSBmb3IgdCBpbiB0ZXh0c1tpOmkgKyBiYXRjaF1dCiAgICAgICAgTCA9IG1heChsZW4oZVswXSkgZm9yIGUgaW4gZW5jXykKICAgICAgICBpZHMgPSB0b3JjaC5mdWxsKChsZW4oZW5jXyksIEwpLCB0b2sucGFkX3Rva2VuX2lkLCBkdHlwZT10b3JjaC5sb25nKTsgYW0gPSB0b3JjaC56ZXJvc19saWtlKGlkcykKICAgICAgICBmb3IgYiwgKHMsIF8pIGluIGVudW1lcmF0ZShlbmNfKTogaWRzW2IsIDpsZW4ocyldID0gdG9yY2gudGVuc29yKHMpOyBhbVtiLCA6bGVuKHMpXSA9IDEKICAgICAgICBzdCA9IGNhcHR1cmUoaWRzLnRvKElOUFVUX0RFViksIGFtLnRvKElOUFVUX0RFViksIGxheWVycykKICAgICAgICBmb3IgYiwgKHMsIG9mZnMpIGluIGVudW1lcmF0ZShlbmNfKToKICAgICAgICAgICAgd2wgPSB3b3JkX2xpc3RzW2kgKyBiXTsgZmlyc3QgPSBsZW4oQk9TKQogICAgICAgICAgICB0b2tzID0gW3Rva2Vuc19pbl9zcGFuKG9mZnMsIHdbMV0sIHdbMl0pIGZvciB3IGluIHdsXQogICAgICAgICAgICB2YWxpZCA9IG5wLmFycmF5KFtsZW4odCkgPiAwIGFuZCBmaXJzdCBub3QgaW4gdCBmb3IgdCBpbiB0b2tzXSwgYm9vbCkKICAgICAgICAgICAgTSA9IHRvcmNoLnplcm9zKGxlbih3bCksIGxlbihzKSkKICAgICAgICAgICAgZm9yIGssIHQgaW4gZW51bWVyYXRlKHRva3MpOgogICAgICAgICAgICAgICAgaWYgdDogTVtrLCB0XSA9IDEuMCAvIGxlbih0KQogICAgICAgICAgICBNID0gTS50byhzdFtsYXllcnNbMF1dLmRldmljZSkKICAgICAgICAgICAgb3V0LmFwcGVuZCgoe2w6IE0gQCBzdFtsXVtiLCA6bGVuKHMpXS5mbG9hdCgpIGZvciBsIGluIGxheWVyc30sIHZhbGlkKSkKICAgICAgICBkZWwgc3QKICAgIHJldHVybiBvdXQKCmRlZiBjb2xsZWN0X3BhaXIobGFuZyk6CiAgICB0YWcgPSBmIkNfe2xhbmd9IgogICAgY292X3BhdGhzID0ge2w6IG9zLnBhdGguam9pbihDQUNIRSwgZiJ7dGFnfV9jb3ZfTHtsfS5wdCIpIGZvciBsIGluIENfTEFZRVJTfQogICAgaGVsZF9wYXRoID0gb3MucGF0aC5qb2luKENBQ0hFLCBmInt0YWd9X2hlbGQucHQiKQogICAgaWYgYWxsKG9zLnBhdGguZXhpc3RzKHApIGZvciBwIGluIGNvdl9wYXRocy52YWx1ZXMoKSkgYW5kIG9zLnBhdGguZXhpc3RzKGhlbGRfcGF0aCk6IHJldHVybgogICAgd2l0aCBUaW1lcihmIlBhcnQgQyBbe2xhbmd9XTogY29ycHVzIik6CiAgICAgICAgcGFpcnMgPSBsb2FkX3BhaXJzKGxhbmcsIE5fU0VOVF9GSVQgKyBOX1NFTlRfSEVMRCkKICAgICAgICB4cywgZW5zID0gW3BbMF0gZm9yIHAgaW4gcGFpcnNdLCBbcFsxXSBmb3IgcCBpbiBwYWlyc10KICAgICAgICB3eCwgd2UgPSB3b3Jkc19mb3IobGFuZywgeHMpLCB3b3Jkc19mb3IoImVuIiwgZW5zKQogICAgd2l0aCBUaW1lcihmIlBhcnQgQyBbe2xhbmd9XTogd29yZCBhbGlnbm1lbnQgKHtsZW4ocGFpcnMpfSBwYWlycykiKToKICAgICAgICBsaW5rcyA9IGFsaWduX3BhaXJzKHd4LCB3ZSkKICAgIG5fZml0ID0gbGVuKHBhaXJzKSAtIE5fU0VOVF9IRUxECiAgICBhY2MgPSB7bDogQ292QWNjKERfTU9ERUwsIElOUFVUX0RFVikgZm9yIGwgaW4gQ19MQVlFUlN9CiAgICBoZWxkID0ge2w6IChbXSwgW10pIGZvciBsIGluIENfTEFZRVJTfTsgaGVsZF9tZXRhID0gW10KICAgIHdpdGggVGltZXIoZiJQYXJ0IEMgW3tsYW5nfV06IGNvbGxlY3Rpbmcgc3RhdGVzIGF0IGxheWVycyB7Q19MQVlFUlN9Iik6CiAgICAgICAgQl8gPSAzMgogICAgICAgIGZvciBpIGluIHJhbmdlKDAsIGxlbihwYWlycyksIEJfKToKICAgICAgICAgICAgc3ggPSB3b3JkX3N0YXRlcyh4c1tpOmkgKyBCX10sIHd4W2k6aSArIEJfXSwgQ19MQVlFUlMpCiAgICAgICAgICAgIHN5ID0gd29yZF9zdGF0ZXMoZW5zW2k6aSArIEJfXSwgd2VbaTppICsgQl9dLCBDX0xBWUVSUykKICAgICAgICAgICAgZm9yIGIgaW4gcmFuZ2UobGVuKHN4KSk6CiAgICAgICAgICAgICAgICBrID0gaSArIGI7IChyeCwgdngpLCAocnksIHZ5KSA9IHN4W2JdLCBzeVtiXQogICAgICAgICAgICAgICAgbGsgPSBbKGEsIGMpIGZvciBhLCBjIGluIGxpbmtzW2tdIGlmIHZ4W2FdIGFuZCB2eVtjXV0KICAgICAgICAgICAgICAgIGlmIG5vdCBsazogY29udGludWUKICAgICAgICAgICAgICAgIGlhID0gdG9yY2gudGVuc29yKFthIGZvciBhLCBfIGluIGxrXSk7IGljID0gdG9yY2gudGVuc29yKFtjIGZvciBfLCBjIGluIGxrXSkKICAgICAgICAgICAgICAgIGZvciBsIGluIENfTEFZRVJTOgogICAgICAgICAgICAgICAgICAgIFgsIFkgPSByeFtsXVtpYS50byhyeFtsXS5kZXZpY2UpXSwgcnlbbF1baWMudG8ocnlbbF0uZGV2aWNlKV0KICAgICAgICAgICAgICAgICAgICBpZiBrIDwgbl9maXQ6IGFjY1tsXS5hZGQoWCwgWSkKICAgICAgICAgICAgICAgICAgICBlbHNlOiBoZWxkW2xdWzBdLmFwcGVuZChYLmhhbGYoKS5jcHUoKSk7IGhlbGRbbF1bMV0uYXBwZW5kKFkuaGFsZigpLmNwdSgpKQogICAgICAgICAgICAgICAgaWYgayA+PSBuX2ZpdDoKICAgICAgICAgICAgICAgICAgICBoZWxkX21ldGEgKz0gW2RpY3Qoc2VudD1rLCB4PXd4W2tdW2FdWzBdLCB5PXdlW2tdW2NdWzBdLCBtb3JwaD13eFtrXVthXVszXSwgcG9zPXd4W2tdW2FdWzRdKSBmb3IgYSwgYyBpbiBsa10KICAgICAgICAgICAgaWYgKGkgLy8gQl8pICUgMTAwID09IDA6IHByaW50KGYiICAgIHtpfS97bGVuKHBhaXJzKX0gc2VudGVuY2VzLCB7YWNjW0NfTEFZRVJTWzBdXS5ufSBmaXQgcGFpcnMiKQogICAgZm9yIGwgaW4gQ19MQVlFUlM6IHRvcmNoLnNhdmUoYWNjW2xdLmZpbmlzaCgpLCBjb3ZfcGF0aHNbbF0pCiAgICB0b3JjaC5zYXZlKGRpY3QoWD17bDogdG9yY2guY2F0KGhlbGRbbF1bMF0pIGZvciBsIGluIENfTEFZRVJTfSwgWT17bDogdG9yY2guY2F0KGhlbGRbbF1bMV0pIGZvciBsIGluIENfTEFZRVJTfSwKICAgICAgICAgICAgICAgICAgICBtZXRhPWhlbGRfbWV0YSwgbl9maXRfcGFpcnM9YWNjW0NfTEFZRVJTWzBdXS5uLCBuX3NlbnQ9bGVuKHBhaXJzKSksIGhlbGRfcGF0aCkKICAgIGRlbCBhY2M7IGdjLmNvbGxlY3QoKTsgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSBOb25lCgppZiBSVU5fQzoKICAgIGZvciBsYW5nIGluIFBBSVJTOiBjb2xsZWN0X3BhaXIobGFuZykKCnBhY2soKQo='))
open('/content/ut/C2_spectrum.py','wb').write(base64.b64decode('IyBSZWxvYWQgZWFybGllciBzdGFnZXMnIHJlc3VsdHMsIHNvIHRoaXMgc3RhZ2UgYWxzbyB3b3JrcyBpbiBhIGZyZXNoIGtlcm5lbCAoYWZ0ZXIgMDBfc2V0dXAucHkpLgpBID0gbG9hZF9qc29uKCJBX2RldG91ci5qc29uIikgaWYgaGF2ZSgiQV9kZXRvdXIuanNvbiIpIGVsc2UgTm9uZQpsYWRkZXIgPSBsb2FkX2pzb24oIkJfbGFkZGVyLmpzb24iKSBpZiBoYXZlKCJCX2xhZGRlci5qc29uIikgZWxzZSBOb25lCkIgPSBsb2FkX2pzb24oIkJfcm91bmR0cmlwLmpzb24iKSBpZiBoYXZlKCJCX3JvdW5kdHJpcC5qc29uIikgZWxzZSBOb25lCnN1bW1hcnkgPSBsb2FkX2pzb24oIkJfc3VtbWFyeS5qc29uIilbInBlcl9zZW50ZW5jZSJdIGlmIGhhdmUoIkJfc3VtbWFyeS5qc29uIikgZWxzZSB7fQoKCkNERVYgPSAiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiCgpkZWYgaW52X3NxcnQoQyk6CiAgICBlLCBWID0gdG9yY2gubGluYWxnLmVpZ2goQyk7IHJldHVybiAoViAqIGUuY2xhbXBfbWluKDFlLTEyKS5yc3FydCgpKSBAIFYuVAoKZGVmIGNjYV9maXQoY292LCBsYW0pOgogICAgQ3h4LCBDeXksIEN4eSA9IChjb3Zba10udG8oQ0RFVikgZm9yIGsgaW4gWyJDeHgiLCAiQ3l5IiwgIkN4eSJdKTsgZCA9IEN4eC5zaGFwZVswXQogICAgSSA9IHRvcmNoLmV5ZShkLCBkdHlwZT1DeHguZHR5cGUsIGRldmljZT1DREVWKQogICAgQ3h4X3IgPSBDeHggKyBsYW0gKiB0b3JjaC50cmFjZShDeHgpIC8gZCAqIEk7IEN5eV9yID0gQ3l5ICsgbGFtICogdG9yY2gudHJhY2UoQ3l5KSAvIGQgKiBJCiAgICBXeCwgV3kgPSBpbnZfc3FydChDeHhfciksIGludl9zcXJ0KEN5eV9yKQogICAgVSwgUywgVmggPSB0b3JjaC5saW5hbGcuc3ZkKFd4IEAgQ3h5IEAgV3kpCiAgICByZXR1cm4gZGljdChsYW09bGFtLCByaG9faW49UywgVT1VLCBXeD1XeCwgV3k9V3ksIEE9V3ggQCBVLCBCPVd5IEAgVmguVCwgQ3h4X3I9Q3h4X3IsIEN5eV9yPUN5eV9yLAogICAgICAgICAgICAgICAgQ3h4PUN4eCwgQ3l5PUN5eSwgQ3h5PUN4eSwgbXg9Y292WyJteCJdLnRvKENERVYpLCBteT1jb3ZbIm15Il0udG8oQ0RFVikpCgpkZWYgY29sY29ycihQLCBRKToKICAgIFAgPSBQIC0gUC5tZWFuKDApOyBRID0gUSAtIFEubWVhbigwKQogICAgcmV0dXJuIChQICogUSkuc3VtKDApIC8gKFAubm9ybShkaW09MCkgKiBRLm5vcm0oZGltPTApICsgMWUtMTIpCgpkZWYgaGVsZG91dF9yaG8oZml0LCBYLCBZKToKICAgIHJldHVybiBjb2xjb3JyKChYIC0gZml0WyJteCJdKSBAIGZpdFsiQSJdLCAoWSAtIGZpdFsibXkiXSkgQCBmaXRbIkIiXSkKCmRlZiByMihZYywgWXApOgogICAgcmF3ID0gMSAtICgoWWMgLSBZcCkgKiogMikuc3VtKCkgLyAoWWMgKiogMikuc3VtKCkKICAgIHBlciA9IDEgLSAoKFljIC0gWXApICoqIDIpLnN1bSgwKSAvICgoWWMgLSBZYy5tZWFuKDApKSAqKiAyKS5zdW0oMCkuY2xhbXBfbWluKDFlLTEyKQogICAgcmV0dXJuIGZsb2F0KHJhdyksIGZsb2F0KHBlci5tZWFuKCkpCgpkZWYgYW5hbHlzZV9sYXllcihsYW5nLCBsKToKICAgIGNvdiA9IHRvcmNoLmxvYWQob3MucGF0aC5qb2luKENBQ0hFLCBmIkNfe2xhbmd9X2Nvdl9Me2x9LnB0IiksIHdlaWdodHNfb25seT1GYWxzZSkKICAgIGhlbGQgPSB0b3JjaC5sb2FkKG9zLnBhdGguam9pbihDQUNIRSwgZiJDX3tsYW5nfV9oZWxkLnB0IiksIHdlaWdodHNfb25seT1GYWxzZSkKICAgIFgsIFkgPSBoZWxkWyJYIl1bbF0udG8oQ0RFVikuZG91YmxlKCksIGhlbGRbIlkiXVtsXS50byhDREVWKS5kb3VibGUoKQogICAgc2VudCA9IHRvcmNoLnRlbnNvcihbbVsic2VudCJdIGZvciBtIGluIGhlbGRbIm1ldGEiXV0pCiAgICBoYWxmID0gKHNlbnQgJSAyID09IDApLnRvKENERVYpICAgICAgICAgICAgICAgICAgICAgICAjIHNwbGl0IGhlbGQtb3V0IGJ5IHNlbnRlbmNlCiAgICBiZXN0ID0gTm9uZQogICAgZm9yIGxhbSBpbiBMQU1CREFTOgogICAgICAgIGZpdCA9IGNjYV9maXQoY292LCBsYW0pOyByaG8gPSBoZWxkb3V0X3JobyhmaXQsIFhbaGFsZl0sIFlbaGFsZl0pCiAgICAgICAgczUwID0gZmxvYXQocmhvWzo1MF0uc3VtKCkpCiAgICAgICAgaWYgYmVzdCBpcyBOb25lIG9yIHM1MCA+IGJlc3RbMF06IGJlc3QgPSAoczUwLCBsYW0pCiAgICAgICAgZGVsIGZpdAogICAgZml0ID0gY2NhX2ZpdChjb3YsIGJlc3RbMV0pCiAgICByaG9faCA9IGhlbGRvdXRfcmhvKGZpdCwgWFt+aGFsZl0sIFlbfmhhbGZdKSAgICAgICAgICAjIHJlcG9ydGVkIG9uIHRoZSBvdGhlciBoYWxmCiAgICBwZXJtID0gdG9yY2gucmFuZHBlcm0oaW50KCh+aGFsZikuc3VtKCkpLCBnZW5lcmF0b3I9dG9yY2guR2VuZXJhdG9yKCkubWFudWFsX3NlZWQoU0VFRCkpLnRvKENERVYpCiAgICByaG9fbnVsbCA9IGhlbGRvdXRfcmhvKGZpdCwgWFt+aGFsZl0sIFlbfmhhbGZdW3Blcm1dKQogICAgdGhyID0gZmxvYXQodG9yY2gucXVhbnRpbGUocmhvX251bGwuYWJzKCkuZmxvYXQoKSwgMC45OSkpCiAgICBzaGFyZWQgPSB0b3JjaC5ub256ZXJvKHJob19oID4gdGhyKVs6LCAwXQogICAgIyBFbmdsaXNoIGJpYXM6IGhlbGQtb3V0IHNoYXJlIG9mIGVhY2ggbGFuZ3VhZ2UncyB2YXJpYW5jZSBwcmVkaWN0YWJsZSBmcm9tIHRoZSBvdGhlcgogICAgWGMsIFljID0gWFt+aGFsZl0gLSBmaXRbIm14Il0sIFlbfmhhbGZdIC0gZml0WyJteSJdCiAgICBHeHkgPSB0b3JjaC5saW5hbGcuc29sdmUoZml0WyJDeHhfciJdLCBmaXRbIkN4eSJdKTsgR3l4ID0gdG9yY2gubGluYWxnLnNvbHZlKGZpdFsiQ3l5X3IiXSwgZml0WyJDeHkiXS5UKQogICAgRV9lbiwgRV9lbl9zdGQgPSByMihZYywgWGMgQCBHeHkpICAgICAgICAgICMgRW5nbGlzaCBwcmVkaWN0ZWQgZnJvbSB0aGUgb3RoZXIgbGFuZ3VhZ2UKICAgIEVfeCwgRV94X3N0ZCA9IHIyKFhjLCBZYyBAIEd5eCkgICAgICAgICAgICAjIG90aGVyIGxhbmd1YWdlIHByZWRpY3RlZCBmcm9tIEVuZ2xpc2gKICAgIHNwZWNfeCA9IGZsb2F0KHRvcmNoLnRyYWNlKGZpdFsiQ3h5Il0gQCBHeXgpIC8gdG9yY2gudHJhY2UoZml0WyJDeHgiXSkpCiAgICByZXMgPSBkaWN0KGxheWVyPWwsIGxhbT1iZXN0WzFdLCBuX2ZpdF9wYWlycz1pbnQoY292WyJuIl0pLCBuX2hlbGRfcGFpcnM9aW50KCh+aGFsZikuc3VtKCkpLAogICAgICAgICAgICAgICByaG9faW49Zml0WyJyaG9faW4iXVs6MTAyNF0uY3B1KCkubnVtcHkoKSwgcmhvX2hlbGQ9cmhvX2hbOjEwMjRdLmNwdSgpLm51bXB5KCksCiAgICAgICAgICAgICAgIG51bGxfdGhyPXRociwgbl9zaGFyZWQ9aW50KGxlbihzaGFyZWQpKSwKICAgICAgICAgICAgICAgbl9zdHJvbmc9aW50KChyaG9faCA+IDAuNSAqKiAwLjUpLnN1bSgpKSwgICAjIGRpcmVjdGlvbnMgdGhhdCBrZWVwIG1vcmUgdGhhbiBoYWxmIHBlciByb3VuZCB0cmlwIChyaG9eMiA+IDAuNSkKICAgICAgICAgICAgICAgbl9zdHJvbmdfdG9wPWludCgoZml0WyJyaG9faW4iXSA+IDAuNSAqKiAwLjUpLnN1bSgpKSwgdG9wNTBfaGVsZD1mbG9hdChyaG9faFs6NTBdLnN1bSgpKSwgdG9wNTBfaW49ZmxvYXQoZml0WyJyaG9faW4iXVs6NTBdLnN1bSgpKSwKICAgICAgICAgICAgICAgRV9lbl9mcm9tX3g9RV9lbiwgRV94X2Zyb21fZW49RV94LCBFX2VuX2Zyb21feF9zdGQ9RV9lbl9zdGQsIEVfeF9mcm9tX2VuX3N0ZD1FX3hfc3RkLAogICAgICAgICAgICAgICBiaWFzPUVfZW4gLSBFX3gsIGJpYXNfc3RkPUVfZW5fc3RkIC0gRV94X3N0ZCwgRV94X3NwZWN0cmFsX2luX3NhbXBsZT1zcGVjX3gpCiAgICByZXR1cm4gcmVzLCBmaXQsIHJob19oLCBzaGFyZWQsIEd4eSwgR3l4CgpDX1JFUywgQ19GSVRTLCBCRVNUX0xBWUVSID0ge30sIHt9LCB7fQppZiBSVU5fQzoKICAgIGZvciBsYW5nIGluIFBBSVJTOgogICAgICAgIHdpdGggVGltZXIoZiJQYXJ0IEMgW3tsYW5nfV06IENDQSBhdCB7bGVuKENfTEFZRVJTKX0gbGF5ZXJzIik6CiAgICAgICAgICAgIHJvd3MgPSBbXQogICAgICAgICAgICBmb3IgbCBpbiBDX0xBWUVSUzoKICAgICAgICAgICAgICAgIG91dCA9IGFuYWx5c2VfbGF5ZXIobGFuZywgbCk7IHJlcyA9IG91dFswXTsgcm93cy5hcHBlbmQocmVzKTsgZGVsIG91dAogICAgICAgICAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKTogdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCiAgICAgICAgICAgICAgICBwcmludChmIiAgTHtsOjNkfSAgzrs9e3Jlc1snbGFtJ106LjBlfSAgc2hhcmVkIGRpcnM9e3Jlc1snbl9zaGFyZWQnXTo1ZH0gKHJobz4wLjcxOiB7cmVzWyduX3N0cm9uZyddOjRkfSkgIHRvcC01MCDPgSBoZWxkL2luLXNhbXBsZT0iCiAgICAgICAgICAgICAgICAgICAgICBmIntyZXNbJ3RvcDUwX2hlbGQnXTouMWZ9L3tyZXNbJ3RvcDUwX2luJ106LjFmfSAgRV9lbuKGkHtsYW5nfT17cmVzWydFX2VuX2Zyb21feCddOi4zZn0gICIKICAgICAgICAgICAgICAgICAgICAgIGYiRV97bGFuZ33ihpBlbj17cmVzWydFX3hfZnJvbV9lbiddOi4zZn0gIGJpYXM9e3Jlc1snYmlhcyddOisuM2Z9IikKICAgICAgICAgICAgc2F2ZV9qc29uKGYiQ19zcGVjdHJ1bV97bGFuZ30uanNvbiIsIHJvd3MpOyBDX1JFU1tsYW5nXSA9IHJvd3MKICAgICAgICAgICAgQkVTVF9MQVlFUltsYW5nXSA9IG1heChyb3dzLCBrZXk9bGFtYmRhIHI6IHJbInRvcDUwX2hlbGQiXSlbImxheWVyIl0KICAgIHByaW50KCJiZXN0IChtb3N0IHNoYXJlZCkgbGF5ZXI6IiwgQkVTVF9MQVlFUikKICAgICMgcmVmaXQgYW5kIGtlZXAgb25seSB3aGF0IGxhdGVyIGNlbGxzIG5lZWQ6IHRoZSBiZXN0IGxheWVyIHBlciBsYW5ndWFnZSwgcGx1cyB0aGUgZGV0b3VyIGxheWVyIGZvciBTcGFuaXNoIChQYXJ0IEQpCiAgICBrZWVwID0geyhsYW5nLCBCRVNUX0xBWUVSW2xhbmddKSBmb3IgbGFuZyBpbiBDX1JFU30KICAgIGlmIEEgYW5kICJlcyIgaW4gQ19SRVM6IGtlZXAuYWRkKCgiZXMiLCBtaW4oQ19MQVlFUlMsIGtleT1sYW1iZGEgbDogYWJzKGwgLSBBWyJkZXRvdXJfcGVha19sYXllciJdKSkpKQogICAgZm9yIChsYW5nLCBsKSBpbiBzb3J0ZWQoa2VlcCk6CiAgICAgICAgcmVzLCBmaXQsIHJob19oLCBzaGFyZWQsIEd4eSwgR3l4ID0gYW5hbHlzZV9sYXllcihsYW5nLCBsKQogICAgICAgIENfRklUU1sobGFuZywgbCldID0gZGljdChmaXQ9e2s6IGZpdFtrXSBmb3IgayBpbiBbIld4IiwgIlUiLCAiQSIsICJDeHhfciIsICJteCJdfSwgcmhvX2g9cmhvX2gsIHNoYXJlZD1zaGFyZWQsIEd4eT1HeHksIEd5eD1HeXgpCiAgICAgICAgZGVsIGZpdDsgZ2MuY29sbGVjdCgpCiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKTogdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCgpkZWYgc3Bhbl9zY29yZXMobGFuZywgbCwgc2VudGVuY2VzKToKICAgICIiIlUgKHdoaXRlbmVkKSwgVV9yYXcsIGhhbGYtbGlmZSBhbmQgI3Rva2VucyBmb3IgZXZlcnkgd29yZCBvZiB0aGUgZ2l2ZW4gdGVzdCBzZW50ZW5jZXMuIiIiCiAgICBGXyA9IENfRklUU1sobGFuZywgbCldOyBmaXQgPSBGX1siZml0Il07IHJobzIgPSBGX1sicmhvX2giXS5jbGFtcCgwLCAxKS5kb3VibGUoKSAqKiAyCiAgICB0ZXh0cyA9IFtTWyJ0ZXh0Il0gZm9yIFMgaW4gc2VudGVuY2VzXQogICAgd2wgPSBbWyh3LCBhLCBiKSBmb3IgdywgYSwgYiBpbiB3b3Jkc193aXRoX3NwYW5zKHQpXSBmb3IgdCBpbiB0ZXh0c10KICAgIHN0ID0gd29yZF9zdGF0ZXModGV4dHMsIHdsLCBbbF0pCiAgICBucyA9IG5wLmxpbnNwYWNlKDAsIDUwLCA1MDEpOyBvdXQgPSB7fQogICAgZm9yIFMsIHdvcmRzLCAocmVjLCB2YWxpZCkgaW4gemlwKHNlbnRlbmNlcywgd2wsIHN0KToKICAgICAgICBYID0gcmVjW2xdLmRvdWJsZSgpLnRvKENERVYpIC0gZml0WyJteCJdCiAgICAgICAgYzIgPSAoKFggQCBmaXRbIld4Il0pIEAgZml0WyJVIl0pICoqIDIgICAgICAgICAgICAgICAjIGNhbm9uaWNhbCBjb29yZGluYXRlcyBzcXVhcmVkCiAgICAgICAgUm4gPSB0b3JjaC5zdGFjayhbKGMyICogcmhvMiAqKiBuKS5zdW0oMSkgLyBjMi5zdW0oMSkgZm9yIG4gaW4gbnNdLCAxKS5jcHUoKS5udW1weSgpCiAgICAgICAgWHJ0ID0gKFggQCBGX1siR3h5Il0pIEAgRl9bIkd5eCJdCiAgICAgICAgVV9yYXcgPSAoKChYIC0gWHJ0KSAqKiAyKS5zdW0oMSkgLyAoWCAqKiAyKS5zdW0oMSkpLmNwdSgpLm51bXB5KCkKICAgICAgICBoYWxmID0gW2Zsb2F0KG5zW25wLmFyZ21heChyIDw9IDAuNSldKSBpZiAociA8PSAwLjUpLmFueSgpIGVsc2UgNTAuMCBmb3IgciBpbiBSbl0KICAgICAgICBvdXRbU1siaWQiXV0gPSBkaWN0KHdvcmRzPVt3IGZvciB3LCBfLCBfIGluIHdvcmRzXSwgVT0oMSAtIFJuWzosIDEwXSkudG9saXN0KCksIFVfcmF3PVVfcmF3LnRvbGlzdCgpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgaGFsZl9saWZlPWhhbGYsIFJfY3VydmU9Um5bOiwgOjoxMF1bOiwgOjExXS50b2xpc3QoKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIG50b2s9W2xlbihlbmModylbImlucHV0X2lkcyJdKSBmb3IgdywgXywgXyBpbiB3b3Jkc10sIHZhbGlkPXZhbGlkLnRvbGlzdCgpKQogICAgcmV0dXJuIG91dAoKZGVmIGhvdF93b3JkX2luZGV4KFMpOgogICAgaWR4ID0gc2V0KCkKICAgIGZvciBzcGFuIGluIFNbImhvdCJdOgogICAgICAgIHNwID0gZmluZF9waHJhc2UoU1sidGV4dCJdLCBzcGFuKQogICAgICAgIGlmIHNwOiBpZHggfD0ge2sgZm9yIGssIChfLCBhLCBiKSBpbiBlbnVtZXJhdGUod29yZHNfd2l0aF9zcGFucyhTWyJ0ZXh0Il0pKSBpZiBiID4gc3BbMF0gYW5kIGEgPCBzcFsxXX0KICAgIHJldHVybiBpZHgKCkNfU1BBTlMgPSB7fQppZiBSVU5fQyBhbmQgQ19SRVM6CiAgICBmb3IgbGFuZyBpbiBDX1JFUzoKICAgICAgICB0ZXN0ID0gW1MgZm9yIFMgaW4gU0VOVEVOQ0VTIGlmIFNbImxhbmciXSA9PSBsYW5nXQogICAgICAgIGlmIG5vdCB0ZXN0OiBjb250aW51ZQogICAgICAgIHJvd3MgPSB7fQogICAgICAgIGZvciAobGcsIGwpIGluIENfRklUUzoKICAgICAgICAgICAgaWYgbGcgIT0gbGFuZzogY29udGludWUKICAgICAgICAgICAgc2MgPSBzcGFuX3Njb3JlcyhsYW5nLCBsLCB0ZXN0KQogICAgICAgICAgICBob3QsIG90aCA9IGRlZmF1bHRkaWN0KGxpc3QpLCBkZWZhdWx0ZGljdChsaXN0KQogICAgICAgICAgICBmb3IgUyBpbiB0ZXN0OgogICAgICAgICAgICAgICAgciA9IHNjW1NbImlkIl1dOyBoaSA9IGhvdF93b3JkX2luZGV4KFMpCiAgICAgICAgICAgICAgICBmb3IgaywgdyBpbiBlbnVtZXJhdGUoclsid29yZHMiXSk6CiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHJbInZhbGlkIl1ba106IGNvbnRpbnVlCiAgICAgICAgICAgICAgICAgICAgdGd0ID0gaG90IGlmIGsgaW4gaGkgZWxzZSAob3RoIGlmIGxlbih3KSA+PSA0IGVsc2UgTm9uZSkKICAgICAgICAgICAgICAgICAgICBpZiB0Z3QgaXMgTm9uZTogY29udGludWUKICAgICAgICAgICAgICAgICAgICBmb3IgbSBpbiBbIlUiLCAiVV9yYXciLCAiaGFsZl9saWZlIiwgIm50b2siXTogdGd0W21dLmFwcGVuZChyW21dW2tdKQogICAgICAgICAgICBhdSA9IHttOiBhdXJvYyhob3RbbV0sIG90aFttXSkgZm9yIG0gaW4gWyJVIiwgIlVfcmF3IiwgImhhbGZfbGlmZSIsICJudG9rIl19CiAgICAgICAgICAgICMgYWdyZWVtZW50IHdpdGggUGFydCBCIChzZW50ZW5jZS1sZXZlbCwgY29udGVudCB3b3JkcyB3ZWlnaHRlZCBieSByb3VuZC10cmlwIGluZm9ybWF0aW9uIHdlaWdodCkKICAgICAgICAgICAgYWdyZWUgPSBOb25lCiAgICAgICAgICAgIGlmIEI6CiAgICAgICAgICAgICAgICB1aSwgdWIgPSBbXSwgW10KICAgICAgICAgICAgICAgIGZvciBTIGluIHRlc3Q6CiAgICAgICAgICAgICAgICAgICAgciA9IHNjW1NbImlkIl1dOyBXID0gbnAuYXJyYXkoQltTWyJpZCJdXVsid29yZF93ZWlnaHQiXSk7IHUgPSBucC5hcnJheShyWyJVIl0pOyB2ID0gbnAuYXJyYXkoclsidmFsaWQiXSkKICAgICAgICAgICAgICAgICAgICBpZiAoV1t2XSA+IDApLmFueSgpOiB1aS5hcHBlbmQoZmxvYXQoKFdbdl0gKiB1W3ZdKS5zdW0oKSAvIFdbdl0uc3VtKCkpKTsgdWIuYXBwZW5kKHN1bW1hcnlbU1siaWQiXV1bIkEiXSkKICAgICAgICAgICAgICAgIGFncmVlID0gc3BlYXJtYW4odWksIHViKQogICAgICAgICAgICByb3dzW2xdID0gZGljdChzY29yZXM9c2MsIGF1cm9jPWF1LCBzcGVhcm1hbl93aXRoX3JvdW5kdHJpcF9BPWFncmVlLCBuX2hvdD1sZW4oaG90WyJVIl0pLCBuX290aGVyPWxlbihvdGhbIlUiXSkpCiAgICAgICAgICAgIHByaW50KGYiW3tsYW5nfV0gbGF5ZXIge2x9OiBBVVJPQyBob3QgdnMgb3RoZXIgY29udGVudCB3b3JkcyDigJQgVSB7YXVbJ1UnXTouMmZ9LCBVX3JhdyB7YXVbJ1VfcmF3J106LjJmfSwgIgogICAgICAgICAgICAgICAgICBmImhhbGYtbGlmZSB7MS1hdVsnaGFsZl9saWZlJ106LjJmfSAoc2hvcnRlciA9IGhhcmRlciksICN0b2tlbnMgYmFzZWxpbmUge2F1WydudG9rJ106LjJmfTsgIgogICAgICAgICAgICAgICAgICBmImFncmVlbWVudCB3aXRoIHJvdW5kIHRyaXAgz4Ffcz17YWdyZWUgaWYgYWdyZWUgaXMgTm9uZSBlbHNlIHJvdW5kKGFncmVlLCAyKX0iKQogICAgICAgICAgICBpZiBsYW5nID09ICJlcyIgYW5kICJkdWVuZGUiIGluIHNjOgogICAgICAgICAgICAgICAgciA9IHNjWyJkdWVuZGUiXTsgZCA9IGRpY3QoemlwKHJbIndvcmRzIl0sIHppcChyWyJVIl0sIHJbImhhbGZfbGlmZSJdKSkpCiAgICAgICAgICAgICAgICBwcmludCgiICAgTWUgZ3VzdGEgbXVjaG8gZWwgZHVlbmRlOiAiICsgIiwgIi5qb2luKGYie3d9IFU9e2Rbd11bMF06LjNmfSAoaGFsZi1saWZlIHtkW3ddWzFdOi4xZn0pIiBmb3IgdyBpbiBbImR1ZW5kZSIsICJndXN0YSIsICJtdWNobyJdIGlmIHcgaW4gZCkpCiAgICAgICAgQ19TUEFOU1tsYW5nXSA9IHJvd3MKICAgICAgICBzYXZlX2pzb24oZiJDX3NwYW5zX3tsYW5nfS5qc29uIiwgcm93cykKCmlmIFJVTl9DIGFuZCAiZXMiIGluIENfUkVTIGFuZCBDX1NQQU5TLmdldCgiZXMiKToKICAgIGhlbGQgPSB0b3JjaC5sb2FkKG9zLnBhdGguam9pbihDQUNIRSwgIkNfZXNfaGVsZC5wdCIpLCB3ZWlnaHRzX29ubHk9RmFsc2UpOyBtZXRhID0gaGVsZFsibWV0YSJdCiAgICBsID0gQkVTVF9MQVlFUlsiZXMiXTsgRl8gPSBDX0ZJVFNbKCJlcyIsIGwpXTsgZml0ID0gRl9bImZpdCJdCiAgICBpZiBhbnkobVsibW9ycGgiXSBmb3IgbSBpbiBtZXRhKToKICAgICAgICBmcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBSaWRnZUNsYXNzaWZpZXIKICAgICAgICBmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCBjcm9zc192YWxfc2NvcmUKICAgICAgICBaID0gKCgoaGVsZFsiWCJdW2xdLnRvKENERVYpLmRvdWJsZSgpIC0gZml0WyJteCJdKSBAIGZpdFsiV3giXSkgQCBmaXRbIlUiXSkuZmxvYXQoKS5jcHUoKS5udW1weSgpCiAgICAgICAgWiAvPSBaLnN0ZCgwLCBrZWVwZGltcz1UcnVlKSArIDFlLTgKICAgICAgICBzaCA9IG5wLnplcm9zKFouc2hhcGVbMV0sIGJvb2wpOyBzaFtGX1sic2hhcmVkIl0uY3B1KCkubnVtcHkoKV0gPSBUcnVlCiAgICAgICAgZGVmIGZlYXQobSwga2V5KTogcmV0dXJuIGRpY3Qoa3Yuc3BsaXQoIj0iKSBmb3Iga3YgaW4gbVsibW9ycGgiXS5zcGxpdCgifCIpIGlmICI9IiBpbiBrdikuZ2V0KGtleSkKICAgICAgICB0YXNrcyA9IHsKICAgICAgICAgICAgImdyYW1tYXRpY2FsIGdlbmRlciI6IFsoaSwgZmVhdChtLCAiR2VuZGVyIikpIGZvciBpLCBtIGluIGVudW1lcmF0ZShtZXRhKSBpZiBtWyJwb3MiXSBpbiAoIk5PVU4iLCAiQURKIikgYW5kIGZlYXQobSwgIkdlbmRlciIpIGluICgiTWFzYyIsICJGZW0iKV0sCiAgICAgICAgICAgICJudW1iZXIiOiBbKGksIGZlYXQobSwgIk51bWJlciIpKSBmb3IgaSwgbSBpbiBlbnVtZXJhdGUobWV0YSkgaWYgbVsicG9zIl0gaW4gKCJOT1VOIiwgIkFESiIsICJWRVJCIikgYW5kIGZlYXQobSwgIk51bWJlciIpIGluICgiU2luZyIsICJQbHVyIildLAogICAgICAgICAgICAic3VianVuY3RpdmUgdnMgaW5kaWNhdGl2ZSI6IFsoaSwgZmVhdChtLCAiTW9vZCIpKSBmb3IgaSwgbSBpbiBlbnVtZXJhdGUobWV0YSkgaWYgbVsicG9zIl0gaW4gKCJWRVJCIiwgIkFVWCIpIGFuZCBmZWF0KG0sICJNb29kIikgaW4gKCJTdWIiLCAiSW5kIildLAogICAgICAgICAgICAiaW1wZXJmZWN0IHZzIHByZXRlcml0ZSI6IFsoaSwgZmVhdChtLCAiVGVuc2UiKSkgZm9yIGksIG0gaW4gZW51bWVyYXRlKG1ldGEpIGlmIG1bInBvcyJdIGluICgiVkVSQiIsICJBVVgiKSBhbmQgZmVhdChtLCAiTW9vZCIpID09ICJJbmQiIGFuZCBmZWF0KG0sICJUZW5zZSIpIGluICgiSW1wIiwgIlBhc3QiKV0sCiAgICAgICAgICAgICJub3VuIHZzIHZlcmIgKHNhbml0eSkiOiBbKGksIG1bInBvcyJdKSBmb3IgaSwgbSBpbiBlbnVtZXJhdGUobWV0YSkgaWYgbVsicG9zIl0gaW4gKCJOT1VOIiwgIlZFUkIiKV0sCiAgICAgICAgfQogICAgICAgIHByb2JlID0ge30KICAgICAgICBmb3IgbmFtZSwgaXRlbXMgaW4gdGFza3MuaXRlbXMoKToKICAgICAgICAgICAgbGFicyA9IFtiIGZvciBfLCBiIGluIGl0ZW1zXQogICAgICAgICAgICBpZiBsZW4oaXRlbXMpIDwgNjAgb3IgbWluKGxhYnMuY291bnQodikgZm9yIHYgaW4gc2V0KGxhYnMpKSA8IDIwOiBwcmludChmIiAge25hbWV9OiB0b28gZmV3IGV4YW1wbGVzIik7IGNvbnRpbnVlCiAgICAgICAgICAgIGlpID0gbnAuYXJyYXkoW2kgZm9yIGksIF8gaW4gaXRlbXNdKTsgeSA9IG5wLmFycmF5KFtiID09IHNvcnRlZChzZXQobGFicykpWzBdIGZvciBiIGluIGxhYnNdKQogICAgICAgICAgICBjbGYgPSBSaWRnZUNsYXNzaWZpZXIoYWxwaGE9ZmxvYXQobGVuKGlpKSksIGNsYXNzX3dlaWdodD0iYmFsYW5jZWQiKQogICAgICAgICAgICBhY2MgPSB7azogZmxvYXQoY3Jvc3NfdmFsX3Njb3JlKGNsZiwgWltpaV1bOiwgY29sc10sIHksIGN2PTUsIHNjb3Jpbmc9ImJhbGFuY2VkX2FjY3VyYWN5IikubWVhbigpKQogICAgICAgICAgICAgICAgICAgZm9yIGssIGNvbHMgaW4gWygiYWxsIiwgbnAub25lc19saWtlKHNoKSksICgic2hhcmVkIG9ubHkiLCBzaCksICgicHJpdmF0ZSBvbmx5IiwgfnNoKV0gaWYgY29scy5hbnkoKX0KICAgICAgICAgICAgY2xmLmZpdChaW2lpXSwgeSk7IHdndCA9IG5wLnJhdmVsKGNsZi5jb2VmXykgKiogMiAgICAgICMgY29lZl8gaXMgKDEsIGQpIG9yIChkLCkgZGVwZW5kaW5nIG9uIHRoZSBza2xlYXJuIHZlcnNpb24KICAgICAgICAgICAgcHJvYmVbbmFtZV0gPSBkaWN0KG49bGVuKGlpKSwgc2hhcmVkX3NoYXJlPWZsb2F0KHdndFtzaF0uc3VtKCkgLyB3Z3Quc3VtKCkpLCBleHBlY3RlZD1mbG9hdChzaC5tZWFuKCkpLCBhY2M9YWNjKQogICAgICAgICAgICBwcmludChmIiAge25hbWU6MjhzfSBuPXtsZW4oaWkpOjVkfSAgd2VpZ2h0IG9uIHNoYXJlZCBkaXJlY3Rpb25zIHtwcm9iZVtuYW1lXVsnc2hhcmVkX3NoYXJlJ106LjJmfSAiCiAgICAgICAgICAgICAgICAgIGYiKGNoYW5jZSB7c2gubWVhbigpOi4yZn0pICBhY2N1cmFjeSBhbGwvc2hhcmVkL3ByaXZhdGUgPSAiICsgIi8iLmpvaW4oZiJ7djouMmZ9IiBmb3IgdiBpbiBhY2MudmFsdWVzKCkpKQogICAgICAgIHNhdmVfanNvbigiQ19wcm9iZXNfZXMuanNvbiIsIGRpY3QobGF5ZXI9bCwgcHJvYmVzPXByb2JlKSkKCmlmIFJVTl9DIGFuZCBDX1JFUzoKICAgIGZpZywgYXhlcyA9IHBsdC5zdWJwbG90cygxLCAyLCBmaWdzaXplPSgxMSwgMy42KSkKICAgIGF4ID0gYXhlc1swXQogICAgZm9yIGxhbmcsIGNvbCBpbiB6aXAoQ19SRVMsIFtIQVJELCBMT0FOLCBFQVNZXSk6CiAgICAgICAgciA9IG5leHQoeCBmb3IgeCBpbiBDX1JFU1tsYW5nXSBpZiB4WyJsYXllciJdID09IEJFU1RfTEFZRVJbbGFuZ10pOyBrID0gbnAuYXJhbmdlKDEsIGxlbihyWyJyaG9faW4iXSkgKyAxKQogICAgICAgIGF4LnBsb3QoaywgclsicmhvX2luIl0sIGNvbG9yPWNvbCwgbHc9MSwgYWxwaGE9MC41LCBscz0iLS0iKQogICAgICAgIGF4LnBsb3QoaywgclsicmhvX2hlbGQiXSwgY29sb3I9Y29sLCBsdz0xLjYsIGxhYmVsPWYie2xhbmd94oCTZW4sIGxheWVyIHtyWydsYXllciddfSAoaGVsZC1vdXQpIikKICAgICAgICBheC5heGhsaW5lKHJbIm51bGxfdGhyIl0sIGNvbG9yPWNvbCwgbHc9MC44LCBscz0iOiIpCiAgICBheC5zZXRfeHNjYWxlKCJsb2ciKTsgYXguc2V0X3hsYWJlbCgiY2Fub25pY2FsIGRpcmVjdGlvbiBrIik7IGF4LnNldF95bGFiZWwoIs+BX2siKQogICAgYXguc2V0X3RpdGxlKCJSb3VuZC10cmlwIHNwZWN0cnVtIChkYXNoZWQ6IGluLXNhbXBsZTsgZG90dGVkOiBzaHVmZmxlZC1wYWlyIG51bGwpIiwgbG9jPSJsZWZ0IiwgZm9udHNpemU9OSk7IGF4LmxlZ2VuZChmcmFtZW9uPUZhbHNlLCBmb250c2l6ZT04KQogICAgYXggPSBheGVzWzFdCiAgICBmb3IgbGFuZywgY29sIGluIHppcChDX1JFUywgW0hBUkQsIExPQU4sIEVBU1ldKToKICAgICAgICBheC5wbG90KFtyWyJsYXllciJdIC8gTl9MQVlFUlMgZm9yIHIgaW4gQ19SRVNbbGFuZ11dLCBbclsidG9wNTBfaGVsZCJdIC8gNTAgZm9yIHIgaW4gQ19SRVNbbGFuZ11dLCAiby0iLCBjb2xvcj1jb2wsIGxhYmVsPWYie2xhbmd94oCTZW4iKQogICAgaWYgQTogYXguYXh2bGluZShBWyJkZXRvdXJfcGVha19sYXllciJdIC8gTl9MQVlFUlMsIGNvbG9yPSIjODg4IiwgbHM9Ii0tIiwgbHc9MSk7IGF4LnRleHQoQVsiZGV0b3VyX3BlYWtfbGF5ZXIiXSAvIE5fTEFZRVJTLCBheC5nZXRfeWxpbSgpWzBdLCAiIEVuZ2xpc2ggZGV0b3VyIHBlYWsiLCBmb250c2l6ZT04LCBjb2xvcj0iIzg4OCIpCiAgICBheC5zZXRfeGxhYmVsKCJkZXB0aCIpOyBheC5zZXRfeWxhYmVsKCJtZWFuIGhlbGQtb3V0IM+BLCB0b3AgNTAgZGlyZWN0aW9ucyIpOyBheC5sZWdlbmQoZnJhbWVvbj1GYWxzZSkKICAgIGF4LnNldF90aXRsZSgiV2hlcmUgdGhlIHR3byBsYW5ndWFnZXMgc2hhcmUgbW9zdCIsIGxvYz0ibGVmdCIsIGZvbnRzaXplPTkpCiAgICBzYXZlZmlnKGZpZywgIkMxX3NwZWN0cnVtIikKCiAgICBpZiBDX1NQQU5TLmdldCgiZXMiKToKICAgICAgICBzYyA9IENfU1BBTlNbImVzIl1bQkVTVF9MQVlFUlsiZXMiXV1bInNjb3JlcyJdCiAgICAgICAgcGljayA9IFsoImR1ZW5kZSIsICJkdWVuZGUiKSwgKCJkdWVuZGUiLCAiZ3VzdGEiKSwgKCJzb2JyZW1lc2EiLCAic29icmVtZXNhIiksICgibWFkcnVnYWRhIiwgIm1hZHJ1Z2FkYSIpLAogICAgICAgICAgICAgICAgKCJ0cmVuIiwgInRyZW4iKSwgKCJoZXJtYW5vIiwgImhlcm1hbm8iKSwgKCJnYXRvcyIsICJnYXRvcyIpXQogICAgICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oNywgMy42KSkKICAgICAgICBmb3Igc2lkLCB3IGluIHBpY2s6CiAgICAgICAgICAgIGlmIHNpZCBub3QgaW4gc2Mgb3IgdyBub3QgaW4gc2Nbc2lkXVsid29yZHMiXTogY29udGludWUKICAgICAgICAgICAgayA9IHNjW3NpZF1bIndvcmRzIl0uaW5kZXgodyk7IGNvbCA9IEVBU1kgaWYgU0lEW3NpZF1bImNhdCJdID09ICJjb250cm9sIiBlbHNlIEhBUkQKICAgICAgICAgICAgYXgucGxvdChyYW5nZSgxMSksIHNjW3NpZF1bIlJfY3VydmUiXVtrXSwgY29sb3I9Y29sLCBsdz0yLCBhbHBoYT0wLjg1KQogICAgICAgICAgICBheC50ZXh0KDEwLjIsIHNjW3NpZF1bIlJfY3VydmUiXVtrXVstMV0sIGYie3d9IChoYWxmLWxpZmUge3NjW3NpZF1bJ2hhbGZfbGlmZSddW2tdOi4xZn0pIiwgZm9udHNpemU9OCwgc3R5bGU9Iml0YWxpYyIsIGNvbG9yPWNvbCwgdmE9ImNlbnRlciIpCiAgICAgICAgYXguYXhobGluZSgwLjUsIGNvbG9yPSIjODg4IiwgbHM9IjoiLCBsdz0xKQogICAgICAgIGF4LnNldF94bGFiZWwoIm51bWJlciBvZiByb3VuZCB0cmlwcyBTcGFuaXNoIOKGkiBFbmdsaXNoIOKGkiBTcGFuaXNoIChpbnNpZGUgdGhlIG1vZGVsKSIpOyBheC5zZXRfeWxhYmVsKCJzaGFyZSBvZiB0aGUgd29yZCB0aGF0IHN1cnZpdmVzIikKICAgICAgICBheC5zZXRfdGl0bGUoIldoYXQgc3Vydml2ZXMgcmVwZWF0ZWQgcm91bmQgdHJpcHMiLCBsb2M9ImxlZnQiKTsgYXguc2V0X3hsaW0oMCwgMTMuNSkKICAgICAgICBzYXZlZmlnKGZpZywgIkMyX3JldGVudGlvbiIpCgogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg1LjUsIDMuNCkpOyB4cyA9IG5wLmFyYW5nZShsZW4oQ19SRVMpKTsgdyA9IDAuMzUKICAgIHJfID0gW25leHQoeCBmb3IgeCBpbiBDX1JFU1tsYW5nXSBpZiB4WyJsYXllciJdID09IEJFU1RfTEFZRVJbbGFuZ10pIGZvciBsYW5nIGluIENfUkVTXQogICAgYXguYmFyKHhzIC0gdyAvIDIsIFtyWyJFX2VuX2Zyb21feCJdIGZvciByIGluIHJfXSwgdywgY29sb3I9R1JFWSwgbGFiZWw9IkVuZ2xpc2gsIHByZWRpY3RlZCBmcm9tIHRoZSBvdGhlciBsYW5ndWFnZSIpCiAgICBheC5iYXIoeHMgKyB3IC8gMiwgW3JbIkVfeF9mcm9tX2VuIl0gZm9yIHIgaW4gcl9dLCB3LCBjb2xvcj1IQVJELCBsYWJlbD0ib3RoZXIgbGFuZ3VhZ2UsIHByZWRpY3RlZCBmcm9tIEVuZ2xpc2giKQogICAgYXguc2V0X3h0aWNrcyh4cyk7IGF4LnNldF94dGlja2xhYmVscyhbZiJ7TEFOR19OQU1FW2xdfeKAk0VuZ2xpc2giIGZvciBsIGluIENfUkVTXSk7IGF4LnNldF95bGFiZWwoInNoYXJlIG9mIHZhcmlhdGlvbiBwcmVkaWN0YWJsZSIpCiAgICBheC5zZXRfdGl0bGUoIkhvdyBtdWNoIG9mIGVhY2ggbGFuZ3VhZ2Ugc3Vydml2ZXMgdGhlIGNyb3NzaW5nIiwgbG9jPSJsZWZ0Iik7IGF4LmxlZ2VuZChmcmFtZW9uPUZhbHNlLCBmb250c2l6ZT04KQogICAgc2F2ZWZpZyhmaWcsICJDM19lbmdsaXNoX2JpYXMiKQoKcGFjaygpCg=='))
open('/content/ut/D_mediation.py','wb').write(base64.b64decode('IyBSZWxvYWQgZWFybGllciBzdGFnZXMnIHJlc3VsdHMsIHNvIHRoaXMgc3RhZ2UgYWxzbyB3b3JrcyBpbiBhIGZyZXNoIGtlcm5lbCAoYWZ0ZXIgMDBfc2V0dXAucHkpLgpBID0gbG9hZF9qc29uKCJBX2RldG91ci5qc29uIikgaWYgaGF2ZSgiQV9kZXRvdXIuanNvbiIpIGVsc2UgTm9uZQpsYWRkZXIgPSBsb2FkX2pzb24oIkJfbGFkZGVyLmpzb24iKSBpZiBoYXZlKCJCX2xhZGRlci5qc29uIikgZWxzZSBOb25lCkIgPSBsb2FkX2pzb24oIkJfcm91bmR0cmlwLmpzb24iKSBpZiBoYXZlKCJCX3JvdW5kdHJpcC5qc29uIikgZWxzZSBOb25lCnN1bW1hcnkgPSBsb2FkX2pzb24oIkJfc3VtbWFyeS5qc29uIilbInBlcl9zZW50ZW5jZSJdIGlmIGhhdmUoIkJfc3VtbWFyeS5qc29uIikgZWxzZSB7fQoKCmFzc2VydCBDX0ZJVFMsICdydW4gQzJfc3BlY3RydW0ucHkgaW4gdGhpcyBzYW1lIGtlcm5lbCBmaXJzdCAoaXQga2VlcHMgdGhlIGZpdHRlZCBiYXNlcyBpbiBHUFUgbWVtb3J5KScKCmRlZiBkZWNvbXBvc2VyKGxhbmcsIGwpOgogICAgRl8gPSBDX0ZJVFNbKGxhbmcsIGwpXTsgZml0ID0gRl9bImZpdCJdCiAgICBpZHggPSBGX1sic2hhcmVkIl0KICAgIEFfciA9IGZpdFsiQSJdWzosIGlkeF0uZmxvYXQoKTsgRF9yID0gKGZpdFsiQ3h4X3IiXSBAIGZpdFsiQSJdWzosIGlkeF0pLmZsb2F0KCk7IG14ID0gZml0WyJteCJdLmZsb2F0KCkKICAgIGRlZiBzcGxpdChoKToKICAgICAgICB4YyA9IGguZmxvYXQoKS50byhBX3IuZGV2aWNlKSAtIG14OyBzaCA9ICh4YyBAIEFfcikgQCBEX3IuVAogICAgICAgIHJldHVybiBzaCwgeGMgLSBzaAogICAgdGVzdCA9IHRvcmNoLnJhbmRuKDQsIERfTU9ERUwsIGRldmljZT1BX3IuZGV2aWNlKSAqIDMgKyBteAogICAgc2gsIF8gPSBzcGxpdCh0ZXN0KTsgc2gyLCBfID0gc3BsaXQoc2ggKyBteCkKICAgIHByaW50KGYiICBsYXllciB7bH06IHNoYXJlZCBkaW1zIHtsZW4oaWR4KX0gb2Yge0RfTU9ERUx9OyBvYmxpcXVlIHByb2plY3Rpb24gaWRlbXBvdGVuY2UgZXJyb3Ige2Zsb2F0KChzaDIgLSBzaCkubm9ybSgpIC8gc2gubm9ybSgpKTouMWV9IikKICAgIHJldHVybiBzcGxpdCwgbXgKCmRlZiBhYmxhdGVfZm4oc3BsaXQsIG14LCB3aGljaCk6CiAgICBkZWYgZm4oaCk6CiAgICAgICAgc2gsIHByID0gc3BsaXQoaCk7IGRldiA9IGguZGV2aWNlCiAgICAgICAgaWYgd2hpY2ggPT0gInByaXZhdGUiOiByZXR1cm4gKG14ICsgc2gpLnRvKGRldikKICAgICAgICBpZiB3aGljaCA9PSAic2hhcmVkIjogcmV0dXJuIChteCArIHByKS50byhkZXYpCiAgICAgICAgcmV0dXJuIG14LmV4cGFuZF9hcyhzaCkudG8oZGV2KQogICAgcmV0dXJuIGZuCgpkZWYgaW50ZXJjaGFuZ2VfZm4oc3BsaXQsIG14LCBkb25vciwgd2hpY2gpOgogICAgZGVmIGZuKGgpOgogICAgICAgIHNoLCBwciA9IHNwbGl0KGgpOyBkc2gsIGRwciA9IHNwbGl0KGRvbm9yKTsgeGMgPSBzaCArIHByCiAgICAgICAgaWYgd2hpY2ggPT0gInByaXZhdGUiOiBuZXcgPSB4YyAtIHByICsgZHByCiAgICAgICAgZWxpZiB3aGljaCA9PSAic2hhcmVkIjogbmV3ID0geGMgLSBzaCArIGRzaAogICAgICAgIGVsc2U6IG5ldyA9IGRzaCArIGRwcgogICAgICAgIHJldHVybiAobmV3ICsgbXgpLnRvKGguZGV2aWNlKQogICAgcmV0dXJuIGZuCgppZiBSVU5fRCBhbmQgImVzIiBpbiBDX1JFUyBhbmQgbGFkZGVyIGFuZCBub3QgaGF2ZSgiRF9tZWRpYXRpb24uanNvbiIpOgogICAgRF9vdXQgPSB7fQogICAgbWVkX2xheWVycyA9IHNvcnRlZCh7bCBmb3IgKGxnLCBsKSBpbiBDX0ZJVFMgaWYgbGcgPT0gImVzIn0pCiAgICBmb3IgbCBpbiBtZWRfbGF5ZXJzOgogICAgICAgIHdpdGggVGltZXIoZiJQYXJ0IEQ6IGxheWVyIHtsfSIpOgogICAgICAgICAgICBzcGxpdCwgbXggPSBkZWNvbXBvc2VyKCJlcyIsIGwpCiAgICAgICAgICAgICMgLS0tIDEuIGFibGF0aW9uIGRpc3NvY2lhdGlvbiAtLS0KICAgICAgICAgICAgZXNfc2VudHMgPSBbUyBmb3IgUyBpbiBTRU5URU5DRVMgaWYgU1sibGFuZyJdID09ICJlcyJdCiAgICAgICAgICAgIHRhcmdldHMgPSBbXQogICAgICAgICAgICBmb3IgUyBpbiBlc19zZW50czoKICAgICAgICAgICAgICAgIGhpID0gaG90X3dvcmRfaW5kZXgoUykKICAgICAgICAgICAgICAgIGZvciBrLCAodywgYSwgYikgaW4gZW51bWVyYXRlKHdvcmRzX3dpdGhfc3BhbnMoU1sidGV4dCJdKSk6CiAgICAgICAgICAgICAgICAgICAgaWYgayBpbiBoaSBvciAobGVuKHcpID49IDQgYW5kIFNbImNhdCJdID09ICJjb250cm9sIik6IHRhcmdldHMuYXBwZW5kKChTLCB3LCBrIGluIGhpLCAoYSwgYikpKQogICAgICAgICAgICBhc2sgPSBbY2hhdF9wcm9tcHQoZid7U1sidGV4dCJdfVxuXG5JbiB0aGUgU3BhbmlzaCBzZW50ZW5jZSBhYm92ZSwgd2hhdCBkb2VzIHRoZSB3b3JkICJ7d30iIG1lYW4gaGVyZT8gQW5zd2VyIGluIG9uZSBsaW5lLicpCiAgICAgICAgICAgICAgICAgICBmb3IgUywgdywgXywgXyBpbiB0YXJnZXRzXQogICAgICAgICAgICBleHBsID0gZ2VuZXJhdGUoYXNrLCBtYXhfbmV3X3Rva2Vucz00MCkKICAgICAgICAgICAgUCwgQ24sIEUsIG1ldGEgPSBbXSwgW10sIFtdLCBbXQogICAgICAgICAgICBmb3IgKFMsIHcsIGlzX2hvdCwgc3BhbiksIGFwLCBleCBpbiB6aXAodGFyZ2V0cywgYXNrLCBleHBsKToKICAgICAgICAgICAgICAgIHRwID0gY2hhdF9wcm9tcHQocnVuZ191c2VyKCJCMSIsIFNbInRleHQiXSwgImVzIikpCiAgICAgICAgICAgICAgICBmb3IgdGFzaywgcHJvbXB0LCBjb250IGluIFsoInRyYW5zbGF0ZSIsIHRwLCBsYWRkZXJbU1siaWQiXV1bIkIxIl0pLCAoImV4cGxhaW4iLCBhcCwgZXgpXToKICAgICAgICAgICAgICAgICAgICBwb3MgPSBwb3NpdGlvbnNfaW5fcHJvbXB0KHByb21wdCwgU1sidGV4dCJdLCBzcGFuPXNwYW4pCiAgICAgICAgICAgICAgICAgICAgaWYgdGFzayA9PSAiZXhwbGFpbiI6ICAgICAgICAgICAgICAgICAgICAgICMgYWxzbyB0aGUgd29yZCBxdW90ZWQgaW4gdGhlIHF1ZXN0aW9uCiAgICAgICAgICAgICAgICAgICAgICAgIHEgPSBwcm9tcHQucmZpbmQoZicie3d9IicpICsgMTsgcG9zID0gc29ydGVkKHNldChwb3MpIHwgc2V0KHBvc2l0aW9uc19hdChwcm9tcHQsIHEsIHEgKyBsZW4odykpKSkKICAgICAgICAgICAgICAgICAgICBmb3IgY29uZCBpbiBbIm5vbmUiLCAicHJpdmF0ZSIsICJzaGFyZWQiLCAiZnVsbCJdOgogICAgICAgICAgICAgICAgICAgICAgICBQLmFwcGVuZChwcm9tcHQpOyBDbi5hcHBlbmQoY29udCkKICAgICAgICAgICAgICAgICAgICAgICAgRS5hcHBlbmQoTm9uZSBpZiBjb25kID09ICJub25lIiBlbHNlIFsobCwgcG9zLCBhYmxhdGVfZm4oc3BsaXQsIG14LCBjb25kKSldKQogICAgICAgICAgICAgICAgICAgICAgICBtZXRhLmFwcGVuZCgoU1siaWQiXSwgdywgaXNfaG90LCB0YXNrLCBjb25kKSkKICAgICAgICAgICAgc2MgPSBzY29yZShQLCBDbiwgZWRpdHM9RSkKICAgICAgICAgICAgbHAgPSB7bTogZmxvYXQoclsibHAiXS5tZWFuKCkpIGZvciBtLCByIGluIHppcChtZXRhLCBzYyl9CiAgICAgICAgICAgIGFibCA9IFtdCiAgICAgICAgICAgIGZvciAoUywgdywgaXNfaG90LCBfKSwgZXggaW4gemlwKHRhcmdldHMsIGV4cGwpOgogICAgICAgICAgICAgICAgcm93ID0gZGljdChzaWQ9U1siaWQiXSwgd29yZD13LCBob3Q9aXNfaG90LCBleHBsYW5hdGlvbj1leCkKICAgICAgICAgICAgICAgIGZvciB0YXNrIGluIFsidHJhbnNsYXRlIiwgImV4cGxhaW4iXToKICAgICAgICAgICAgICAgICAgICBiYXNlID0gbHBbKFNbImlkIl0sIHcsIGlzX2hvdCwgdGFzaywgIm5vbmUiKV0KICAgICAgICAgICAgICAgICAgICBmb3IgY29uZCBpbiBbInByaXZhdGUiLCAic2hhcmVkIiwgImZ1bGwiXToKICAgICAgICAgICAgICAgICAgICAgICAgcm93W2Yie3Rhc2t9X3tjb25kfSJdID0gYmFzZSAtIGxwWyhTWyJpZCJdLCB3LCBpc19ob3QsIHRhc2ssIGNvbmQpXSAgICMgZHJvcCBpbiBsb2ctcHJvYiBwZXIgdG9rZW4KICAgICAgICAgICAgICAgIGFibC5hcHBlbmQocm93KQogICAgICAgICAgICAjIC0tLSAyLiBpbnRlcmNoYW5nZSBvbiBzZW5zZSBwYWlycyAtLS0KICAgICAgICAgICAgaW50ZXIgPSBbXQogICAgICAgICAgICBmb3IgU1AgaW4gU0VOU0VfUEFJUlM6CiAgICAgICAgICAgICAgICBydW5zID0ge30KICAgICAgICAgICAgICAgIGZvciBtZSwgb3RoZXIgaW4gWygiYSIsICJiIiksICgiYiIsICJhIildOgogICAgICAgICAgICAgICAgICAgIHBtID0gY2hhdF9wcm9tcHQocnVuZ191c2VyKCJCMSIsIFNQW21lXSwgImVzIikpOyBwbyA9IGNoYXRfcHJvbXB0KHJ1bmdfdXNlcigiQjEiLCBTUFtvdGhlcl0sICJlcyIpKQogICAgICAgICAgICAgICAgICAgIHBvc19tID0gcG9zaXRpb25zX2luX3Byb21wdChwbSwgU1BbbWVdLCBTUFsid29yZCJdKTsgcG9zX28gPSBwb3NpdGlvbnNfaW5fcHJvbXB0KHBvLCBTUFtvdGhlcl0sIFNQWyJ3b3JkIl0pCiAgICAgICAgICAgICAgICAgICAgaWYgbGVuKHBvc19tKSAhPSBsZW4ocG9zX28pOiBwcmludChmIiAgc2tpcCB7U1BbJ2lkJ119OiB0b2tlbiBjb3VudHMgZGlmZmVyIik7IHJ1bnMgPSBOb25lOyBicmVhawogICAgICAgICAgICAgICAgICAgIGRvbm9yID0gY2FwdHVyZV9wcm9tcHRfc3RhdGVzKHBvLCBsKVtwb3Nfb10KICAgICAgICAgICAgICAgICAgICB0bXBsID0gU1BbZiJlbl97bWV9Il07IGNhbmRzID0gW3RtcGwuZm9ybWF0KFNQWyJzZW5zZV9hIl0pLCB0bXBsLmZvcm1hdChTUFsic2Vuc2VfYiJdKV0KICAgICAgICAgICAgICAgICAgICBjb25kcyA9IHsibm9uZSI6IE5vbmUsICJwcml2YXRlIjogaW50ZXJjaGFuZ2VfZm4oc3BsaXQsIG14LCBkb25vciwgInByaXZhdGUiKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAic2hhcmVkIjogaW50ZXJjaGFuZ2VfZm4oc3BsaXQsIG14LCBkb25vciwgInNoYXJlZCIpLCAiZnVsbCI6IGludGVyY2hhbmdlX2ZuKHNwbGl0LCBteCwgZG9ub3IsICJmdWxsIil9CiAgICAgICAgICAgICAgICAgICAgUHMsIENzLCBFcywga2V5cyA9IFtdLCBbXSwgW10sIFtdCiAgICAgICAgICAgICAgICAgICAgZm9yIGNuYW1lLCBmbiBpbiBjb25kcy5pdGVtcygpOgogICAgICAgICAgICAgICAgICAgICAgICBmb3IgY2ksIGMgaW4gZW51bWVyYXRlKGNhbmRzKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIFBzLmFwcGVuZChwbSk7IENzLmFwcGVuZChjKTsgRXMuYXBwZW5kKE5vbmUgaWYgZm4gaXMgTm9uZSBlbHNlIFsobCwgcG9zX20sIGZuKV0pOyBrZXlzLmFwcGVuZCgoY25hbWUsIGNpKSkKICAgICAgICAgICAgICAgICAgICByID0gc2NvcmUoUHMsIENzLCBlZGl0cz1Fcyk7IHRvdCA9IHtrOiBmbG9hdCh4WyJscCJdLnN1bSgpKSBmb3IgaywgeCBpbiB6aXAoa2V5cywgcil9CiAgICAgICAgICAgICAgICAgICAgbG8gPSB7YzogdG90WyhjLCAwKV0gLSB0b3RbKGMsIDEpXSBmb3IgYyBpbiBjb25kc30gICAgICAgICAgIyBsb2ctb2RkcyBzZW5zZV9hIHZzIHNlbnNlX2IKICAgICAgICAgICAgICAgICAgICBnZW4gPSB7YzogZ2VuZXJhdGVfd2l0aF9lZGl0cyhwbSwgW10gaWYgZm4gaXMgTm9uZSBlbHNlIFsobCwgcG9zX20sIGZuKV0pIGZvciBjLCBmbiBpbiBjb25kcy5pdGVtcygpfQogICAgICAgICAgICAgICAgICAgIHJ1bnNbbWVdID0gZGljdChsb2dvZGRzPWxvLCBnZW5lcmF0aW9ucz1nZW4pCiAgICAgICAgICAgICAgICBpZiBydW5zIGlzIE5vbmU6IGNvbnRpbnVlCiAgICAgICAgICAgICAgICBiYXNlX2EsIGJhc2VfYiA9IHJ1bnNbImEiXVsibG9nb2RkcyJdWyJub25lIl0sIHJ1bnNbImIiXVsibG9nb2RkcyJdWyJub25lIl0KICAgICAgICAgICAgICAgIGdhcCA9IGJhc2VfYiAtIGJhc2VfYQogICAgICAgICAgICAgICAgZmxpcCA9IHt9CiAgICAgICAgICAgICAgICBmb3IgYyBpbiBbInByaXZhdGUiLCAic2hhcmVkIiwgImZ1bGwiXToKICAgICAgICAgICAgICAgICAgICBmYSA9IChydW5zWyJhIl1bImxvZ29kZHMiXVtjXSAtIGJhc2VfYSkgLyBnYXAgaWYgYWJzKGdhcCkgPiAxZS02IGVsc2UgZmxvYXQoIm5hbiIpCiAgICAgICAgICAgICAgICAgICAgZmIgPSAocnVuc1siYiJdWyJsb2dvZGRzIl1bY10gLSBiYXNlX2IpIC8gKC1nYXApIGlmIGFicyhnYXApID4gMWUtNiBlbHNlIGZsb2F0KCJuYW4iKQogICAgICAgICAgICAgICAgICAgIGZsaXBbY10gPSBmbG9hdChucC5uYW5tZWFuKFtmYSwgZmJdKSkKICAgICAgICAgICAgICAgIGludGVyLmFwcGVuZChkaWN0KGlkPVNQWyJpZCJdLCBiYXNlX2dhcD1nYXAsIGZsaXA9ZmxpcCwgcnVucz1ydW5zKSkKICAgICAgICAgICAgICAgIHByaW50KGYiICB7U1BbJ2lkJ106MTFzfSBiYXNlIGxvZy1vZGRzIGE9e2Jhc2VfYTorLjFmfSBiPXtiYXNlX2I6Ky4xZn0gIGZsaXAgcHJpdmF0ZT17ZmxpcFsncHJpdmF0ZSddOisuMmZ9ICIKICAgICAgICAgICAgICAgICAgICAgIGYic2hhcmVkPXtmbGlwWydzaGFyZWQnXTorLjJmfSBmdWxsPXtmbGlwWydmdWxsJ106Ky4yZn0iKQogICAgICAgICAgICBEX291dFtsXSA9IGRpY3QoYWJsYXRpb249YWJsLCBpbnRlcmNoYW5nZT1pbnRlcikKICAgIHNhdmVfanNvbigiRF9tZWRpYXRpb24uanNvbiIsIERfb3V0KQoKRCA9IGxvYWRfanNvbigiRF9tZWRpYXRpb24uanNvbiIpIGlmIGhhdmUoIkRfbWVkaWF0aW9uLmpzb24iKSBlbHNlIE5vbmUKaWYgRDoKICAgIGZvciBsLCBkIGluIEQuaXRlbXMoKToKICAgICAgICBhYiA9IGRbImFibGF0aW9uIl0KICAgICAgICBkZWYgcmF0aW8ociwgdGFzaywgY29uZCk6IGYgPSByW2Yie3Rhc2t9X2Z1bGwiXTsgcmV0dXJuIHJbZiJ7dGFza31fe2NvbmR9Il0gLyBmIGlmIGFicyhmKSA+IDFlLTYgZWxzZSBucC5uYW4KICAgICAgICBwcmludChmIlxubGF5ZXIge2x9OiBlZmZlY3Qgb2YgcmVtb3ZpbmcgdGhlIFBSSVZBVEUgcGFydCwgYXMgYSBzaGFyZSBvZiByZW1vdmluZyB0aGUgd2hvbGUgc3RhdGUgKG1lYW4sIDk1JSBDSSkiKQogICAgICAgIGZvciBsYWIsIGdycCBpbiBbKCJob3Qgd29yZHMiLCBbciBmb3IgciBpbiBhYiBpZiByWyJob3QiXV0pLCAoImNvbnRyb2wgd29yZHMiLCBbciBmb3IgciBpbiBhYiBpZiBub3QgclsiaG90Il1dKV06CiAgICAgICAgICAgIHQgPSBbcmF0aW8ociwgInRyYW5zbGF0ZSIsICJwcml2YXRlIikgZm9yIHIgaW4gZ3JwXTsgZSA9IFtyYXRpbyhyLCAiZXhwbGFpbiIsICJwcml2YXRlIikgZm9yIHIgaW4gZ3JwXQogICAgICAgICAgICBjdCwgY2UgPSBib290X2NpKHQpLCBib290X2NpKGUpCiAgICAgICAgICAgIHByaW50KGYiICB7bGFiOjE0c30gbj17bGVuKGdycCk6M2R9ICBvbiB0cmFuc2xhdGlvbiB7bnAubmFubWVhbih0KTouMmZ9IFt7Y3RbMF06LjJmfSwge2N0WzFdOi4yZn1dIChtZWRpYW4ge25wLm5hbm1lZGlhbih0KTouMmZ9KSIKICAgICAgICAgICAgICAgICAgZiIgICBvbiBleHBsYW5hdGlvbiB7bnAubmFubWVhbihlKTouMmZ9IFt7Y2VbMF06LjJmfSwge2NlWzFdOi4yZn1dIChtZWRpYW4ge25wLm5hbm1lZGlhbihlKTouMmZ9KSIpCiAgICAgICAgaG90ID0gW3IgZm9yIHIgaW4gYWIgaWYgclsiaG90Il1dCiAgICAgICAgZ2FwID0gW3JhdGlvKHIsICJleHBsYWluIiwgInByaXZhdGUiKSAtIHJhdGlvKHIsICJ0cmFuc2xhdGUiLCAicHJpdmF0ZSIpIGZvciByIGluIGhvdF0KICAgICAgICBjZyA9IGJvb3RfY2koZ2FwKQogICAgICAgIHByaW50KGYiICBob3Qgd29yZHM6IGV4cGxhbmF0aW9uIG1pbnVzIHRyYW5zbGF0aW9uID0ge25wLm5hbm1lYW4oZ2FwKTorLjJmfSBbe2NnWzBdOisuMmZ9LCB7Y2dbMV06Ky4yZn1dIChtZWRpYW4ge25wLm5hbm1lZGlhbihnYXApOisuMmZ9KTsgcG9zaXRpdmUgPSB1bmRlcnN0b29kIGJ1dCBub3QgdHJhbnNsYXRlZCIpCiAgICBsID0gbGlzdChEKVswXTsgYWIgPSBEW2xdWyJhYmxhdGlvbiJdCiAgICBmaWcsIGF4ZXMgPSBwbHQuc3VicGxvdHMoMSwgMiwgZmlnc2l6ZT0oMTEsIDQpKQogICAgYXggPSBheGVzWzBdCiAgICBmb3IgZ3JwLCBjb2wsIGxhYiBpbiBbKFtyIGZvciByIGluIGFiIGlmIG5vdCByWyJob3QiXV0sIEVBU1ksICJvcmRpbmFyeSB3b3JkcyIpLCAoW3IgZm9yIHIgaW4gYWIgaWYgclsiaG90Il1dLCBIQVJELCAiaGFyZCB3b3JkcyIpXToKICAgICAgICB4ID0gW3JbInRyYW5zbGF0ZV9wcml2YXRlIl0gLyByWyJ0cmFuc2xhdGVfZnVsbCJdIGlmIGFicyhyWyJ0cmFuc2xhdGVfZnVsbCJdKSA+IDFlLTYgZWxzZSBucC5uYW4gZm9yIHIgaW4gZ3JwXQogICAgICAgIHkgPSBbclsiZXhwbGFpbl9wcml2YXRlIl0gLyByWyJleHBsYWluX2Z1bGwiXSBpZiBhYnMoclsiZXhwbGFpbl9mdWxsIl0pID4gMWUtNiBlbHNlIG5wLm5hbiBmb3IgciBpbiBncnBdCiAgICAgICAgeCwgeSA9IG5wLmNsaXAoeCwgLTAuNSwgMS41KSwgbnAuY2xpcCh5LCAtMC41LCAxLjUpICAgICAjIGV4dHJlbWUgcmF0aW9zICh0aW55IGRlbm9taW5hdG9ycykgcGlubmVkIHRvIHRoZSBlZGdlCiAgICAgICAgYXguc2NhdHRlcih4LCB5LCBjb2xvcj1jb2wsIHM9MjgsIGFscGhhPTAuOCwgbGFiZWw9bGFiKQogICAgICAgIGZvciByLCB4aSwgeWkgaW4gemlwKGdycCwgeCwgeSk6CiAgICAgICAgICAgIGlmIHJbImhvdCJdOiBheC50ZXh0KHhpLCB5aSwgIiAiICsgclsid29yZCJdLCBmb250c2l6ZT03LCBzdHlsZT0iaXRhbGljIiwgY29sb3I9Y29sKQogICAgYXgucGxvdChbLTEsIDJdLCBbLTEsIDJdLCBjb2xvcj0iI2JiYiIsIGx3PTAuOCk7IGF4LnNldF94bGltKC0wLjUsIDEuNSk7IGF4LnNldF95bGltKC0wLjUsIDEuNSkKICAgIGF4LnNldF94bGFiZWwoImhhcm0gdG8gdGhlIFRSQU5TTEFUSU9OXG4ocmVtb3ZpbmcgdGhlIGxhbmd1YWdlLXByaXZhdGUgcGFydCwgc2hhcmUgb2YgcmVtb3ZpbmcgZXZlcnl0aGluZykiKQogICAgYXguc2V0X3lsYWJlbCgiaGFybSB0byBFWFBMQUlOSU5HIHRoZSB3b3JkIikKICAgIGF4LnNldF90aXRsZSgiQWJvdmUgdGhlIGRpYWdvbmFsOiB0aGUgbW9kZWwgdW5kZXJzdGFuZHMgbW9yZSB0aGFuIGl0IHRyYW5zbGF0ZXMiLCBsb2M9ImxlZnQiLCBmb250c2l6ZT05KTsgYXgubGVnZW5kKGZyYW1lb249RmFsc2UsIGZvbnRzaXplPTgpCiAgICBheCA9IGF4ZXNbMV07IGl0ID0gRFtsXVsiaW50ZXJjaGFuZ2UiXTsgdyA9IDAuMjYKICAgIGZvciBrLCAoYywgY29sKSBpbiBlbnVtZXJhdGUoWygicHJpdmF0ZSIsIEhBUkQpLCAoInNoYXJlZCIsIEVBU1kpLCAoImZ1bGwiLCBHUkVZKV0pOgogICAgICAgIGF4LmJhcihucC5hcmFuZ2UobGVuKGl0KSkgKyAoayAtIDEpICogdywgW3hbImZsaXAiXVtjXSBmb3IgeCBpbiBpdF0sIHcsIGNvbG9yPWNvbCwgbGFiZWw9ZiJzd2FwIHRoZSB7Y30gcGFydCIgaWYgYyAhPSAiZnVsbCIgZWxzZSAic3dhcCBldmVyeXRoaW5nIikKICAgIGF4LnNldF94dGlja3MobnAuYXJhbmdlKGxlbihpdCkpKTsgYXguc2V0X3h0aWNrbGFiZWxzKFt4WyJpZCJdIGZvciB4IGluIGl0XSwgcm90YXRpb249MzAsIGhhPSJyaWdodCIsIHN0eWxlPSJpdGFsaWMiKQogICAgYXguYXhobGluZSgwLCBjb2xvcj0iIzg4OCIsIGx3PTAuOCk7IGF4LmF4aGxpbmUoMSwgY29sb3I9IiM4ODgiLCBsdz0wLjgsIGxzPSI6IikKICAgIGF4LnNldF95bGFiZWwoImhvdyBmYXIgdGhlIEVuZ2xpc2ggbW92ZXMgdG8gdGhlIG90aGVyIHNlbnNlIik7IGF4LmxlZ2VuZChmcmFtZW9uPUZhbHNlLCBmb250c2l6ZT04KQogICAgYXguc2V0X3RpdGxlKCJTd2FwcGluZyBwYXJ0IG9mIG9uZSB3b3JkJ3Mgc3RhdGUgYmV0d2VlbiB0d28gc2Vuc2VzIiwgbG9jPSJsZWZ0IiwgZm9udHNpemU9OSkKICAgIHNhdmVmaWcoZmlnLCBmIkRfbWVkaWF0aW9uX0x7bH0iKQogICAgcHJpbnQoIlxuRXhhbXBsZSBnZW5lcmF0aW9ucyB1bmRlciBpbnRlcmNoYW5nZSAoZnJhbWUgYSwgcGF0Y2hlZCBmcm9tIGZyYW1lIGIpOiIpCiAgICBmb3IgeCBpbiBEW2xdWyJpbnRlcmNoYW5nZSJdWzo4XToKICAgICAgICBnID0geFsicnVucyJdWyJhIl1bImdlbmVyYXRpb25zIl07IHByaW50KGYiICB7eFsnaWQnXToxMXN9IG5vbmU6IHtnWydub25lJ11bOjYwXSFyfSB8IHByaXZhdGU6IHtnWydwcml2YXRlJ11bOjYwXSFyfSB8IGZ1bGw6IHtnWydmdWxsJ11bOjYwXSFyfSIpCgpwYWNrKCkK'))
open('/content/ut/Z_summary.py','wb').write(base64.b64decode('IyBSZWxvYWQgZWFybGllciBzdGFnZXMnIHJlc3VsdHMsIHNvIHRoaXMgc3RhZ2UgYWxzbyB3b3JrcyBpbiBhIGZyZXNoIGtlcm5lbCAoYWZ0ZXIgMDBfc2V0dXAucHkpLgpBID0gbG9hZF9qc29uKCJBX2RldG91ci5qc29uIikgaWYgaGF2ZSgiQV9kZXRvdXIuanNvbiIpIGVsc2UgTm9uZQpsYWRkZXIgPSBsb2FkX2pzb24oIkJfbGFkZGVyLmpzb24iKSBpZiBoYXZlKCJCX2xhZGRlci5qc29uIikgZWxzZSBOb25lCkIgPSBsb2FkX2pzb24oIkJfcm91bmR0cmlwLmpzb24iKSBpZiBoYXZlKCJCX3JvdW5kdHJpcC5qc29uIikgZWxzZSBOb25lCnN1bW1hcnkgPSBsb2FkX2pzb24oIkJfc3VtbWFyeS5qc29uIilbInBlcl9zZW50ZW5jZSJdIGlmIGhhdmUoIkJfc3VtbWFyeS5qc29uIikgZWxzZSB7fQpEID0gbG9hZF9qc29uKCJEX21lZGlhdGlvbi5qc29uIikgaWYgaGF2ZSgiRF9tZWRpYXRpb24uanNvbiIpIGVsc2UgTm9uZQpDX1JFUyA9IHtsYW5nOiBsb2FkX2pzb24oZiJDX3NwZWN0cnVtX3tsYW5nfS5qc29uIikgZm9yIGxhbmcgaW4gUEFJUlMgaWYgaGF2ZShmIkNfc3BlY3RydW1fe2xhbmd9Lmpzb24iKX0KQkVTVF9MQVlFUiA9IHtsYW5nOiBtYXgocm93cywga2V5PWxhbWJkYSByOiByWyJ0b3A1MF9oZWxkIl0pWyJsYXllciJdIGZvciBsYW5nLCByb3dzIGluIENfUkVTLml0ZW1zKCl9CkNfU1BBTlMgPSB7bGFuZzoge2ludChrKTogdiBmb3IgaywgdiBpbiBsb2FkX2pzb24oZiJDX3NwYW5zX3tsYW5nfS5qc29uIikuaXRlbXMoKX0gZm9yIGxhbmcgaW4gQ19SRVMgaWYgaGF2ZShmIkNfc3BhbnNfe2xhbmd9Lmpzb24iKX0KCgpsaW5lcyA9IFtmIiMgU3VtbWFyeToge01PREVMX0lEfSIsICIiLCBmIntOX0xBWUVSU30gbGF5ZXJzLCBkID0ge0RfTU9ERUx9LiBFdmVyeSBudW1iZXIgaXMgaW5zaWRlIHRoaXMgb25lIG1vZGVsLiIsICIiXQppZiBBOgogICAgbGluZXMgKz0gWyIjIyBBLiBFbmdsaXNoIGRldG91ciIsIGYiLSBFbmdsaXNoIHBlYWtzIGF0IGxheWVyIHtBWydkZXRvdXJfcGVha19sYXllciddfS97Tl9MQVlFUlN9OyBJdGFsaWFuIG92ZXJ0YWtlcyBhdCB7QVsnaXRhbGlhbl9vdmVydGFrZXNfYXQnXX0uIiwKICAgICAgICAgICAgICBmIi0gUGVhayBtZWFuIFAoRW5nbGlzaCkgPSB7bnAuYXJyYXkoQVsnUF9lbiddKS5tZWFuKDApLm1heCgpOi4zZn0gb3ZlciB7bGVuKEFbJ1BfZW4nXSl9IHdvcmRzLiIsICIiXQppZiBoYXZlKCJCX3N1bW1hcnkuanNvbiIpOgogICAgYnMgPSBsb2FkX2pzb24oIkJfc3VtbWFyeS5qc29uIik7IHBzID0gYnNbInBlcl9zZW50ZW5jZSJdCiAgICBsaW5lcyArPSBbIiMjIEIuIFJvdW5kIHRyaXAgKG1hc2tlZCkiLCBmIi0gVG9sZXJhbmNlIM61ID0ge2JzWydlcHNpbG9uJ106LjNmfTsgaG90LXdvcmQgQVVST0MgPSB7YnNbJ2F1cm9jX2hvdF93b3JkcyddOi4zZn0uIl0KICAgIGZvciBjIGluIFsidW50cmFuc2xhdGFibGUiLCAiY29uc3RydWN0aW9uIiwgImlkaW9tIiwgImNvbnRyb2wiXToKICAgICAgICB2ID0gW3ggZm9yIHggaW4gcHMudmFsdWVzKCkgaWYgeFsiY2F0Il0gPT0gY10KICAgICAgICBpZiB2OiBsaW5lcy5hcHBlbmQoZiItIHtjfTogVSA9IHtucC5tZWFuKFt4WydVJ10gZm9yIHggaW4gdl0pOi4yZn0gZG91YmxpbmdzLCBib3Jyb3dpbmcgYnV5cyB7bnAubmFubWVhbihbeFsnYm9ycm93X2dhaW4nXSBmb3IgeCBpbiB2XSk6LjNmfSAobWFza2VkOiB7bnAubmFubWVhbihbeFsnYm9ycm93X2dhaW5fbWFza2VkJ10gZm9yIHggaW4gdl0pOi4zZn0pLiIpCiAgICBmYSA9IEJbImR1ZW5kZSJdOyB3bCA9IGRpY3QoemlwKGZhWyJ3b3JkcyJdLCBmYVsicnVuZ3MiXVsiQjEiXVsid29yZF9sZWZ0X21hc2tlZCJdKSkKICAgIGxpbmVzICs9IFtmIi0gKk1lIGd1c3RhIG11Y2hvIGVsIGR1ZW5kZSog4oaSIHtsYWRkZXJbJ2R1ZW5kZSddWydCMSddIXJ9OyBsZWZ0b3ZlciBkdWVuZGUge3dsLmdldCgnZHVlbmRlJywgbnAubmFuKTouMmZ9LCBndXN0YSB7d2wuZ2V0KCdndXN0YScsIG5wLm5hbik6LjJmfSwgbXVjaG8ge3dsLmdldCgnbXVjaG8nLCBucC5uYW4pOi4yZn0uIiwKICAgICAgICAgICAgICAiLSBTZW5zZSByZWNvdmVyZWQgZm9yICpkdWVuZGUqIChzdGFuZGFyZCAvIGtlZXAgLyBnbG9zcyAvIGZvb3Rub3RlIC8gdGhpY2spOiAiICsgIiAvICIuam9pbihmIntmYVsnc2Vuc2UnXVsnZHVlbmRlJ11bJ25vcm0nXVtyXTouMmZ9IiBmb3IgciBpbiBbIkIxIiwgIkIyIiwgIkIzIiwgIkI1IiwgIkI2Il0pLCAiIl0KZm9yIGxhbmcgaW4gQ19SRVM6CiAgICByID0gbmV4dCh4IGZvciB4IGluIENfUkVTW2xhbmddIGlmIHhbImxheWVyIl0gPT0gQkVTVF9MQVlFUltsYW5nXSkKICAgIGxpbmVzICs9IFtmIiMjIEMuIFNwZWN0cnVtIHtsYW5nfeKAk2VuIChsYXllciB7clsnbGF5ZXInXX0pIiwKICAgICAgICAgICAgICBmIi0ge3JbJ25fc2hhcmVkJ119IHNoYXJlZCBkaXJlY3Rpb25zIGFib3ZlIHRoZSBudWxsIChvZiB7RF9NT0RFTH0pOyBmaXQgb24ge3JbJ25fZml0X3BhaXJzJ119IHdvcmQgcGFpcnMuIiwKICAgICAgICAgICAgICBmIi0gRW5nbGlzaCBwcmVkaWN0ZWQgZnJvbSB7bGFuZ306IHtyWydFX2VuX2Zyb21feCddOi4zZn07IHtsYW5nfSBwcmVkaWN0ZWQgZnJvbSBFbmdsaXNoOiB7clsnRV94X2Zyb21fZW4nXTouM2Z9OyBiaWFzIHtyWydiaWFzJ106Ky4zZn0gKHN0YW5kYXJkaXNlZCB7clsnYmlhc19zdGQnXTorLjNmfSkuIl0KICAgIGlmIENfU1BBTlMuZ2V0KGxhbmcpOgogICAgICAgIGF1ID0gQ19TUEFOU1tsYW5nXVtCRVNUX0xBWUVSW2xhbmddXVsiYXVyb2MiXTsgYWcgPSBDX1NQQU5TW2xhbmddW0JFU1RfTEFZRVJbbGFuZ11dWyJzcGVhcm1hbl93aXRoX3JvdW5kdHJpcF9BIl0KICAgICAgICBsaW5lcy5hcHBlbmQoZiItIEFVUk9DIGhvdCB2cyBvdGhlciB3b3JkczogVSB7YXVbJ1UnXTouMmZ9LCBVX3JhdyB7YXVbJ1VfcmF3J106LjJmfSwgI3Rva2VucyBiYXNlbGluZSB7YXVbJ250b2snXTouMmZ9OyBhZ3JlZW1lbnQgd2l0aCByb3VuZCB0cmlwIHthZ30uIikKICAgIGxpbmVzLmFwcGVuZCgiIikKaWYgRDoKICAgIGZvciBsLCBkIGluIEQuaXRlbXMoKToKICAgICAgICBhYiA9IGRbImFibGF0aW9uIl07IGhvdCA9IFtyIGZvciByIGluIGFiIGlmIHJbImhvdCJdXQogICAgICAgIGYgPSBsYW1iZGEgciwgdDogcltmInt0fV9wcml2YXRlIl0gLyByW2Yie3R9X2Z1bGwiXSBpZiBhYnMocltmInt0fV9mdWxsIl0pID4gMWUtNiBlbHNlIG5wLm5hbgogICAgICAgIGxpbmVzICs9IFtmIiMjIEQuIE1lZGlhdGlvbiAobGF5ZXIge2x9KSIsCiAgICAgICAgICAgICAgICAgIGYiLSBIb3Qgd29yZHMsIHByaXZhdGUtcGFydCBzaGFyZSBvZiBoYXJtOiB0cmFuc2xhdGlvbiB7bnAubmFubWVhbihbZihyLCAndHJhbnNsYXRlJykgZm9yIHIgaW4gaG90XSk6LjJmfSwgZXhwbGFuYXRpb24ge25wLm5hbm1lYW4oW2YociwgJ2V4cGxhaW4nKSBmb3IgciBpbiBob3RdKTouMmZ9LiIsCiAgICAgICAgICAgICAgICAgICItIEZsaXAgZnJhY3Rpb25zIChwcml2YXRlIC8gc2hhcmVkIC8gZnVsbCk6ICIgKyAiOyAiLmpvaW4oZiJ7eFsnaWQnXX0ge3hbJ2ZsaXAnXVsncHJpdmF0ZSddOisuMmZ9L3t4WydmbGlwJ11bJ3NoYXJlZCddOisuMmZ9L3t4WydmbGlwJ11bJ2Z1bGwnXTorLjJmfSIgZm9yIHggaW4gZFsiaW50ZXJjaGFuZ2UiXSksICIiXQpvcGVuKG9zLnBhdGguam9pbihPVVQsICJTVU1NQVJZLm1kIiksICJ3Iikud3JpdGUoIlxuIi5qb2luKGxpbmVzKSkKcHJpbnQoIlxuIi5qb2luKGxpbmVzKSkKCnBhY2soKQo='))
open('/content/ut/run_bg.py','wb').write(base64.b64decode('IyBCYWNrZ3JvdW5kIGRyaXZlciAocnVucyBhcyBhIGRldGFjaGVkIHByb2Nlc3Mgb24gdGhlIFZNLCBzbyBhIGRyb3BwZWQgQ0xJIGNvbm5lY3Rpb24gY2Fubm90IHN0YWxsIG9yIGtpbGwgaXQpOgojIHNldHVwLCB0aGVuIHN0YWdlcyBBLCBCLCBDMS4gIFRoZSBub3RlYm9vayBrZXJuZWwgbGF0ZXIgcmUtcnVucyBzZXR1cCBhbmQgQzEgKHdoaWNoIHRoZW4gZmluZHMgaXRzIGNhY2hlKSBiZWZvcmUgQzIuCmltcG9ydCB0cmFjZWJhY2ssIHN5cwp0cnk6CiAgICBmb3IgZiBpbiBbIjAwX3NldHVwLnB5IiwgIkFfZGV0b3VyLnB5IiwgIkJfcm91bmR0cmlwLnB5IiwgIkMxX2NvbGxlY3QucHkiXToKICAgICAgICBwcmludChmIj09PSBCRyBTVEFSVCB7Zn0iLCBmbHVzaD1UcnVlKQogICAgICAgIGV4ZWMob3BlbihmIi9jb250ZW50L3V0L3tmfSIpLnJlYWQoKSwgZ2xvYmFscygpKQogICAgICAgIHByaW50KGYiPT09IEJHIE9LIHtmfSIsIGZsdXNoPVRydWUpCiAgICBwcmludCgiPT09IEJHX0RPTkUiLCBmbHVzaD1UcnVlKQpleGNlcHQgRXhjZXB0aW9uOgogICAgdHJhY2ViYWNrLnByaW50X2V4YygpOyBwcmludCgiPT09IEJHX0ZBSUxFRCIsIGZsdXNoPVRydWUpOyBzeXMuZXhpdCgxKQo='))
print(sorted(os.listdir('/content/ut')))


['00_setup.py', 'A_detour.py', 'B_roundtrip.py', 'C1_collect.py', 'C2_spectrum.py', 'D_mediation.py', 'Z_summary.py', 'run_bg.py']


In [ ]:
import subprocess, sys, os
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.used","--format=csv,noheader"],capture_output=True,text=True).stdout)
log = open("/content/bg.log", "w")
p = subprocess.Popen([sys.executable, "-u", "/content/ut/run_bg.py"], stdout=log, stderr=subprocess.STDOUT, cwd="/content",
                     start_new_session=True)
open("/content/bg.pid", "w").write(str(p.pid)); print("launched pid", p.pid)


NVIDIA A100-SXM4-80GB, 81920 MiB, 0 MiB

launched pid 1715


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START 00_setup.py | ▶ load unsloth/Llama-3.3-70B-Instruct-bnb-4bit ... 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START 00_setup.py | ▶ load unsloth/Llama-3.3-70B-Instruct-bnb-4bit ... 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START B_roundtrip.py | ▶ Part B: generating the ladder ... 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     3200/31200 sentences, 41337 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     6400/31200 sentences, 82273 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     9600/31200 sentences, 123249 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     12800/31200 sentences, 164421 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     12800/31200 sentences, 164421 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     22400/31200 sentences, 287747 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     25600/31200 sentences, 328791 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     28800/31200 sentences, 369620 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py | ▶ Part C [hi]: word alignment (31200 pairs) ... 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     0/31200 sentences, 227 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     3200/31200 sentences, 22950 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     6400/31200 sentences, 45813 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     6400/31200 sentences, 45813 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     9600/31200 sentences, 68537 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     9600/31200 sentences, 68537 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     12800/31200 sentences, 91214 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     16000/31200 sentences, 113890 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     16000/31200 sentences, 113890 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     19200/31200 sentences, 136676 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     22400/31200 sentences, 159102 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     22400/31200 sentences, 159102 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     22400/31200 sentences, 159102 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     25600/31200 sentences, 181359 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     25600/31200 sentences, 181359 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     25600/31200 sentences, 181359 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS alive | === BG START C1_collect.py |     28800/31200 sentences, 203624 fit pairs 


In [ ]:
import os, subprocess
pid = int(open("/content/bg.pid").read())
alive = os.path.exists(f"/proc/{pid}") and open(f"/proc/{pid}/stat").read().split()[2] != "Z"
L = [l.rstrip() for l in open("/content/bg.log", errors="replace") if l.strip()]
marks = [l for l in L if l.startswith("=== BG")]
prog = [l for l in L if "sentences," in l or l.startswith("▶") or l.startswith("✓")]
print("STATUS", "alive" if alive else "exited", "|", marks[-1] if marks else "-", "|", (prog[-1] if prog else "-")[:110],
      "| traceback" if any("Traceback" in l for l in L) else "")


STATUS exited | === BG_DONE | ✓ Part C [hi]: collecting states at layers [20, 32, 40, 48, 60]: 2864s 


In [ ]:
# 00_setup.py: run once per Colab session (via `colab exec -f`), before any stage.
# Settings can be overridden by executing assignments first, e.g.  echo "MODEL_ID = '...'" | colab exec -s ut
g = globals()
MODEL_ID    = g.get("MODEL_ID", "unsloth/Llama-3.3-70B-Instruct-bnb-4bit")   # alt: unsloth/Qwen2.5-72B-Instruct-bnb-4bit; dry run: Qwen/Qwen2.5-0.5B-Instruct
DRY_RUN     = g.get("DRY_RUN", any(s in MODEL_ID for s in ["0.5B", "1.5B", "tiny"]))
OUT_ROOT    = g.get("OUT_ROOT", "/content/ut_results")       # on the VM; fetched with `colab download` (see pack())
LOCAL_CACHE = g.get("LOCAL_CACHE", "/content/cache")         # big covariance files; stay on the VM
RUN_A, RUN_B, RUN_C, RUN_D = g.get("RUN_A", True), g.get("RUN_B", True), g.get("RUN_C", True), g.get("RUN_D", True)
PAIRS       = g.get("PAIRS", ["es", "hi"])
N_SENT_FIT  = g.get("N_SENT_FIT", 2000 if DRY_RUN else 30000)
N_SENT_HELD = g.get("N_SENT_HELD", 200 if DRY_RUN else 1200)
LAMBDAS     = g.get("LAMBDAS", [1e-3, 1e-2, 1e-1])
SEED        = g.get("SEED", 0)
CHAT_DATE   = "26 Jul 2024"
MIN_GPU_GB  = g.get("MIN_GPU_GB", 0 if DRY_RUN else 75)

import subprocess, sys, matplotlib
matplotlib.use("Agg")
import torch
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0); print(f"GPU: {_p.name}, {_p.total_memory/1e9:.0f} GB")
    if _p.total_memory / 1e9 < MIN_GPU_GB:
        raise RuntimeError(f"This GPU has {_p.total_memory/1e9:.0f} GB; the 4-bit 70B model needs an 80 GB card. "
                           "Stop this session and create one with --gpu H100 (or an 80 GB A100).")
else:
    print("WARNING: no GPU visible")
try:
    import spacy; spacy.load("es_core_news_sm")
except Exception:
    subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm", "-q"], check=False)


import os, re, json, math, time, random, difflib, gc
from collections import defaultdict
import numpy as np
import torch
import matplotlib.pyplot as plt

def slug(s): return re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")
OUT = os.path.join(OUT_ROOT, slug(MODEL_ID)); os.makedirs(OUT, exist_ok=True)
FIG = os.path.join(OUT, "figures"); os.makedirs(FIG, exist_ok=True)
CACHE = os.path.join(LOCAL_CACHE, slug(MODEL_ID)); os.makedirs(CACHE, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def _np(o):
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, torch.Tensor): return o.tolist()
    raise TypeError(type(o))
def save_json(name, obj):
    with open(os.path.join(OUT, name), "w") as f: json.dump(obj, f, indent=1, default=_np, ensure_ascii=False)
def load_json(name):
    with open(os.path.join(OUT, name)) as f: return json.load(f)
def have(name): return os.path.exists(os.path.join(OUT, name))

class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); print(f"▶ {self.label} ..."); return self
    def __exit__(self, *a): print(f"✓ {self.label}: {time.time()-self.t:.0f}s")

def auroc(pos, neg):
    """Probability that a random positive scores above a random negative (ties count half)."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    pos, neg = pos[~np.isnan(pos)], neg[~np.isnan(neg)]
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    allv = np.concatenate([pos, neg]); r = rankdata(allv)
    return float((r[:len(pos)].sum() - len(pos)*(len(pos)+1)/2) / (len(pos)*len(neg)))
def rankdata(a):
    a = np.asarray(a, float); order = np.argsort(a, kind="mergesort"); r = np.empty(len(a))
    sa = a[order]; i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and sa[j+1] == sa[i]: j += 1
        r[order[i:j+1]] = (i + j) / 2 + 1; i = j + 1
    return r
def spearman(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float); m = ~(np.isnan(x) | np.isnan(y))
    if m.sum() < 3: return float("nan")
    return float(np.corrcoef(rankdata(x[m]), rankdata(y[m]))[0, 1])
def boot_ci(x, n=2000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed); m = rng.choice(x, (n, len(x))).mean(1)
    return (float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5)))

# Visual language shared with the earlier figures
EASY, HARD, LOAN, GREY, INK = "#2f7d6d", "#c4572e", "#5b4fc4", "#d9d6cf", "#2b2b2b"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
def savefig(fig, name):
    fig.savefig(os.path.join(FIG, name + ".png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG, name + ".svg"), bbox_inches="tight")
    plt.show(); plt.close(fig)
print("results ->", OUT)

LANG_NAME = {"es": "Spanish", "pt": "Portuguese", "it": "Italian", "de": "German", "hi": "Hindi", "en": "English"}

SENTENCES = [
    # --- untranslatable words in context ---
    dict(id="duende", lang="es", cat="untranslatable", text="Me gusta mucho el duende.",
         hot={"duende": "the mysterious, almost demonic power and emotional intensity of a great artistic performance, as in flamenco (Lorca's sense)",
              "gusta": "it pleases me: the thing is the subject and I am the one affected, like having a taste for it"}),
    dict(id="sobremesa", lang="es", cat="untranslatable", text="Después de comer nos quedamos dos horas de sobremesa.",
         hot={"sobremesa": "the relaxed time spent talking at the table after a meal has finished"}),
    dict(id="madrugada", lang="es", cat="untranslatable", text="Llegamos a casa de madrugada, muertos de sueño.",
         hot={"madrugada": "the small hours, between midnight and dawn"}),
    dict(id="estrenar", lang="es", cat="untranslatable", text="Hoy estreno los zapatos que me regalaste.",
         hot={"estreno": "I am using or wearing something for the very first time"}),
    dict(id="verguenza", lang="es", cat="untranslatable", text="Sentí vergüenza ajena durante todo su discurso.",
         hot={"vergüenza ajena": "embarrassment felt on behalf of someone else who is embarrassing themselves"}),
    dict(id="tertulia", lang="es", cat="untranslatable", text="Los jueves vamos a la tertulia del café de la esquina.",
         hot={"tertulia": "a regular informal gathering of people for conversation, often literary or political"}),
    # --- constructions ---
    dict(id="olvido", lang="es", cat="construction", text="Se me olvidó la llave en el coche.",
         hot={"Se me olvidó": "I forgot it, but framed as something that happened to me by accident, as if the key forgot itself on me"}),
    dict(id="estar_aburrido", lang="es", cat="construction", text="Juan está aburrido en la reunión.",
         hot={"está aburrido": "is bored, a temporary state (not 'is boring')"}),
    dict(id="ser_aburrido", lang="es", cat="construction", text="Juan es aburrido en las reuniones.",
         hot={"es aburrido": "is boring, a lasting characteristic (not 'is bored')"}),
    dict(id="ojala", lang="es", cat="construction", text="Ojalá llueva mañana por la tarde.",
         hot={"Ojalá": "a strong hope or wish, 'if only', originally 'may God will it'"}),
    # --- idioms ---
    dict(id="pelo", lang="es", cat="idiom", text="Creo que me estás tomando el pelo.",
         hot={"tomando el pelo": "teasing me, pulling my leg"}),
    dict(id="nubes", lang="es", cat="idiom", text="Mi hermana siempre está en las nubes.",
         hot={"en las nubes": "daydreaming, absent-minded"}),
    dict(id="pelos", lang="es", cat="idiom", text="Mi abuela no tiene pelos en la lengua.",
         hot={"no tiene pelos en la lengua": "speaks very frankly, without holding anything back"}),
    # --- controls ---
    dict(id="tren", lang="es", cat="control", text="El tren sale a las ocho de la mañana.", hot={}),
    dict(id="hermano", lang="es", cat="control", text="Mi hermano vive en un piso pequeño.", hot={}),
    dict(id="pan", lang="es", cat="control", text="Compré pan y leche en la tienda.", hot={}),
    dict(id="reunion", lang="es", cat="control", text="La reunión empieza a las diez en punto.", hot={}),
    dict(id="gatos", lang="es", cat="control", text="Tengo dos gatos negros y un perro.", hot={}),
    dict(id="invierno", lang="es", cat="control", text="En invierno hace mucho frío en el norte.", hot={}),
    # --- other European languages (Part B only) ---
    dict(id="saudade", lang="pt", cat="untranslatable", text="Tenho muita saudade da minha avó.",
         hot={"saudade": "a deep, bittersweet longing for someone or something absent, that may never return"}),
    dict(id="magari", lang="it", cat="construction", text="Magari fosse vero!",
         hot={"Magari": "if only (a wistful wish)"}),
    dict(id="fernweh", lang="de", cat="untranslatable", text="Im Winter habe ich immer Fernweh.",
         hot={"Fernweh": "an ache to travel to faraway places, the opposite of homesickness"}),
    # --- Hindi (DRAFTS: to be checked by a native speaker) ---
    dict(id="hi_pasand", lang="hi", cat="construction", draft=True, text="मुझे चाय बहुत पसंद है।",
         hot={"मुझे": "to me (dative experiencer: the tea is liked by me, like Spanish 'me gusta')"}),
    dict(id="hi_jugaad", lang="hi", cat="untranslatable", draft=True, text="उसने जुगाड़ से पुरानी गाड़ी ठीक कर दी।",
         hot={"जुगाड़": "a clever improvised fix made with whatever is at hand"}),
    dict(id="hi_mann", lang="hi", cat="untranslatable", draft=True, text="आज पढ़ाई में मेरा मन नहीं लग रहा।",
         hot={"मन": "heart-mind: the seat of feeling, will and attention together"}),
    dict(id="hi_apnapan", lang="hi", cat="untranslatable", draft=True, text="इस घर में मुझे बहुत अपनापन लगता है।",
         hot={"अपनापन": "a warm sense of belonging and intimate familiarity, of being among one's own"}),
    dict(id="hi_train", lang="hi", cat="control", draft=True, text="ट्रेन सुबह आठ बजे चलती है।", hot={}),
    dict(id="hi_bhai", lang="hi", cat="control", draft=True, text="मेरा भाई दिल्ली में रहता है।", hot={}),
]

# Sense minimal pairs for Part D: same word, two contexts, two senses.
# The English candidates differ only in the critical word, so their log-odds measures which sense the translation carries.
SENSE_PAIRS = [
    dict(id="cura", word="cura", a="Me preocupa el cura.", b="Me preocupa la cura.",
         en_a="I am worried about the {}.", en_b="I am worried about the {}.", sense_a="priest", sense_b="cure"),
    dict(id="cometa", word="cometa", a="Miramos el cometa.", b="Miramos la cometa.",
         en_a="We looked at the {}.", en_b="We looked at the {}.", sense_a="comet", sense_b="kite"),
    dict(id="pendiente", word="pendiente", a="Vi el pendiente.", b="Vi la pendiente.",
         en_a="I saw the {}.", en_b="I saw the {}.", sense_a="earring", sense_b="slope"),
    dict(id="corte", word="corte", a="Me sorprendió el corte.", b="Me sorprendió la corte.",
         en_a="The {} surprised me.", en_b="The {} surprised me.", sense_a="cut", sense_b="court"),
    dict(id="frente", word="frente", a="Señaló el frente.", b="Señaló la frente.",
         en_a="He pointed at the {}.", en_b="He pointed at the {}.", sense_a="front", sense_b="forehead"),
    dict(id="papa", word="papa", a="Me gusta el papa.", b="Me gusta la papa.",
         en_a="I like the {}.", en_b="I like the {}.", sense_a="pope", sense_b="potato"),
    dict(id="aburrido", word="aburrido", a="Juan es aburrido.", b="Juan está aburrido.",
         en_a="Juan is {}.", en_b="Juan is {}.", sense_a="boring", sense_b="bored"),
    dict(id="duende_ctx", word="duende", a="En ese cuento infantil, lo que más me gusta es el duende.",
         b="En ese cante flamenco, lo que más me gusta es el duende.",
         en_a="In that children's story, what I like most is the {}.",
         en_b="In that flamenco song, what I like most is the {}.", sense_a="goblin", sense_b="soul"),
]

SID = {S["id"]: S for S in SENTENCES}

# Part A: Spanish -> Italian word translation, with the English equivalent for the logit lens
DETOUR_SHOTS = [("perro", "cane"), ("ventana", "finestra"), ("mesa", "tavolo"), ("leche", "latte")]
DETOUR_WORDS = [("árbol", "albero", "tree"), ("pájaro", "uccello", "bird"), ("manzana", "mela", "apple"),
    ("cuchillo", "coltello", "knife"), ("calle", "strada", "street"), ("niño", "bambino", "child"),
    ("nube", "nuvola", "cloud"), ("queso", "formaggio", "cheese"), ("llave", "chiave", "key"),
    ("zapato", "scarpa", "shoe"), ("camisa", "camicia", "shirt"), ("mantequilla", "burro", "butter"),
    ("cerdo", "maiale", "pig"), ("caballo", "cavallo", "horse"), ("hermano", "fratello", "brother"),
    ("pescado", "pesce", "fish"), ("cuchara", "cucchiaio", "spoon"), ("tenedor", "forchetta", "fork"),
    ("silla", "sedia", "chair"), ("puerta", "porta", "door"), ("ciudad", "città", "city"),
    ("rodilla", "ginocchio", "knee"), ("hielo", "ghiaccio", "ice"), ("lluvia", "pioggia", "rain"),
    ("nieve", "neve", "snow"), ("hoja", "foglia", "leaf"), ("lengua", "lingua", "tongue"),
    ("oreja", "orecchio", "ear"), ("huevo", "uovo", "egg"), ("ajo", "aglio", "garlic"),
    ("cerebro", "cervello", "brain"), ("espejo", "specchio", "mirror")]

from transformers import AutoTokenizer, AutoModelForCausalLM
with Timer(f"load {MODEL_ID}"):
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    _dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=_dtype, device_map="auto")
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=_dtype, device_map="auto")
    model.eval()

def _get_attr_path(obj, path):
    for p in path.split("."): obj = getattr(obj, p)
    return obj
def find_decoder(m):
    for base in ["model", "model.model", "model.language_model", "transformer"]:
        try:
            b = _get_attr_path(m, base)
            layers = getattr(b, "layers", None) or getattr(b, "h", None)
            norm = getattr(b, "norm", None) or getattr(b, "ln_f", None)
            if layers is not None and norm is not None: return b, layers, norm
        except AttributeError: pass
    raise ValueError("could not find decoder layers")

BASE, LAYERS_MOD, FINAL_NORM = find_decoder(model)
LM_HEAD = model.get_output_embeddings()
N_LAYERS = len(LAYERS_MOD)
D_MODEL = model.config.hidden_size if hasattr(model.config, "hidden_size") else model.config.text_config.hidden_size
INPUT_DEV = model.get_input_embeddings().weight.device
BOS = [tok.bos_token_id] if tok.bos_token_id is not None else []
print(f"{N_LAYERS} layers, d={D_MODEL}, BOS={BOS}, input device {INPUT_DEV}")
if torch.cuda.is_available(): print(f"GPU memory in use: {torch.cuda.memory_allocated()/1e9:.1f} GB")

WORD_RE = re.compile(r"[\wऀ-ॣ०-ॿ]+(?:['’]\w+)?")
def words_with_spans(text): return [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(text)]
def enc(text, offsets=False): return tok(text, add_special_tokens=False, return_offsets_mapping=offsets)
def ntok(text): return len(enc(text)["input_ids"])

def chat_prompt(user, prefix=""):
    msgs = [{"role": "user", "content": user}]
    try: p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, date_string=CHAT_DATE)
    except Exception: p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return p + prefix

def tokens_in_span(offsets, a, b):
    return [i for i, (x, y) in enumerate(offsets) if y > x and y > a and x < b]

def find_phrase(text, phrase):
    m = re.search(r"(?<!\w)" + re.escape(phrase) + r"(?!\w)", text, flags=re.I)
    return (m.start(), m.end()) if m else None

def positions_at(prompt, a, b):
    return tokens_in_span(enc(prompt, offsets=True)["offset_mapping"], a, b)

def positions_in_prompt(prompt, sentence, phrase=None, span=None):
    """Token positions of `phrase` (or of the character span `span`) inside `sentence`, inside `prompt`."""
    base = prompt.rfind(sentence); assert base >= 0, "sentence not in prompt"
    sp = span or find_phrase(sentence, phrase); assert sp is not None, f"{phrase!r} not in {sentence!r}"
    return positions_at(prompt, base + sp[0], base + sp[1])

class _Stop(Exception): pass

def make_edit_hook(edits_by_b):
    """edits_by_b: {batch index: [(positions LongTensor, fn)]}; fn maps states [k,d] -> new states [k,d]."""
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        if h.shape[1] <= 1: return None              # cached decoding step: leave untouched
        for b, items in edits_by_b.items():
            for pos, fn in items:
                pos = pos.to(h.device)
                if int(pos.max()) < h.shape[1]:
                    h[b, pos] = fn(h[b, pos]).to(h.dtype)
        return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
    return hook

def _register_edits(edits_for_batch):
    """edits_for_batch: list (one per batch row) of lists of (layer, positions, fn)."""
    per_layer = defaultdict(lambda: defaultdict(list))
    for b, items in enumerate(edits_for_batch):
        for (layer, pos, fn) in (items or []):
            per_layer[layer][b].append((torch.as_tensor(pos, dtype=torch.long), fn))
    return [LAYERS_MOD[layer - 1].register_forward_hook(make_edit_hook(dict(eb))) for layer, eb in per_layer.items()]

@torch.no_grad()
def score(prompts, conts, batch=8, edits=None):
    """Teacher-forced log p of each continuation given its prompt.
    edits: optional list (per item) of [(layer, prompt token positions, fn)]; 'layer l' = hidden_states[l] = output of block l-1.
    Returns per item: dict(lp=array over continuation tokens, offsets=char offsets of those tokens in the continuation)."""
    res = [None] * len(prompts)
    for i in range(0, len(prompts), batch):
        idx = list(range(i, min(i + batch, len(prompts))))
        seqs, spans, offs = [], [], []
        for j in idx:
            p = enc(prompts[j])["input_ids"]; c = enc(conts[j], offsets=True)
            seqs.append(p + c["input_ids"]); spans.append((len(p), len(p) + len(c["input_ids"]))); offs.append(c["offset_mapping"])
        L = max(map(len, seqs))
        ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long); am = torch.zeros_like(ids)
        for b, s in enumerate(seqs): ids[b, :len(s)] = torch.tensor(s); am[b, :len(s)] = 1
        handles = _register_edits([edits[j] for j in idx]) if edits is not None else []
        try:
            out = BASE(input_ids=ids.to(INPUT_DEV), attention_mask=am.to(INPUT_DEV))
        finally:
            for h in handles: h.remove()
        H = out.last_hidden_state
        for b, j in enumerate(idx):
            s, e = spans[b]
            if e <= s: res[j] = dict(lp=np.zeros(0), offsets=[]); continue
            logits = LM_HEAD(H[b, s - 1:e - 1].to(LM_HEAD.weight.device)).float()
            tgt = ids[b, s:e].to(logits.device)
            res[j] = dict(lp=torch.log_softmax(logits, -1).gather(-1, tgt[:, None])[:, 0].cpu().numpy(), offsets=offs[b])
        del out, H
    return res

@torch.no_grad()
def generate(prompts, max_new_tokens=64, batch=16):
    tok.padding_side = "left"; outs = []
    for i in range(0, len(prompts), batch):
        e = tok(prompts[i:i + batch], add_special_tokens=False, return_tensors="pt", padding=True).to(INPUT_DEV)
        g = model.generate(**e, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                           top_k=None, pad_token_id=tok.pad_token_id)
        outs += [tok.decode(r, skip_special_tokens=True).strip() for r in g[:, e["input_ids"].shape[1]:]]
    tok.padding_side = "right"
    return outs

@torch.no_grad()
def generate_with_edits(prompt, edits, max_new_tokens=40):
    """Greedy generation for one prompt, with edits applied to the prompt positions (prefill only)."""
    e = tok(prompt, add_special_tokens=False, return_tensors="pt").to(INPUT_DEV)
    handles = _register_edits([edits])
    try:
        g = model.generate(**e, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                           top_k=None, pad_token_id=tok.pad_token_id)
    finally:
        for h in handles: h.remove()
    return tok.decode(g[0, e["input_ids"].shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def capture(ids, am, layers):
    """Residual states at `layers` (dict layer -> [B,T,d]); stops the forward pass after the deepest one."""
    store, deepest = {}, max(layers)
    def mk(l):
        def hook(m, i, o):
            store[l] = (o[0] if isinstance(o, tuple) else o).detach()
            if l == deepest: raise _Stop()
        return hook
    hs = [LAYERS_MOD[l - 1].register_forward_hook(mk(l)) for l in layers]
    try: BASE(input_ids=ids, attention_mask=am)
    except _Stop: pass
    finally:
        for h in hs: h.remove()
    return store

@torch.no_grad()
def capture_prompt_states(prompt, layer):
    e = tok(prompt, add_special_tokens=False, return_tensors="pt").to(INPUT_DEV)
    return capture(e["input_ids"], e["attention_mask"], [layer])[layer][0]

def encode_raw(text):
    e = enc(text, offsets=True)
    return BOS + e["input_ids"], [(0, 0)] * len(BOS) + list(e["offset_mapping"])

print("helpers ready")

RUNGS = ["B0", "B1", "B2", "B3", "B5", "B6"]
RUNG_NAME = {"B0": "word for word", "B1": "standard", "B2": "keep the word", "B3": "inline gloss",
             "B5": "footnote", "B6": "thick translation"}
NON_BORROW = ["B0", "B1", "B3", "B5", "B6"]
MAXTOK = {"B0": 64, "B1": 64, "B2": 72, "B3": 120, "B5": 150, "B6": 260}

def rung_user(r, s, L):
    n = LANG_NAME[L]
    return {
        "B0": f"Give a word-for-word English gloss of this {n} sentence: one English word for each {n} word, in the original order. Reply with the gloss only.\n\n{s}",
        "B1": f"Translate this {n} sentence into natural English. Reply with the translation only.\n\n{s}",
        "B2": f"Translate this {n} sentence into English, but keep any word that has no good English equivalent in the original {n}, in italics with *asterisks*. Reply with the translation only.\n\n{s}",
        "B3": f"Translate this {n} sentence into English. After each word or phrase that is hard to translate, add a short gloss in square brackets (at most 8 words). Do not quote the {n} words. Reply with the translation only.\n\n{s}",
        "B5": f"Translate this {n} sentence into English, then add one footnote (at most 40 words) explaining what the translation loses. Do not quote the {n} words. Format: the translation, a blank line, then 'Note: ...'.\n\n{s}",
        "B6": f"Translate this {n} sentence into English, then write one paragraph (at most 120 words) opening up the meanings, connotations and grammar of its hard parts, so that an English reader understands what the original says. Do not quote the {n} words.\n\n{s}",
    }[r]



def hot_word_index(S):
    idx = set()
    for span in S["hot"]:
        sp = find_phrase(S["text"], span)
        if sp: idx |= {k for k, (_, a, b) in enumerate(words_with_spans(S["text"])) if b > sp[0] and a < sp[1]}
    return idx




import tarfile
def pack():
    """Bundle everything under OUT into one archive for `colab download`."""
    p = f"{OUT_ROOT.rstrip('/')}_{slug(MODEL_ID)}.tar.gz"
    with tarfile.open(p, "w:gz") as t: t.add(OUT, arcname=slug(MODEL_ID))
    print("PACKED", p)


print("setup done:", MODEL_ID)


GPU: NVIDIA A100-SXM4-80GB, 85 GB


results -> /content/ut_results/unsloth_Llama_3_3_70B_Instruct_bnb_4bit


▶ load unsloth/Llama-3.3-70B-Instruct-bnb-4bit ...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

✓ load unsloth/Llama-3.3-70B-Instruct-bnb-4bit: 30s
80 layers, d=8192, BOS=[128000], input device cuda:0
GPU memory in use: 39.5 GB
helpers ready
setup done: unsloth/Llama-3.3-70B-Instruct-bnb-4bit


In [ ]:
# Reload earlier stages' results, so this stage also works in a fresh kernel (after 00_setup.py).
A = load_json("A_detour.json") if have("A_detour.json") else None
ladder = load_json("B_ladder.json") if have("B_ladder.json") else None
B = load_json("B_roundtrip.json") if have("B_roundtrip.json") else None
summary = load_json("B_summary.json")["per_sentence"] if have("B_summary.json") else {}


def pick_layers():
    f = [0.25, 0.4, 0.5, 0.6, 0.75]
    ls = {max(1, min(N_LAYERS - 1, round(x * N_LAYERS))) for x in f}
    if A and A.get("detour_peak_layer"): ls.add(int(min(A["detour_peak_layer"], N_LAYERS - 1)))
    return sorted(ls)
C_LAYERS = pick_layers(); print("Part C layers:", C_LAYERS)

def load_pairs(lang, n):
    """n parallel sentence pairs (x, en, source); a mix of domains so 'conventional practice' isn't one register."""
    from datasets import load_dataset
    # Europarl first: if it fails to load, OPUS-100 takes the whole quota
    sources = ([("Helsinki-NLP/europarl", "en-es")] if lang == "es" else []) + [("Helsinki-NLP/opus-100", f"en-{lang}")]
    want, pairs, seen = int(n * 1.3), [], set()
    for k, (name, cfg) in enumerate(sources):
        quota = (want - len(pairs)) // (len(sources) - k)
        try: ds = load_dataset(name, cfg, split="train", streaming=True)
        except Exception as e: print(f"  could not load {name}/{cfg}: {e}"); continue
        got = 0
        for ex in ds:
            tr = ex.get("translation", {}); en, xx = tr.get("en"), tr.get(lang)
            if not en or not xx: continue
            ne, nx = len(en.split()), len(xx.split())
            if not (8 <= ne <= 40 and 8 <= nx <= 40) or abs(ne - nx) > 0.5 * max(ne, nx): continue
            if sum(ch.isdigit() for ch in xx) > 6 or xx in seen: continue
            seen.add(xx); pairs.append((xx.strip(), en.strip(), name)); got += 1
            if got >= quota: break
        print(f"  {name}/{cfg}: {got} pairs")
    random.Random(SEED).shuffle(pairs)
    return pairs[:n]

def words_for(lang, texts):
    """Word tokens (text, start, end, morph, pos). Spanish uses spaCy when available, for morphology."""
    if lang == "es":
        try:
            import spacy
            nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
            out = []
            for doc in nlp.pipe(texts, batch_size=256):
                out.append([(t.text, t.idx, t.idx + len(t.text), str(t.morph), t.pos_) for t in doc
                            if WORD_RE.fullmatch(t.text)])
            return out
        except Exception as e:
            print("  spaCy unavailable, using regex words (no morphology probes):", e)
    return [[(w, a, b, "", "") for w, a, b in words_with_spans(t)] for t in texts]

_AL = None
def align_pairs(src_words, tgt_words, batch=64, layer=8):
    """1-to-1 word links (i, j) per sentence pair: mutual nearest neighbours of XLM-R (layer 8) word embeddings,
    i.e. SimAlign's high-precision 'argmax' method, implemented here to avoid a fragile dependency."""
    global _AL
    from transformers import AutoTokenizer, AutoModel
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if _AL is None:
        _AL = (AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base"),
               AutoModel.from_pretrained("FacebookAI/xlm-roberta-base").to(dev).eval())
    atok, amod = _AL
    def embed(word_lists):
        wl_ = [[w[0] for w in wl] or ["."] for wl in word_lists]
        e = atok(wl_, is_split_into_words=True, return_tensors="pt", padding=True, truncation=True, max_length=256)
        with torch.no_grad():
            H = amod(**{k: v.to(dev) for k, v in e.items()}, output_hidden_states=True).hidden_states[layer]
        out = []
        for b, wl in enumerate(wl_):
            M = torch.zeros(len(wl), H.shape[1], device=dev)
            for p, w in enumerate(e.word_ids(b)):
                if w is not None: M[w, p] = 1.0
            cnt = M.sum(1, keepdim=True)
            out.append((torch.nn.functional.normalize((M @ H[b].float()) / cnt.clamp_min(1), dim=-1), cnt[:, 0] > 0))
        return out
    links = []
    for i in range(0, len(src_words), batch):
        es, et = embed(src_words[i:i + batch]), embed(tgt_words[i:i + batch])
        for (a, va), (b, vb), sw, tw in zip(es, et, src_words[i:i + batch], tgt_words[i:i + batch]):
            if not sw or not tw: links.append([]); continue
            S = a @ b.T; S[~va] = -2; S[:, ~vb] = -2
            fwd, bwd = S.argmax(1), S.argmax(0)
            links.append([(k, int(fwd[k])) for k in range(len(sw)) if int(bwd[fwd[k]]) == k and float(S[k, fwd[k]]) > -1
                          and not any(ch.isdigit() for ch in sw[k][0] + tw[int(fwd[k])][0])])
    return links

# ---- collection ----

class CovAcc:
    def __init__(self, d, dev):
        z = lambda *s: torch.zeros(*s, dtype=torch.float64, device=dev)
        self.n, self.sx, self.sy = 0, z(d), z(d)
        self.Sxx, self.Syy, self.Sxy = z(d, d), z(d, d), z(d, d)
    def add(self, X, Y):
        X, Y = X.double(), Y.double()
        self.n += X.shape[0]; self.sx += X.sum(0); self.sy += Y.sum(0)
        self.Sxx += X.T @ X; self.Syy += Y.T @ Y; self.Sxy += X.T @ Y
    def finish(self):
        n = self.n; mx, my = self.sx / n, self.sy / n
        return dict(n=n, mx=mx.cpu(), my=my.cpu(),
                    Cxx=((self.Sxx - n * torch.outer(mx, mx)) / (n - 1)).cpu(),
                    Cyy=((self.Syy - n * torch.outer(my, my)) / (n - 1)).cpu(),
                    Cxy=((self.Sxy - n * torch.outer(mx, my)) / (n - 1)).cpu())

def word_states(texts, word_lists, layers, batch=32):
    """For each text: dict layer -> [n_words, d] float32 (word = mean of its tokens), plus a validity mask
    (words containing the first content token, an attention sink, are invalid)."""
    out = []
    for i in range(0, len(texts), batch):
        enc_ = [encode_raw(t) for t in texts[i:i + batch]]
        L = max(len(e[0]) for e in enc_)
        ids = torch.full((len(enc_), L), tok.pad_token_id, dtype=torch.long); am = torch.zeros_like(ids)
        for b, (s, _) in enumerate(enc_): ids[b, :len(s)] = torch.tensor(s); am[b, :len(s)] = 1
        st = capture(ids.to(INPUT_DEV), am.to(INPUT_DEV), layers)
        for b, (s, offs) in enumerate(enc_):
            wl = word_lists[i + b]; first = len(BOS)
            toks = [tokens_in_span(offs, w[1], w[2]) for w in wl]
            valid = np.array([len(t) > 0 and first not in t for t in toks], bool)
            M = torch.zeros(len(wl), len(s))
            for k, t in enumerate(toks):
                if t: M[k, t] = 1.0 / len(t)
            M = M.to(st[layers[0]].device)
            out.append(({l: M @ st[l][b, :len(s)].float() for l in layers}, valid))
        del st
    return out

def collect_pair(lang):
    tag = f"C_{lang}"
    cov_paths = {l: os.path.join(CACHE, f"{tag}_cov_L{l}.pt") for l in C_LAYERS}
    held_path = os.path.join(CACHE, f"{tag}_held.pt")
    if all(os.path.exists(p) for p in cov_paths.values()) and os.path.exists(held_path): return
    with Timer(f"Part C [{lang}]: corpus"):
        pairs = load_pairs(lang, N_SENT_FIT + N_SENT_HELD)
        xs, ens = [p[0] for p in pairs], [p[1] for p in pairs]
        wx, we = words_for(lang, xs), words_for("en", ens)
    with Timer(f"Part C [{lang}]: word alignment ({len(pairs)} pairs)"):
        links = align_pairs(wx, we)
    n_fit = len(pairs) - N_SENT_HELD
    acc = {l: CovAcc(D_MODEL, INPUT_DEV) for l in C_LAYERS}
    held = {l: ([], []) for l in C_LAYERS}; held_meta = []
    with Timer(f"Part C [{lang}]: collecting states at layers {C_LAYERS}"):
        B_ = 32
        for i in range(0, len(pairs), B_):
            sx = word_states(xs[i:i + B_], wx[i:i + B_], C_LAYERS)
            sy = word_states(ens[i:i + B_], we[i:i + B_], C_LAYERS)
            for b in range(len(sx)):
                k = i + b; (rx, vx), (ry, vy) = sx[b], sy[b]
                lk = [(a, c) for a, c in links[k] if vx[a] and vy[c]]
                if not lk: continue
                ia = torch.tensor([a for a, _ in lk]); ic = torch.tensor([c for _, c in lk])
                for l in C_LAYERS:
                    X, Y = rx[l][ia.to(rx[l].device)], ry[l][ic.to(ry[l].device)]
                    if k < n_fit: acc[l].add(X, Y)
                    else: held[l][0].append(X.half().cpu()); held[l][1].append(Y.half().cpu())
                if k >= n_fit:
                    held_meta += [dict(sent=k, x=wx[k][a][0], y=we[k][c][0], morph=wx[k][a][3], pos=wx[k][a][4]) for a, c in lk]
            if (i // B_) % 100 == 0: print(f"    {i}/{len(pairs)} sentences, {acc[C_LAYERS[0]].n} fit pairs")
    for l in C_LAYERS: torch.save(acc[l].finish(), cov_paths[l])
    torch.save(dict(X={l: torch.cat(held[l][0]) for l in C_LAYERS}, Y={l: torch.cat(held[l][1]) for l in C_LAYERS},
                    meta=held_meta, n_fit_pairs=acc[C_LAYERS[0]].n, n_sent=len(pairs)), held_path)
    del acc; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

if RUN_C:
    for lang in PAIRS: collect_pair(lang)

pack()


Part C layers: [20, 32, 40, 48, 60]
PACKED /content/ut_results_unsloth_Llama_3_3_70B_Instruct_bnb_4bit.tar.gz


*File Operation*: `download` on `/content/ut_results_unsloth_Llama_3_3_70B_Instruct_bnb_4bit.tar.gz`

In [ ]:
import threading, contextlib, traceback
def _run_stages():
    with open('/content/k.log', 'a', buffering=1) as f, contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
        try:
            for st in ['C2_spectrum.py', 'D_mediation.py', 'Z_summary.py']:
                print('=== K START', st, flush=True)
                exec(open('/content/ut/' + st).read(), globals())
                print('=== K OK', st, flush=True)
            print('=== K_DONE', flush=True)
        except Exception:
            traceback.print_exc(); print('=== K_FAILED', flush=True)
_t = threading.Thread(target=_run_stages, daemon=True); _t.start(); print("thread started")


In [ ]:
import base64, threading, contextlib, traceback
open("/content/ut/strict_pass.py","wb").write(base64.b64decode("IyBzdHJpY3RfcGFzcy5weTogcmUtcnVuIHRoZSBDMiBwcm9iZXMgYW5kIFBhcnQgRCB3aXRoICJzaGFyZWQiID0gaGVsZC1vdXQgcmhvID4gMC43MSAobW9yZSB0aGFuIGhhbGYga2VwdCBwZXIgcm91bmQKIyB0cmlwKSBpbnN0ZWFkIG9mICJhYm92ZSB0aGUgc2h1ZmZsZWQtcGFpciBudWxsIiwgd2hpY2ggc2F0dXJhdGVkIGF0IH44MTkwIG9mIDgxOTIgZGltZW5zaW9ucyBmb3IgZXMgb24gdGhlIDcwQi4KIyBSdW4gaW4gdGhlIHNhbWUga2VybmVsIGFmdGVyIEMyIGFuZCBEOyBvdXRwdXRzIGdldCBhIF9zdHJpY3Qgc3VmZml4LCB0aGUgb3JpZ2luYWwgZmlsZXMgYXJlIHVudG91Y2hlZC4KaW1wb3J0IHRvcmNoLCB0aHJlYWRpbmcsIGNvbnRleHRsaWIsIHRyYWNlYmFjawpTVFJJQ1QgPSAwLjUgKiogMC41CmZvciBfaywgX0YgaW4gQ19GSVRTLml0ZW1zKCk6CiAgICBfRlsic2hhcmVkX251bGwiXSA9IF9GWyJzaGFyZWQiXQogICAgX0ZbInNoYXJlZCJdID0gdG9yY2gubm9uemVybyhfRlsicmhvX2giXSA+IFNUUklDVClbOiwgMF0KICAgIHByaW50KCJzdHJpY3Qgc2hhcmVkIiwgX2ssIGxlbihfRlsic2hhcmVkIl0pLCAib2YiLCBsZW4oX0ZbInJob19oIl0pKQpfc2F2ZV9qc29uX29yaWcsIF9zYXZlZmlnX29yaWcgPSBzYXZlX2pzb24sIHNhdmVmaWcKc2F2ZV9qc29uID0gbGFtYmRhIG5hbWUsIG9iajogX3NhdmVfanNvbl9vcmlnKG5hbWUucmVwbGFjZSgiLmpzb24iLCAiX3N0cmljdC5qc29uIiksIG9iaikKc2F2ZWZpZyA9IGxhbWJkYSBmaWcsIG5hbWU6IF9zYXZlZmlnX29yaWcoZmlnLCBuYW1lICsgIl9zdHJpY3QiKQppZiBSVU5fQyBhbmQgImVzIiBpbiBDX1JFUyBhbmQgQ19TUEFOUy5nZXQoImVzIik6CiAgICBoZWxkID0gdG9yY2gubG9hZChvcy5wYXRoLmpvaW4oQ0FDSEUsICJDX2VzX2hlbGQucHQiKSwgd2VpZ2h0c19vbmx5PUZhbHNlKTsgbWV0YSA9IGhlbGRbIm1ldGEiXQogICAgbCA9IEJFU1RfTEFZRVJbImVzIl07IEZfID0gQ19GSVRTWygiZXMiLCBsKV07IGZpdCA9IEZfWyJmaXQiXQogICAgaWYgYW55KG1bIm1vcnBoIl0gZm9yIG0gaW4gbWV0YSk6CiAgICAgICAgZnJvbSBza2xlYXJuLmxpbmVhcl9tb2RlbCBpbXBvcnQgUmlkZ2VDbGFzc2lmaWVyCiAgICAgICAgZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgY3Jvc3NfdmFsX3Njb3JlCiAgICAgICAgWiA9ICgoKGhlbGRbIlgiXVtsXS50byhDREVWKS5kb3VibGUoKSAtIGZpdFsibXgiXSkgQCBmaXRbIld4Il0pIEAgZml0WyJVIl0pLmZsb2F0KCkuY3B1KCkubnVtcHkoKQogICAgICAgIFogLz0gWi5zdGQoMCwga2VlcGRpbXM9VHJ1ZSkgKyAxZS04CiAgICAgICAgc2ggPSBucC56ZXJvcyhaLnNoYXBlWzFdLCBib29sKTsgc2hbRl9bInNoYXJlZCJdLmNwdSgpLm51bXB5KCldID0gVHJ1ZQogICAgICAgIGRlZiBmZWF0KG0sIGtleSk6IHJldHVybiBkaWN0KGt2LnNwbGl0KCI9IikgZm9yIGt2IGluIG1bIm1vcnBoIl0uc3BsaXQoInwiKSBpZiAiPSIgaW4ga3YpLmdldChrZXkpCiAgICAgICAgdGFza3MgPSB7CiAgICAgICAgICAgICJncmFtbWF0aWNhbCBnZW5kZXIiOiBbKGksIGZlYXQobSwgIkdlbmRlciIpKSBmb3IgaSwgbSBpbiBlbnVtZXJhdGUobWV0YSkgaWYgbVsicG9zIl0gaW4gKCJOT1VOIiwgIkFESiIpIGFuZCBmZWF0KG0sICJHZW5kZXIiKSBpbiAoIk1hc2MiLCAiRmVtIildLAogICAgICAgICAgICAibnVtYmVyIjogWyhpLCBmZWF0KG0sICJOdW1iZXIiKSkgZm9yIGksIG0gaW4gZW51bWVyYXRlKG1ldGEpIGlmIG1bInBvcyJdIGluICgiTk9VTiIsICJBREoiLCAiVkVSQiIpIGFuZCBmZWF0KG0sICJOdW1iZXIiKSBpbiAoIlNpbmciLCAiUGx1ciIpXSwKICAgICAgICAgICAgInN1Ymp1bmN0aXZlIHZzIGluZGljYXRpdmUiOiBbKGksIGZlYXQobSwgIk1vb2QiKSkgZm9yIGksIG0gaW4gZW51bWVyYXRlKG1ldGEpIGlmIG1bInBvcyJdIGluICgiVkVSQiIsICJBVVgiKSBhbmQgZmVhdChtLCAiTW9vZCIpIGluICgiU3ViIiwgIkluZCIpXSwKICAgICAgICAgICAgImltcGVyZmVjdCB2cyBwcmV0ZXJpdGUiOiBbKGksIGZlYXQobSwgIlRlbnNlIikpIGZvciBpLCBtIGluIGVudW1lcmF0ZShtZXRhKSBpZiBtWyJwb3MiXSBpbiAoIlZFUkIiLCAiQVVYIikgYW5kIGZlYXQobSwgIk1vb2QiKSA9PSAiSW5kIiBhbmQgZmVhdChtLCAiVGVuc2UiKSBpbiAoIkltcCIsICJQYXN0IildLAogICAgICAgICAgICAibm91biB2cyB2ZXJiIChzYW5pdHkpIjogWyhpLCBtWyJwb3MiXSkgZm9yIGksIG0gaW4gZW51bWVyYXRlKG1ldGEpIGlmIG1bInBvcyJdIGluICgiTk9VTiIsICJWRVJCIildLAogICAgICAgIH0KICAgICAgICBwcm9iZSA9IHt9CiAgICAgICAgZm9yIG5hbWUsIGl0ZW1zIGluIHRhc2tzLml0ZW1zKCk6CiAgICAgICAgICAgIGxhYnMgPSBbYiBmb3IgXywgYiBpbiBpdGVtc10KICAgICAgICAgICAgaWYgbGVuKGl0ZW1zKSA8IDYwIG9yIG1pbihsYWJzLmNvdW50KHYpIGZvciB2IGluIHNldChsYWJzKSkgPCAyMDogcHJpbnQoZiIgIHtuYW1lfTogdG9vIGZldyBleGFtcGxlcyIpOyBjb250aW51ZQogICAgICAgICAgICBpaSA9IG5wLmFycmF5KFtpIGZvciBpLCBfIGluIGl0ZW1zXSk7IHkgPSBucC5hcnJheShbYiA9PSBzb3J0ZWQoc2V0KGxhYnMpKVswXSBmb3IgYiBpbiBsYWJzXSkKICAgICAgICAgICAgY2xmID0gUmlkZ2VDbGFzc2lmaWVyKGFscGhhPWZsb2F0KGxlbihpaSkpLCBjbGFzc193ZWlnaHQ9ImJhbGFuY2VkIikKICAgICAgICAgICAgYWNjID0ge2s6IGZsb2F0KGNyb3NzX3ZhbF9zY29yZShjbGYsIFpbaWldWzosIGNvbHNdLCB5LCBjdj01LCBzY29yaW5nPSJiYWxhbmNlZF9hY2N1cmFjeSIpLm1lYW4oKSkKICAgICAgICAgICAgICAgICAgIGZvciBrLCBjb2xzIGluIFsoImFsbCIsIG5wLm9uZXNfbGlrZShzaCkpLCAoInNoYXJlZCBvbmx5Iiwgc2gpLCAoInByaXZhdGUgb25seSIsIH5zaCldIGlmIGNvbHMuYW55KCl9CiAgICAgICAgICAgIGNsZi5maXQoWltpaV0sIHkpOyB3Z3QgPSBucC5yYXZlbChjbGYuY29lZl8pICoqIDIgICAgICAjIGNvZWZfIGlzICgxLCBkKSBvciAoZCwpIGRlcGVuZGluZyBvbiB0aGUgc2tsZWFybiB2ZXJzaW9uCiAgICAgICAgICAgIHByb2JlW25hbWVdID0gZGljdChuPWxlbihpaSksIHNoYXJlZF9zaGFyZT1mbG9hdCh3Z3Rbc2hdLnN1bSgpIC8gd2d0LnN1bSgpKSwgZXhwZWN0ZWQ9ZmxvYXQoc2gubWVhbigpKSwgYWNjPWFjYykKICAgICAgICAgICAgcHJpbnQoZiIgIHtuYW1lOjI4c30gbj17bGVuKGlpKTo1ZH0gIHdlaWdodCBvbiBzaGFyZWQgZGlyZWN0aW9ucyB7cHJvYmVbbmFtZV1bJ3NoYXJlZF9zaGFyZSddOi4yZn0gIgogICAgICAgICAgICAgICAgICBmIihjaGFuY2Uge3NoLm1lYW4oKTouMmZ9KSAgYWNjdXJhY3kgYWxsL3NoYXJlZC9wcml2YXRlID0gIiArICIvIi5qb2luKGYie3Y6LjJmfSIgZm9yIHYgaW4gYWNjLnZhbHVlcygpKSkKICAgICAgICBzYXZlX2pzb24oIkNfcHJvYmVzX2VzLmpzb24iLCBkaWN0KGxheWVyPWwsIHByb2Jlcz1wcm9iZSkpCmV4ZWMob3BlbigiL2NvbnRlbnQvdXQvRF9tZWRpYXRpb24ucHkiKS5yZWFkKCksIGdsb2JhbHMoKSkKc2F2ZV9qc29uLCBzYXZlZmlnID0gX3NhdmVfanNvbl9vcmlnLCBfc2F2ZWZpZ19vcmlnCnBhY2soKQo="))
def _run_strict():
    with open("/content/k2.log", "a", buffering=1) as f, contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
        try:
            print("=== S START", flush=True); exec(open("/content/ut/strict_pass.py").read(), globals()); print("=== S_DONE", flush=True)
        except Exception:
            traceback.print_exc(); print("=== S_FAILED", flush=True)
threading.Thread(target=_run_strict, daemon=True).start(); print("strict thread started")


strict thread started


*File Operation*: `download` on `/content/ut_results_unsloth_Llama_3_3_70B_Instruct_bnb_4bit.tar.gz`

In [ ]:
import threading, contextlib, traceback
def _run_strict_D():
    global save_json, savefig, have
    with open("/content/k3.log", "a", buffering=1) as f, contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
        _sj, _sf, _hv = save_json, savefig, have
        try:
            print("=== SD START", {k: len(v["shared"]) for k, v in C_FITS.items()}, flush=True)   # still the strict sets
            save_json = lambda name, obj: _sj(name.replace(".json", "_strict.json"), obj)
            savefig = lambda fig, name: _sf(fig, name + "_strict")
            have = lambda name: _hv(name.replace(".json", "_strict.json")) if name.startswith("D_mediation") else _hv(name)
            exec(open("/content/ut/D_mediation.py").read(), globals())
            print("=== SD_DONE", flush=True)
        except Exception:
            traceback.print_exc(); print("=== SD_FAILED", flush=True)
        finally:
            save_json, savefig, have = _sj, _sf, _hv
            pack()
threading.Thread(target=_run_strict_D, daemon=True).start(); print("strict D thread started")


strict D thread started


*File Operation*: `download` on `/content/ut_results_unsloth_Llama_3_3_70B_Instruct_bnb_4bit.tar.gz`